# Reviewer verification: RPA-screened kite asymptotics

This notebook accompanies `main_brm_kite_asymptotics.tex`. The main result, in Ry per electron, is
$$e_{\rm kite}=-0.2213782293M_4r_s^{-1/4}+0.5670608947M_5r_s^{-1/2}
-0.4575317487\mathcal S_3r_s^{-3/4}+O_\zeta(r_s^{-1}).$$
Here $r_s=[3/(4\pi n)]^{1/3}/a_B$, $n=n_\uparrow+n_\downarrow$, and $\zeta=(n_\uparrow-n_\downarrow)/n$ is fixed in the limit. Momenta use total-density $k_F=(3\pi^2n)^{1/3}$. Define $x_\sigma=(1+\sigma\zeta)^{1/3}$, $M_n=\tfrac12\sum_\sigma x_\sigma^n$, and $\mathcal S_3=(1160M_6+891M_4M_5)/2051$.

[Symbols used in code](#symbols) are listed in the next cell.

**Read by result.** Every check is reachable from this table. Reading order is independent of execution order; run the cells in their displayed order.

| Result | Manuscript basis | Checks |
|---|---|---|
| Functional, units and coupling average | Sec. II; Eq. (10) | [1](#check-1), [15](#check-15), [25](#check-25), [26](#check-26) |
| Coefficient of $r_s^{-1/4}$: $-0.2213782293$ | Sec. III A; Eqs. (6), (23), (28), (30), (31) | [2](#check-2), [3](#check-3), [4](#check-4), [5](#check-5), [31](#check-31) |
| Matching and $A_2=8/15$ | Sec. III B; Eqs. (35)–(39) | [6](#check-6), [7](#check-7), [21](#check-21), [22](#check-22), [24](#check-24), [31](#check-31) |
| Third-order coefficient $A_3$ | Sec. III C; Eqs. (40)–(42), (A.11) | [8](#check-8), [9](#check-9), [10](#check-10), [11](#check-11), [12](#check-12), [31](#check-31) |
| Spin dependence and exact rescaling | Sec. III D; Eqs. (44), (45), (48), (49) | [13](#check-13), [15](#check-15), [16](#check-16), [25](#check-25), [30](#check-30) |
| Remainder $O_\zeta(q_*^{-4})$ | App. C; Eq. (C.3) | [18](#check-18), [19](#check-19), [20](#check-20), [23](#check-23), [27](#check-27), [31](#check-31) |
| Numerical tests and convergence | Sec. IV; Tables I-III | [17](#check-17), [21](#check-21), [28](#check-28), [29](#check-29) |
| Static screening reference | App. D | [14](#check-14), [16](#check-16) |
| Spin interpolation and its first correction | Sec. V A; Eqs. (51)–(54) | [32](#check-32) |
| Signed interpolation bias and density dependence | Sec. V B; Eqs. (55)–(57); Fig. 8 | [33](#check-33), [34](#check-34) |

**Two notes on layout.** Check 31 appears between Checks 27 and 28 because it collects the exact derivation certificates for Checks 1-27 and belongs with them, ahead of the slow independent integrators of Checks 28-30; the execution summary counts every check regardless of position. Checks 22 and 23 share one code cell, announced in their common heading.

**Where the independent evidence is.** Checks 28-30 evaluate the original functional without any of $F$, $J_C$, $\Phi_4$, $\Psi$, $A_1$, $A_2$, $A_3$, so they can contradict the derivation. Checks labeled `MODEL` assemble profiles that were already used to derive the coefficients and can only test internal consistency. The equation table below links each result to the compiled manuscript.

| Evidence | Meaning |
|---|---|
| `EXACT` | An exact symbolic residual vanishes under the declared assumptions. |
| `NUM` | Numerical quadrature agrees at the displayed parameters and tolerance. |
| `CHECK` | Arithmetic, a sampled inequality, or implementation consistency. |
| `MODEL` | Assembly of the derived asymptotic profiles. |
| `DIAGNOSTIC` | Original-integral evaluation or a change under numerical refinement. |

The continuous-variable remainder estimates are proved in Appendix C. The notebook checks their supporting identities and endpoints.

**Run.** Use Python 3.12.14 and `../requirements-spin.txt`, restart the kernel, then **Run All**. Default execution includes the independent integral; `RUN_EXTENDED=True` adds a larger scan. `python run_inprocess.py` is an alternative when kernel sockets are unavailable. The delivered HTML contains actual outputs from this notebook's fresh execution.

**Data provenance.** Every numerical manuscript figure and table is generated from the replicate records in `../production/raw/` using `../production/render.py`. The integral settings are in `../production/config.json`; [stepwise reproduction instructions](../REPRODUCE_SPIN.md) distinguish integral evaluation from rendering. The production section below verifies coverage, source hashes, aggregate arithmetic and regenerated files. Checks 28–30 use an imaginary-axis implementation independent of the direct BRM estimator. The production shell plot shares that imaginary-axis implementation; its agreement with the notebook is a consistency check. Historical production values have been replaced by newly traceable calculations.

**Production rerun.** `RUN_PRODUCTION=False` verifies and redraws the delivered records without repeating all production integrals. Setting it to `True` runs every configured production dataset; this can take tens of minutes. The individual commands in `REPRODUCE_SPIN.md` allow each dataset to be run and inspected separately.

| Symbols | Meaning |
|---|---|
| $Q=q_*=(4ar_s/3)^{1/4}$, $a=(4/\pi)(4/9\pi)^{1/3}$ | Moving momentum scale and screening constant. |
| $x=q/Q$, $u=\nu/q^2$, $g=x^{-4}$ | Scaled momentum, imaginary frequency, and screening strength. |
| $\mathcal L(q,\nu)$ | Normalized Lindhard shape; distinct from $G_\sigma^\lambda$. |
| $I_0$, $I_2$, $V_B$ | Same-sphere Coulomb moment, quadratic moment, and ball volume. |
| $M_{\rm E}$, $M\ge8$ | Euler integral and fixed momentum cutoff, respectively. |
| $Z(q)$, $Z_Y$ | Geometric occupation factor and static Yukawa residue. |
| $x_{\rm pol}=2^{1/3}$ | Full-polarization scale; distinct from $x=q/Q$. |
| $N_{\rm samp}$, $N_{\rm rep}$, $s_{\rm rep}$ | Points per replicate, replicate count, and sample standard deviation of replicate means. |

Each code cell defines its local auxiliary variables. In Python, `R` is the exact rational-number constructor; `Rc`, `Rn`, and `coupling` represent the physical coupling function $R(c)$.

## Manuscript correspondence

| Result | Equations in this PDF | Checks |
|---|---|---|
| Main result | (6) `eq:main-rs`, (46) `eq:main-qstar-zeta` | 5, 7, 9–13, 25, 31 |
| Functional and normalization | (16) `eq:K-master`, (10) `eq:K-normalization`, (14) `eq:A0-factors`, (20) `eq:Xdyn-definition` | 1, 15, 28 |
| Response | (23) `eq:Gexact`, (28) `eq:lindhard-largeq`, (24) `eq:polarized-response` | 2, 16, 21, 25 |
| Leading coefficient | (31) `eq:Fprofile`, (34) `eq:A1`, (A.1) `eq:A1-beta` | 4, 5, 31 |
| Fixed momentum | (35) `eq:K0-outer`, (37) `eq:Zfinitepart`, (38) `eq:A2` | 6, 7 |
| Third-order coefficients | (A.4) `eq:A3cross-profile`, (A.8) `eq:A3fp-profile`, (A.11) `eq:A3L-profile`, (A.14) `eq:MEuler`, (42) `eq:A3` | 8–12, 24, 31 |
| Full polarization | (49) `eq:full-polarization-scaling`, (50) `eq:ferro-integral` | 13, 16, 30 |
| Remainder | (C.1) `eq:G-bounds`, (C.2) `eq:fixed-q-bound`, (C.3) `eq:composite-density`, (C.9) `eq:cross-integrated` | 18, 27, 31 |
| Static reference | (D.1) `eq:yukawaT` | 14, 20, 31 |
| Numerical residuals | (B.1) `eq:scaled-profile`, (B.2) `eq:energy-residual-tests` | 15, 17, 19, 29 |

Equation numbers are generated from `equation_map.json`; its source hash is checked in the final cell. Stable labels identify the mathematical statements across revisions.


**Code-only distribution.** Manuscript TeX, bibliography sources, compiled manuscript PDF and saved TeX exports are supplied separately in `prb_manuscript_source.tar.gz`. Figure/data checks regenerate the TeX exports in temporary directories and verify their unchanged reference hashes. The final source-hash check is explicitly skipped when the manuscript is absent; mathematical checks and independent integral diagnostics still run.

**Final verification, 2026-09-28.** All reviewer-notebook code cells were executed afresh. The 284 main checks passed, followed by Checks 32–34. Final provenance and Checks 32–34 were repeated in a separate fresh process after the initial process retained an older imported provenance module during the code revision. The original integral diagnostics retain their successful fresh outputs. The 134 locked production records were validated and rendered; the complete production integrals were not recomputed. Detailed execution history, scope and logs are in `audit/20260928/`.


<a id="symbols"></a>

## Symbols used in code

**Symbols in code.** Names follow the manuscript; ASCII names are used in the code and in the printed labels.

| Manuscript | Code identifiers | Meaning | Eq. |
|---|---|---|---|
| $A_0$, $e_{2x}$ | `A0`, `e2x` | prefactor $3/(32\pi^5)$; bare second-order exchange | (13), (14); (11) |
| $\alpha_L$, $a$ | `alphaL_mp`, `a_mp`, `a_np` | $(4/9\pi)^{1/3}$ and $a=4\alpha_L/\pi$ ($a_{\rm TF}$ in the companion paper) | (21) |
| $R(c)$ | `Rc` | dimensionless screening factor; $2\overline W=vR(c)$ | (18) |
| $\mathcal L$ | `lindhard_shape_exact`, `lindhard_shape_large_nu`, `lindhard_shape_num`, `lindhard_shape_spin`, `lindhard_shape_pol` | positive imaginary-axis Lindhard shape | (22), (23), (28) |
| $x_\sigma$, $M_n$, $\mathcal S_3$ | `x_plus` ($x_+$ at $\zeta=1$), `Mn`, `S3` | spin moments | (4), (5) |
| $A_1,A_2,A_3$ | `A1`, `A3`, `A1_np`, `A2_np`, `A3_np` | expansion coefficients | (6), (34), (38), (42) |
| $K_0$ | `K0` | outer kernel | (35) |
| $J_C$, $\Phi_4$ | `JC`, `JCn`, `Phi4` | coupling-integrated frequency functions | (40); App. A |
| $C(q)$, $V(r)$ | `C_of_q`, `C_closed`, `V` | overlap integral and lens volume | (36); App. A |
| $U$ | `U` | $\sqrt{2\pi}\,\Gamma(3/4)/\Gamma(1/4)$ | App. B |
| $\Gamma$ | `Gamma` (sympy) | Euler gamma function (distinct from the many-body vertex by its arguments) | — |

The [equation-by-equation audit](#equation-audit) at the end lists, for all 88 numbered displays, the checks that test each one and what remains analytic.


## Setup

`exact(label, expr)` simplifies an exact residual and requires zero. `num(label, a, b, tol)` tests $|a-b|/\max(1,|b|)\le\mathrm{tol}$. Numerical profile integrals use 30-digit working precision unless a cell states otherwise.

`Rn(c)` evaluates $R(c)$ stably: it uses its Taylor series below $10^{-6}$ and `log1p` otherwise. Check 1 verifies the series and coupling integral.

Production reruns use an exact runtime contract and compare complete result files with a fixed reference before writing. For two fresh evaluations of all 134 records, run `python production/repeat_all.py --jobs 2 --passes 2` from the bundle root. A saved-data consistency check below is distinct from that full rerun. Numerical tolerances used for mathematical quadrature checks are separate from the exact production-file comparison. Runtime logs and notebook serialization are not part of the scientific-result byte contract.

In [1]:
import time, importlib.metadata, json, hashlib
from pathlib import Path
STARTED=time.perf_counter()
RUN_EXTENDED=False
SKIPPED=[]
KINDS=[]
import sympy as sp
import mpmath as mp
mp.mp.dps = 30
pi, R, oo = sp.pi, sp.Rational, sp.oo

RESULTS = []
def check(label, ok, detail="", kind="CHECK"):
    ok = bool(ok); RESULTS.append((label, ok)); KINDS.append(kind)
    print(kind + "  " + ("PASS  " if ok else "FAIL  ") + label + (("   [" + detail + "]") if detail else ""))
def exact(label, expr):
    d = sp.simplify(expr); check(label, d == 0, "simplify -> %s" % d, kind="EXACT")
def num(label, a, b, tol=1e-12):
    a, b = mp.mpf(a), mp.mpf(b); err = abs(a-b)/max(1, abs(b))
    check(label, err <= tol, "|a-b|/max(1,|b|) = %s; tolerance = %s" % (mp.nstr(err, 3),tol), kind="NUM")

# robust numerical version of the coupling average: Taylor series near c=0 avoids 0/0 cancellation
def Rn(c):
    c = mp.mpf(c)
    return 1 - 2*c/3 + c**2/2 - 2*c**3/5 + c**4/3 if c < mp.mpf('1e-6') else 2*(c - mp.log1p(c))/c**2
print("SymPy", sp.__version__, "| mpmath", mp.__version__, "| working precision", mp.mp.dps, "digits")
# Production records are data inputs, not analytical proof certificates.
RUN_PRODUCTION=False
BUNDLE_ROOT=Path.cwd().resolve().parent
import sys
sys.path.insert(0,str(BUNDLE_ROOT/'production'))
import runtime as production_runtime
production_runtime.validate()
import common as production_common
PRODUCTION=json.loads((BUNDLE_ROOT/'production/generated/summary.json').read_text())


SymPy 1.14.0 | mpmath 1.3.0 | working precision 30 digits


<a id="check-1"></a>
## Check 1 — Normalization and coupling average

The convention is $\mathcal K[v]=e_{2x}$ and $e_{\rm kite}=2\int_0^1\mathcal K[W_\lambda]d\lambda=2\mathcal K[\overline W]$ (Eq. (10)). The code multiplies the factors of $A_0=3/(32\pi^5)$, integrates $2\lambda/(1+\lambda c)$ symbolically, and checks the zero-screening limit and Taylor series:
$$R(c)=\frac{2[c-\ln(1+c)]}{c^2}=1-\frac23c+\frac12c^2+O(c^3).$$
The factor $(1/2)_{\rm norm}$ refers to this normalization convention. The bare energy is $e_{2x}=\ln2/3-3\zeta(3)/(2\pi^2)$ Ry, where $\zeta(3)$ is the Riemann zeta value. Multiplying the prefactor tests bookkeeping; the full bare-integral test is in Check 28.

In [2]:
A0 = R(3)/(32*pi**5)
exact("Eqs.(13),(14) 2*(1/2)*2*(1/2)*(4pi)^2 (2pi)^-9 (3pi^2) = 3/(32 pi^5)", 2*R(1,2)*2*R(1,2)*(4*pi)**2*(2*pi)**(-9)*(3*pi**2) - A0)
c, lam = sp.symbols('c lambda', positive=True)
Rc = 2*(c - sp.log(1+c))/c**2
exact("Eq.(18) 2 int lam dlam/(1+lam c) = 2[c - ln(1+c)]/c^2", sp.integrate(2*lam/(1+lam*c), (lam, 0, 1)) - Rc)
exact("Eq.(18) R(0) = 1", sp.limit(Rc, c, 0) - 1)
exact("Eq.(18) R(c) = 1 - 2c/3 + c^2/2 + O(c^3)", sp.series(Rc, c, 0, 3).removeO() - (1 - 2*c/3 + c**2/2))
e2x = sp.log(2)/3 - 3*sp.zeta(3)/(2*pi**2)
num("Eq.(11)  e2x = 0.0483583 Ry", sp.N(e2x, 20), mp.mpf('0.0483583'), tol=1e-6)
# Eq.(19): the split needs R(0)=1, so the screening correction starts at order c
exact("Eqs.(18),(19) (R(c)-1)/c -> -2/3: e_kite - e_2x = K[v(R-1)] vanishes with the screening", sp.limit((Rc - 1)/c, c, 0) + R(2,3))
# Eq.(16), static line: the imaginary-axis frequency integral reproduces the Goldstone denominator
def _freq14(a, b):
    return mp.quad(lambda nu: (a*b - nu**2)/((a*a + nu*nu)*(b*b + nu*nu)), [0, 1, mp.inf])/mp.pi
for a14, b14 in [('0.7', '-1.3'), ('-0.2', '0.9'), ('0.4', '1.1'), ('-0.8', '-0.5')]:
    a14, b14 = mp.mpf(a14), mp.mpf(b14)
    goldstone = -1/(abs(a14) + abs(b14)) if a14*b14 < 0 else mp.mpf(0)
    num("Eq.(16) static line: int dnu/pi Re[1/((i nu-a)(i nu-b))] = -Theta(-ab)/(|a|+|b|), a=%s b=%s" % (a14, b14), _freq14(a14, b14), goldstone, tol=1e-25)


EXACT  PASS  Eqs.(13),(14) 2*(1/2)*2*(1/2)*(4pi)^2 (2pi)^-9 (3pi^2) = 3/(32 pi^5)   [simplify -> 0]
EXACT  PASS  Eq.(18) 2 int lam dlam/(1+lam c) = 2[c - ln(1+c)]/c^2   [simplify -> 0]
EXACT  PASS  Eq.(18) R(0) = 1   [simplify -> 0]
EXACT  PASS  Eq.(18) R(c) = 1 - 2c/3 + c^2/2 + O(c^3)   [simplify -> 0]
NUM  PASS  Eq.(11)  e2x = 0.0483583 Ry   [|a-b|/max(1,|b|) = 1.78e-8; tolerance = 1e-06]
EXACT  PASS  Eqs.(18),(19) (R(c)-1)/c -> -2/3: e_kite - e_2x = K[v(R-1)] vanishes with the screening   [simplify -> 0]
NUM  PASS  Eq.(16) static line: int dnu/pi Re[1/((i nu-a)(i nu-b))] = -Theta(-ab)/(|a|+|b|), a=0.7 b=-1.3   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-25]
NUM  PASS  Eq.(16) static line: int dnu/pi Re[1/((i nu-a)(i nu-b))] = -Theta(-ab)/(|a|+|b|), a=-0.2 b=0.9   [|a-b|/max(1,|b|) = 9.86e-32; tolerance = 1e-25]
NUM  PASS  Eq.(16) static line: int dnu/pi Re[1/((i nu-a)(i nu-b))] = -Theta(-ab)/(|a|+|b|), a=0.4 b=1.1   [|a-b|/max(1,|b|) = 1.07e-35; tolerance = 1e-25]
NUM  PASS  Eq.(16) sta

<a id="check-2"></a>
## Check 2 — Large-momentum Lindhard response

Starting from the exact logarithm in Eq. (23), set $w=1/q$, $\nu=q^2u$, and $A=u^2+1/4$ **locally in this cell**. Canceling $q^4$ in the ratio gives
$$\mathcal L(q,q^2u)=\frac w2\int_0^1 k\log\frac{A+wk+w^2k^2}{A-wk+w^2k^2}\,dk.$$
The code expands the logarithm, integrates each coefficient over $k$, and recovers Eq. (28). The logarithm is odd in $w$; multiplication by $w$ makes the response expansion even. Rescaling $w\mapsto x_\sigma w$ and multiplying the response weight $x_\sigma/2$ gives $M_3=1$ at order $q^{-2}$ and $M_5$ at order $q^{-4}$.

These are exact coefficient identities. The uniform remainder estimate is in Appendix C.

In [3]:
w, A, k, x = sp.symbols('w A k x', positive=True); u = sp.symbols('u', real=True)
Lw  = sp.log((A + w*k + w**2*k**2)/(A - w*k + w**2*k**2))     # Eq.(23) with nu = q^2 u, A = u^2+1/4, w = 1/q
ser = sp.series(Lw, w, 0, 6).removeO()                          # Taylor series in w = 1/q
lindhard_series   = sp.expand(sp.integrate(sp.expand(k*ser), (k, 0, 1))*w/2).subs(A, u**2 + R(1,4))   # (w/2) int_0^1 k log(...) dk
lindhard_coeff_2 = R(4,3)/(1+4*u**2); lindhard_coeff_4 = 16*(1-12*u**2)/(15*(1+4*u**2)**3)
exact("Eq.(28) q^-2 coefficient = 4/[3(1+4u^2)]", lindhard_series.coeff(w, 2) - lindhard_coeff_2)
exact("Eq.(28) q^-4 coefficient = 16(1-12u^2)/[15(1+4u^2)^3]", lindhard_series.coeff(w, 4) - lindhard_coeff_4)
exact("Eq.(28) no odd powers", lindhard_series.coeff(w, 1)**2 + lindhard_series.coeff(w, 3)**2)
exact("App.A   L4(u) = -(16/15)(12u^2-1)/(1+4u^2)^3 equals (4/[3(1+4u^2)]) g4 with g4=-4(12u^2-1)/[5(1+4u^2)^2]",
      lindhard_coeff_4 - lindhard_coeff_2*(-4*(12*u**2-1)/(5*(1+4*u**2)**2)))
# spin composite: L(q/x, nu/x^2) at fixed u = nu/q^2  is the same series with 1/q -> x/q
lindhard_scaled = (lindhard_coeff_2*w**2 + lindhard_coeff_4*w**4).subs(w, w*x)
exact("Eq.(24)/(28) x L(q/x, nu/x^2): leading term scales as x^3  =>  M_3", sp.expand(x*lindhard_scaled).coeff(w, 2) - x**3*lindhard_coeff_2)
exact("Eq.(24)/(28) x L(q/x, nu/x^2): next term scales as x^5     =>  M_5", sp.expand(x*lindhard_scaled).coeff(w, 4) - x**5*lindhard_coeff_4)
zeta = sp.symbols('zeta', real=True)
Mn = lambda n: ((1+zeta)**R(n,3) + (1-zeta)**R(n,3))/2          # Eq.(4)
exact("Eq.(4)  M_3(zeta) = 1 for all zeta", Mn(3) - 1)

EXACT  PASS  Eq.(28) q^-2 coefficient = 4/[3(1+4u^2)]   [simplify -> 0]
EXACT  PASS  Eq.(28) q^-4 coefficient = 16(1-12u^2)/[15(1+4u^2)^3]   [simplify -> 0]
EXACT  PASS  Eq.(28) no odd powers   [simplify -> 0]
EXACT  PASS  App.A   L4(u) = -(16/15)(12u^2-1)/(1+4u^2)^3 equals (4/[3(1+4u^2)]) g4 with g4=-4(12u^2-1)/[5(1+4u^2)^2]   [simplify -> 0]
EXACT  PASS  Eq.(24)/(28) x L(q/x, nu/x^2): leading term scales as x^3  =>  M_3   [simplify -> 0]
EXACT  PASS  Eq.(24)/(28) x L(q/x, nu/x^2): next term scales as x^5     =>  M_5   [simplify -> 0]
EXACT  PASS  Eq.(4)  M_3(zeta) = 1 for all zeta   [simplify -> 0]


<a id="check-3"></a>
## Check 3 — Same-sphere moment $I_0$

For $B=\{\mathbf p:|\mathbf p|<1\}$, the overlap volume at separation $r$ is
$$V(r)=\frac\pi{12}(4+r)(2-r)^2,\qquad 0\le r\le2.$$
Two spherical caps of height $1-r/2$ give this polynomial. With $\mathbf r=\mathbf p-\mathbf k$,
$$I_0=\int_{B^2}\frac{d^3p\,d^3k}{|\mathbf p-\mathbf k|^2}=4\pi\int_0^2V(r)dr=4\pi^2.$$
The cell derives the overlap polynomial and integrates it exactly. The two same-sphere sectors therefore carry $2I_0=8\pi^2$ in the leading profile (Eq. (30)).

In [4]:
r, h = sp.symbols('r h', positive=True)
cap = pi*h**2*(3-h)/3                      # volume of a spherical cap of height h (unit sphere)
V = pi*(4+r)*(2-r)**2/12                   # overlap volume of two unit balls at separation r (App. A)
exact("Eq.(30) overlap volume = 2 caps of height 1-r/2", 2*cap.subs(h, 1-r/2) - V)
exact("Eq.(30) V(0) = 4pi/3, V(2) = 0", (V.subs(r, 0) - 4*pi/3)**2 + V.subs(r, 2)**2)
exact("Eq.(30) 4 pi int_0^2 V(r) dr = 4 pi^2", 4*pi*sp.integrate(V, (r, 0, 2)) - 4*pi**2)

EXACT  PASS  Eq.(30) overlap volume = 2 caps of height 1-r/2   [simplify -> 0]
EXACT  PASS  Eq.(30) V(0) = 4pi/3, V(2) = 0   [simplify -> 0]
EXACT  PASS  Eq.(30) 4 pi int_0^2 V(r) dr = 4 pi^2   [simplify -> 0]


<a id="check-4"></a>
## Check 4 — Frequency profile $F(g)$

Here $g=(q_*/q)^4=x^{-4}$ and $A=iu-1/2$. At fixed $m=\lambda g>0$, put $s=\sqrt{1+m}$. The upper-half-plane screening pole gives
$$H_2(m)=\int_0^\infty\frac{du}{\pi}\frac{(1+4u^2)\Re A^{-2}}{1+4u^2+m}=-\frac{m}{s(s+1)^2}.$$
The cell tests this frequency integral numerically, then integrates $2\lambda H_2(\lambda g)$ symbolically to obtain
$$F(g)=-\frac43\frac{\sqrt{1+g}-1}{(\sqrt{1+g}+1)^2}.$$
It also verifies the bare cancellation using the derivative of $u/(u^2+1/4)$, the endpoint limits, and quadrature of the unreduced defining integral in Eq. (31). Check 31 derives the residue exactly.

In [5]:
g, lam, sS = sp.symbols('g lambda s', positive=True)
# (1) residue identity (*) tested by direct quadrature of the u-integral
for m in [mp.mpf('0.5'), mp.mpf(3), mp.mpf(40)]:
    lhs = mp.quad(lambda uu: (1+4*uu**2)/(1+4*uu**2+m)*(mp.mpf(1)/4-uu**2)/(mp.mpf(1)/4+uu**2)**2/mp.pi, [0, mp.mpf(1)/2, 2, mp.inf])
    sv = mp.sqrt(1+m)
    num("Eq.(31) residue identity, m=%s" % m, lhs, -m/(sv*(sv+1)**2), tol=1e-20)
# coupling average of the residue result (symbolic, as in the manuscript)
H2 = -(lam*g)/(sp.sqrt(1+lam*g)*(sp.sqrt(1+lam*g)+1)**2)
F_cl = -R(4,3)*(sp.sqrt(1+g)-1)/(sp.sqrt(1+g)+1)**2
exact("Eq.(31) 2 int lam dlam [-m/(s(s+1)^2)] = -(4/3)(sqrt(1+g)-1)/(sqrt(1+g)+1)^2", sp.integrate(2*lam*H2, (lam, 0, 1)) - F_cl)
# bare cancellation via the antiderivative
uu = sp.symbols('u', nonnegative=True)
exact("Eq.(32) (1/4-u^2)/(1/4+u^2)^2 = d/du [u/(1/4+u^2)]  =>  bare integral vanishes",
      sp.diff(uu/(R(1,4)+uu**2), uu) - (R(1,4)-uu**2)/(R(1,4)+uu**2)**2)
exact("Eq.(32) int_0^oo (1/4-u^2)/(1/4+u^2)^2 du = 0", sp.limit(uu/(R(1,4)+uu**2), uu, oo) - 0)
# endpoints of the scaled profile
exact("Eq.(31) x^-2 F(x^-4) -> -4/3 as x -> 0", sp.limit(F_cl.subs(g, x**-4)/x**2, x, 0) + R(4,3))
exact("Eq.(31) x^-2 F(x^-4) = O(x^-6): x^4 F(x^-4) -> -1/6  (F(g) = -g/6 + O(g^2))", sp.limit(x**4*F_cl.subs(g, x**-4), x, oo) + R(1,6))
# (2) the defining integral of Eq.(31) with R inside, integrated as written
F_n = lambda gv: -mp.mpf(4)/3*(mp.sqrt(1+gv)-1)/(mp.sqrt(1+gv)+1)**2
for gv in [mp.mpf('0.3'), mp.mpf(7)]:
    direct = mp.quad(lambda v: (Rn(gv/(1+4*v**2))-1)*(mp.mpf(1)/4-v**2)/(mp.mpf(1)/4+v**2)**2/mp.pi, [0, mp.mpf(1)/2, 2, mp.inf])
    num("Eq.(31) full u-integral with R inside = F(g), g=%s" % gv, direct, F_n(gv), tol=1e-20)

NUM  PASS  Eq.(31) residue identity, m=0.5   [|a-b|/max(1,|b|) = 2.47e-32; tolerance = 1e-20]
NUM  PASS  Eq.(31) residue identity, m=3.0   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-20]
NUM  PASS  Eq.(31) residue identity, m=40.0   [|a-b|/max(1,|b|) = 2.47e-32; tolerance = 1e-20]
EXACT  PASS  Eq.(31) 2 int lam dlam [-m/(s(s+1)^2)] = -(4/3)(sqrt(1+g)-1)/(sqrt(1+g)+1)^2   [simplify -> 0]
EXACT  PASS  Eq.(32) (1/4-u^2)/(1/4+u^2)^2 = d/du [u/(1/4+u^2)]  =>  bare integral vanishes   [simplify -> 0]
EXACT  PASS  Eq.(32) int_0^oo (1/4-u^2)/(1/4+u^2)^2 du = 0   [simplify -> 0]
EXACT  PASS  Eq.(31) x^-2 F(x^-4) -> -4/3 as x -> 0   [simplify -> 0]
EXACT  PASS  Eq.(31) x^-2 F(x^-4) = O(x^-6): x^4 F(x^-4) -> -1/6  (F(g) = -g/6 + O(g^2))   [simplify -> 0]
NUM  PASS  Eq.(31) full u-integral with R inside = F(g), g=0.3   [|a-b|/max(1,|b|) = 1.85e-32; tolerance = 1e-20]
NUM  PASS  Eq.(31) full u-integral with R inside = F(g), g=7.0   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-20]


<a id="check-5"></a>
## Check 5 — Leading coefficient $A_1$

The same-sphere factor gives $32\pi^3A_0=3/\pi^2$, so
$$A_1=\frac3{\pi^2}\int_0^\infty x^{-2}F(x^{-4})dx
=-\frac4{7\sqrt2\pi^{3/2}}\frac{\Gamma(1/4)}{\Gamma(3/4)}.$$
Direct $x$ quadrature tests the defining integral independently of the beta substitution. Exact gamma-function identities reduce $B(5/4,1/2)$ and assemble $A_1$. The conversion $q_*=(4ar_s/3)^{1/4}$ then checks the printed coefficient $-0.2213782293$ in Eq. (6). Decimal tolerances match the displayed precision.

In [6]:
exact("Eq.(33) 32 pi^3 A0 = 3/pi^2", 32*pi**3*A0 - 3/pi**2)
I_num = mp.quad(lambda xx: F_n(xx**-4)/xx**2, [0, mp.mpf(1)/2, 1, 2, mp.inf])       # Eq.(33) integral, no substitution
I_beta = -4/(7*mp.sqrt(2))*mp.beta(mp.mpf(5)/4, mp.mpf(1)/2)                          # Eq.(A.1)
num("Eq.(A.1) int F(x^-4)/x^2 dx = -(4/(7 sqrt2)) B(5/4,1/2)", I_num, I_beta, tol=1e-20)
Bv = sp.beta(R(5,4), R(1,2))
exact("Eq.(A.1) -(4/(7 sqrt2)) B(5/4,1/2) = -(4 sqrt(pi)/(21 sqrt2)) Gamma(1/4)/Gamma(3/4)",
      sp.gammasimp(sp.expand_func(-4/(7*sp.sqrt(2))*Bv)) + 4*sp.sqrt(pi)/(21*sp.sqrt(2))*sp.gamma(R(1,4))/sp.gamma(R(3,4)))
A1_s = -4/(7*sp.sqrt(2)*pi**R(3,2))*sp.gamma(R(1,4))/sp.gamma(R(3,4))
exact("Eq.(34) A1 = (3/pi^2) * Beta form", sp.gammasimp(sp.expand_func(3/pi**2*(-4/(7*sp.sqrt(2))*Bv)) - A1_s))
A1 = mp.mpf(sp.N(A1_s, 30))
num("Eq.(34) A1 = -0.21469376098", A1, mp.mpf('-0.21469376098'), tol=1e-11)
alphaL_mp = (4/(9*mp.pi))**(mp.mpf(1)/3); a_mp = 4*alphaL_mp/mp.pi                                  # alpha_L, a (Eq. 19)
num("Eq.(6)  A1 (4a/3)^(-1/4) = -0.2213782293", A1/(4*a_mp/3)**mp.mpf(0.25), mp.mpf('-0.2213782293'), tol=1e-9)
# Cross-paper check of A_1: the companion paper's dispersion route (s_lambda_coupling) is an independent integral
_s_lambda = lambda t: mp.sqrt(1 + t**-4)/2
_I_jpa = mp.quad(lambda t: _s_lambda(t)/((1 + 2*_s_lambda(t))**2*t**2*(t**4 + 1)), [0, mp.mpf(1)/2, 1, 2, mp.inf])
num("Eqs.(33),(34) A1 = (3/pi^2) int F(x^-4)/x^2 dx equals the companion route -(48/(7 pi^2)) int s_lambda/((1+2mu)^2 t^2 (t^4+1)) dt",
    3/mp.pi**2*I_num, -48/(7*mp.pi**2)*_I_jpa, tol=1e-20)


EXACT  PASS  Eq.(33) 32 pi^3 A0 = 3/pi^2   [simplify -> 0]
NUM  PASS  Eq.(A.1) int F(x^-4)/x^2 dx = -(4/(7 sqrt2)) B(5/4,1/2)   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-20]
EXACT  PASS  Eq.(A.1) -(4/(7 sqrt2)) B(5/4,1/2) = -(4 sqrt(pi)/(21 sqrt2)) Gamma(1/4)/Gamma(3/4)   [simplify -> 0]
EXACT  PASS  Eq.(34) A1 = (3/pi^2) * Beta form   [simplify -> 0]
NUM  PASS  Eq.(34) A1 = -0.21469376098   [|a-b|/max(1,|b|) = 9.1e-12; tolerance = 1e-11]
NUM  PASS  Eq.(6)  A1 (4a/3)^(-1/4) = -0.2213782293   [|a-b|/max(1,|b|) = 8.79e-12; tolerance = 1e-09]
NUM  PASS  Eqs.(33),(34) A1 = (3/pi^2) int F(x^-4)/x^2 dx equals the companion route -(48/(7 pi^2)) int s_lambda/((1+2mu)^2 t^2 (t^4+1)) dt   [|a-b|/max(1,|b|) = 2.47e-32; tolerance = 1e-20]


<a id="check-6"></a>
## Check 6 — Fixed-momentum frequency factor $K_0$

At fixed momentum, the leading high-frequency response is $\mathcal L=q^2/(3\nu^2)$. Equation (35) gives
$$K_0=-\int_0^\infty\frac{d\nu}{\pi\nu^2}R\left(\frac{q_{\rm TF}^{2}}{3\nu^2}\right)=-\frac2{\sqrt{3q_{\rm TF}^{2}}}=-\frac4{3q_*^2}.$$
The cell integrates frequency at fixed coupling, then the coupling weight, and tests the defining frequency integral numerically. The bounded-momentum error is estimated separately in Appendix C.

In [7]:
q_tf_sq, nu = sp.symbols('q_tf_sq nu', positive=True)
inner = sp.integrate(1/(pi*nu**2*(1 + lam*q_tf_sq/(3*nu**2))), (nu, 0, oo))               # nu-integral at fixed lambda
exact("Eq.(35) int dnu/(pi nu^2) [1+lam q_tf_sq/(3 nu^2)]^-1 = (1/2) sqrt(3/(lam q_tf_sq))", inner - sp.sqrt(3/(lam*q_tf_sq))/2)
K0 = -sp.integrate(2*lam*inner, (lam, 0, 1))                                              # coupling average
exact("Eq.(35) K0 = -2/sqrt(3 q_tf_sq)", K0 + 2/sp.sqrt(3*q_tf_sq))
qs4 = R(4,3)*q_tf_sq
exact("Eq.(35) -2/sqrt(3 q_tf_sq) = -4/(3 q*^2) with q*^4 = 4 q_tf_sq/3", -2/sp.sqrt(3*q_tf_sq) + 4/(3*sp.sqrt(qs4)))
tv = mp.mpf('2.7')
num("Eq.(35) quadrature with R inside, q_tf_sq=2.7", -mp.quad(lambda xx: Rn(tv/(3*xx**2))/(mp.pi*xx**2), [0, 1, mp.inf]), -2/mp.sqrt(3*tv), tol=1e-20)

EXACT  PASS  Eq.(35) int dnu/(pi nu^2) [1+lam q_tf_sq/(3 nu^2)]^-1 = (1/2) sqrt(3/(lam q_tf_sq))   [simplify -> 0]
EXACT  PASS  Eq.(35) K0 = -2/sqrt(3 q_tf_sq)   [simplify -> 0]
EXACT  PASS  Eq.(35) -2/sqrt(3 q_tf_sq) = -4/(3 q*^2) with q*^4 = 4 q_tf_sq/3   [simplify -> 0]
NUM  PASS  Eq.(35) quadrature with R inside, q_tf_sq=2.7   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-20]


<a id="check-7"></a>
## Check 7 — Geometric finite part and $A_2$

For $\mathbf r=\mathbf p-\mathbf k=(\boldsymbol\rho,z)$, integrate the Coulomb denominator over $q$ first:
$$\int_0^\infty\frac{dq}{\rho^2+(q+z)^2}=\frac{\pi/2-\arctan(z/\rho)}{\rho}.$$
The arctangent is odd in $z$. Integrating the remaining overlap volume gives
$$\int_0^\infty C(q)dq=\pi^3\int_0^2rV(r)dr=\frac{8\pi^4}{15}.$$
Thus $Z=8\pi^2-2C$, $\operatorname{Fp}\int Z=-16\pi^4/15$, and $A_2=8/15$ (Eqs. (36)–(38)). The code checks these integrations exactly.

An independent angular-first reduction is
$$C(q)=\frac\pi q\int_0^2rV(r)\log\frac{(q+r)^2}{(q-r)^2}dr.$$
Quadrature splits at the integrable singularity $r=q$, checks $q^2C(q)\to V_B^2$, and integrates over $q$ for comparison with the exact result.

In [8]:
qv, rho, zz, th = sp.symbols('q rho z theta', positive=True)
# --- the manuscript's route (q-integral first), step by step
exact("Eq.(A.2) int_0^oo dq/(rho^2+(q+z)^2) = [pi/2 - arctan(z/rho)]/rho",
      sp.integrate(1/(rho**2 + (qv+zz)**2), (qv, 0, oo)) - (pi/2 - sp.atan(zz/rho))/rho)
exact("Eq.(A.2) int dOmega / sin(theta) = 2 pi^2", 2*pi*sp.integrate(1/sp.sin(th)*sp.sin(th), (th, 0, pi)) - 2*pi**2)
IC = pi**3*sp.integrate(r*V, (r, 0, 2))
exact("Eq.(A.2) int_0^oo C dq = pi^3 int r V dr = 8 pi^4/15", IC - 8*pi**4/15)
exact("Eq.(37) Fp int Z dq = -2 int C dq = -16 pi^4/15", -2*IC + 16*pi**4/15)
exact("Eq.(38) A2 = 4 pi A0 (-4/3)(-16 pi^4/15) = 8/15", 4*pi*A0*(-R(4,3))*(-16*pi**4/15) - R(8,15))
exact("App.A  int_0^2 r^2 V(r) dr = 4 pi/9 = V_B^2/(4 pi)", sp.integrate(r**2*V, (r, 0, 2)) - 4*pi/9)
# --- independent route: angular integral first, C(q) as a 1-D integral, then quadrature over q
Vn = lambda rr: mp.pi*(4+rr)*(2-rr)**2/12
def C_of_q(q):   # C(q) = int d^3r V(r)/|q zhat + r|^2 = 2 pi int_0^2 r^2 V(r) [ln((q+r)^2/(q-r)^2)/(2 q r)] dr
    def ang(rr):
        if q < mp.mpf('1e-10'): return 2/rr**2
        if rr == q: return mp.mpf(0)                       # integrable log singularity; single point of measure zero
        return mp.log((q+rr)**2/(q-rr)**2)/(2*q*rr)
    pts = [0, q, 2] if 0 < q < 2 else [0, 2]
    return 2*mp.pi*mp.quad(lambda rr: rr**2*Vn(rr)*ang(rr), pts)
num("Eq.(44)/App.A q^2 C(q) -> V_B^2 = (4pi/3)^2 at q=200", mp.mpf(200)**2*C_of_q(mp.mpf(200)), (4*mp.pi/3)**2, tol=1e-4)
num("Eq.(A.2) int_0^oo C dq by quadrature = 8 pi^4/15", mp.quad(C_of_q, [0, 2, 4, 20, mp.inf]), 8*mp.pi**4/15, tol=1e-15)

EXACT  PASS  Eq.(A.2) int_0^oo dq/(rho^2+(q+z)^2) = [pi/2 - arctan(z/rho)]/rho   [simplify -> 0]
EXACT  PASS  Eq.(A.2) int dOmega / sin(theta) = 2 pi^2   [simplify -> 0]
EXACT  PASS  Eq.(A.2) int_0^oo C dq = pi^3 int r V dr = 8 pi^4/15   [simplify -> 0]
EXACT  PASS  Eq.(37) Fp int Z dq = -2 int C dq = -16 pi^4/15   [simplify -> 0]
EXACT  PASS  Eq.(38) A2 = 4 pi A0 (-4/3)(-16 pi^4/15) = 8/15   [simplify -> 0]
EXACT  PASS  App.A  int_0^2 r^2 V(r) dr = 4 pi/9 = V_B^2/(4 pi)   [simplify -> 0]
NUM  PASS  Eq.(44)/App.A q^2 C(q) -> V_B^2 = (4pi/3)^2 at q=200   [|a-b|/max(1,|b|) = 1.0e-5; tolerance = 0.0001]
NUM  PASS  Eq.(A.2) int_0^oo C dq by quadrature = 8 pi^4/15   [|a-b|/max(1,|b|) = 7.46e-27; tolerance = 1e-15]


<a id="check-8"></a>
## Check 8 — Kinematic moment $I_2$

The required moment is $I_2=52\pi^2/27$ (Eq. (39)). An independent reduction uses the ball potential
$$\phi(p)=\frac{2\pi}{p}\int_0^1 k\log\left|\frac{p+k}{p-k}\right|dk.$$
Let $U_2=(4\pi/3)\int_0^1p^4\phi(p)dp$ and $D_2=(2\pi/3)\int_0^2r^2V(r)dr$. Rotational symmetry and $2p_zk_z=p_z^2+k_z^2-(p_z-k_z)^2$ give $I_2=3U_2-D_2$.

The cell evaluates this reduction by quadrature and independently recovers $I_0$ from $\phi$. Check 24 derives the quadratic denominator combination; Check 31 integrates the lens moments with exact arithmetic.

In [9]:
def phi(p):     # phi(p) = int_B d^3k / |p-k|^2 = (2 pi/p) int_0^1 k ln|(p+k)/(p-k)| dk
    f = lambda kk: kk*mp.log((p+kk)/abs(p-kk)) if kk != p else mp.mpf(0)
    return 2*mp.pi/p*mp.quad(f, [0, p, 1])
I0n = 4*mp.pi*mp.quad(lambda p: p**2*phi(p), [0, 1])          # zeroth moment: must reproduce 4 pi^2 (Check 3)
U2  = 4*mp.pi/3*mp.quad(lambda p: p**4*phi(p), [0, 1])        # int p_z^2/|p-k|^2  (angular average p_z^2 -> p^2/3)
D2  = 2*mp.pi/3*mp.quad(lambda rr: rr**2*Vn(rr), [0, 2])      # (1/2) int d^3r V(r) r_z^2/r^2
num("Eq.(30) zeroth moment = 4 pi^2 (consistency of the phi(p) route)", I0n, 4*mp.pi**2, tol=1e-18)
num("Eq.(39) int (p_z^2 + k_z^2 + p_z k_z)/|p-k|^2 = 3 U2 - D2 = 52 pi^2/27", 3*U2 - D2, 52*mp.pi**2/27, tol=1e-18)
print("U2 = int p_z^2/|p-k|^2 =", mp.nstr(U2, 15), " (= 20 pi^2/27 =", mp.nstr(20*mp.pi**2/27, 15), ")")

NUM  PASS  Eq.(30) zeroth moment = 4 pi^2 (consistency of the phi(p) route)   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-18]
NUM  PASS  Eq.(39) int (p_z^2 + k_z^2 + p_z k_z)/|p-k|^2 = 3 U2 - D2 = 52 pi^2/27   [|a-b|/max(1,|b|) = 1.66e-31; tolerance = 1e-18]
U2 = int p_z^2/|p-k|^2 = 7.31081807488101  (= 20 pi^2/27 = 7.31081807488101 )


<a id="check-9"></a>
## Check 9 — Cross-sphere coefficient

The coupling integral gives $J_C(g)=2\int_0^1\lambda(1+\lambda g)^{-1/2}d\lambda$ (Eq. (40)). Its radial profile starts as $4/(3x^2)-2x^2+O(x^4)$ at zero. The first term belongs to fixed-momentum matching.

The cell verifies the coupling integral, endpoint series, Jacobian for $s=\sqrt{1+x^{-4}}$, and beta/gamma reduction. Numerically it evaluates the finite part with an explicit subtraction:
$$\operatorname{Fp}\int_0^\infty f(x)dx
=\int_0^1\left[f(x)-\frac4{3x^2}\right]dx-\frac43+\int_1^\infty f(x)dx,$$
where $f=x^{-4}J_C(x^{-4})$. Series evaluation near zero avoids cancellation. The result is $A_{3,\rm cross}=-(32/15)C_\Gamma$.

Appendix A and Check 31 establish the subtraction convention by integration by parts, reducing the continued beta value to a convergent integral.

In [10]:
JC_cl = 4/(3*g**2)*((1+g)**R(3,2) - 3*sp.sqrt(1+g) + 2)
exact("Eq.(40) 2 int lam dlam/sqrt(1+lam g) = (4/3g^2)[(1+g)^(3/2) - 3(1+g)^(1/2) + 2]", sp.integrate(2*lam/sp.sqrt(1+lam*g), (lam, 0, 1)) - JC_cl)
exact("Eq.(40) J_C(0) = 1", sp.limit(JC_cl, g, 0) - 1)
exact("Eq.(40) J_C(g) = 1 - g/3 + 3g^2/16 - g^3/8 + 35g^4/384 + O(g^5)  [series used by JCn below]",
      sp.series(JC_cl, g, 0, 5).removeO() - (1 - g/3 + 3*g**2/16 - g**3/8 + 35*g**4/384))
prof = (x**-4*JC_cl.subs(g, x**-4))
exact("Eq.(41) x^-4 J_C = 4/(3x^2) - 2x^2 + O(x^4) as x->0", sp.series(prof, x, 0, 4).removeO() - (R(4,3)/x**2 - 2*x**2))
exact("Eq.(41) x^-4 J_C = 4/(3x^2) - 2x^2 + 8x^4/3 - 3x^6/2 + 5x^10/12 + O(x^12)  [series used by hsub below]",
      sp.series(prof, x, 0, 12).removeO() - (R(4,3)/x**2 - 2*x**2 + R(8,3)*x**4 - R(3,2)*x**6 + R(5,12)*x**10))
exact("Eq.(41) x^-4 J_C = x^-4 + O(x^-8) as x->oo", sp.series(prof, x, oo, 8).removeO() - x**-4)
# s = sqrt(1+x^-4) substitution, App. A
exact("Eq.(A.5) x^-4 J_C = (4/3)(s-1)(s+2)/(s+1) with s=sqrt(1+x^-4)", sp.simplify((g*JC_cl).subs(g, sS**2-1)) - R(4,3)*(sS-1)*(sS+2)/(sS+1))
exact("Eq.(A.5) dx/ds = -(s/2)(s^2-1)^(-5/4)", sp.diff((sS**2-1)**R(-1,4), sS) + sS/2*(sS**2-1)**R(-5,4))
exact("Eq.(A.5) (2/3) s(s+2)(s-1)^(-1/4)(s+1)^(-9/4) = (2/3)[(s^2-1)^(-1/4) - (s-1)^(-1/4)(s+1)^(-9/4)]",
      R(2,3)*sS*(sS+2)*(sS-1)**R(-1,4)*(sS+1)**R(-9,4) - R(2,3)*((sS**2-1)**R(-1,4) - (sS-1)**R(-1,4)*(sS+1)**R(-9,4)))
B1 = sp.beta(R(3,4), -R(1,2)); B2 = sp.beta(R(3,4), R(3,2))    # B(3/4,-1/2) is the analytic continuation
Fp_s = R(2,3)*(sp.sqrt(2)*B1 - 2**R(-3,2)*B2)
Fp_t = -R(8,5)*sp.sqrt(2*pi)*sp.gamma(R(3,4))/sp.gamma(R(1,4))
exact("Eq.(A.5) (2/3)[2^(1/2) B(3/4,-1/2) - 2^(-3/2) B(3/4,3/2)] = -(8/5) sqrt(2pi) Gamma(3/4)/Gamma(1/4)", sp.gammasimp(sp.expand_func(Fp_s - Fp_t)))
CG = sp.sqrt(2*pi)/pi**2*sp.gamma(R(3,4))/sp.gamma(R(1,4))     # C_Gamma, Eq.(A.3)
exact("Eq.(A.4) 8 pi A0 V_B^2 = 4/(3 pi^2)", 8*pi*A0*(4*pi/3)**2 - 4/(3*pi**2))
exact("Eq.(A.4) A3,cross = -(32/15) C_Gamma", 8*pi*A0*(4*pi/3)**2*Fp_t + R(32,15)*CG)
# --- (1) u-integral with R inside
def JCn(gv):
    gv = mp.mpf(gv)
    return 1 - gv/3 + 3*gv**2/16 - gv**3/8 + 35*gv**4/384 if gv < mp.mpf('1e-5') else 4/(3*gv**2)*((1+gv)**mp.mpf(1.5) - 3*mp.sqrt(1+gv) + 2)
num("Eq.(40) u-integral with R inside = J_C(g), g=2", mp.quad(lambda v: Rn(2/(1+4*v**2))/(v**2+mp.mpf(1)/4)/mp.pi, [0, 1, mp.inf]), JCn(2), tol=1e-20)
# --- (2) Hadamard finite part in x:  int_0^1 [f - 4/(3x^2)] dx - 4/3 + int_1^oo f dx
fprof = lambda xx: xx**-4*JCn(xx**-4)
hsub  = lambda xx: (-2*xx**2 + 8*xx**4/3 - 3*xx**6/2 + 5*xx**10/12) if xx < mp.mpf('0.02') else fprof(xx) - mp.mpf(4)/(3*xx**2)
Fp_num = mp.quad(hsub, [0, mp.mpf(1)/2, 1], method='gauss-legendre') - mp.mpf(4)/3 + mp.quad(fprof, [1, 2, 4, mp.inf])
num("Eq.(A.5) Hadamard finite part in x = analytic continuation of the Beta functions", Fp_num, mp.mpf(sp.N(Fp_t, 30)), tol=1e-18)
print("A3,cross =", sp.N(-R(32,15)*CG, 15))

EXACT  PASS  Eq.(40) 2 int lam dlam/sqrt(1+lam g) = (4/3g^2)[(1+g)^(3/2) - 3(1+g)^(1/2) + 2]   [simplify -> 0]
EXACT  PASS  Eq.(40) J_C(0) = 1   [simplify -> 0]
EXACT  PASS  Eq.(40) J_C(g) = 1 - g/3 + 3g^2/16 - g^3/8 + 35g^4/384 + O(g^5)  [series used by JCn below]   [simplify -> 0]
EXACT  PASS  Eq.(41) x^-4 J_C = 4/(3x^2) - 2x^2 + O(x^4) as x->0   [simplify -> 0]
EXACT  PASS  Eq.(41) x^-4 J_C = 4/(3x^2) - 2x^2 + 8x^4/3 - 3x^6/2 + 5x^10/12 + O(x^12)  [series used by hsub below]   [simplify -> 0]
EXACT  PASS  Eq.(41) x^-4 J_C = x^-4 + O(x^-8) as x->oo   [simplify -> 0]
EXACT  PASS  Eq.(A.5) x^-4 J_C = (4/3)(s-1)(s+2)/(s+1) with s=sqrt(1+x^-4)   [simplify -> 0]
EXACT  PASS  Eq.(A.5) dx/ds = -(s/2)(s^2-1)^(-5/4)   [simplify -> 0]
EXACT  PASS  Eq.(A.5) (2/3) s(s+2)(s-1)^(-1/4)(s+1)^(-9/4) = (2/3)[(s^2-1)^(-1/4) - (s-1)^(-1/4)(s+1)^(-9/4)]   [simplify -> 0]
EXACT  PASS  Eq.(A.5) (2/3)[2^(1/2) B(3/4,-1/2) - 2^(-3/2) B(3/4,3/2)] = -(8/5) sqrt(2pi) Gamma(3/4)/Gamma(1/4)   [simplify -> 0]
EXACT

<a id="check-10"></a>
## Check 10 — Kinematic coefficient

At fixed $m=\lambda g$ and $s=\sqrt{1+m}$, the frequency integral in Eq. (A.9) is
$$H_4(m)=-\frac{4m}{s(s+1)^4}.$$
The cell compares it with direct quadrature. The changes $g=x^{-4}$ and $g=t/\lambda$ give the coupling factor $\int_0^1\lambda^{1/4}d\lambda=4/5$; $s=\sqrt{1+t}$ reduces the remaining integral to $-2\sqrt2 B(7/4,3/2)$.

Exact gamma algebra and the two-sector moment $2I_2=104\pi^2/27$ then give $A_{3,\rm kin}=-(416/675)C_\Gamma$. The contour residue is derived symbolically in Check 31.

In [11]:
Acx = lambda uu: 1j*uu - mp.mpf(1)/2                     # A = i u - 1/2
for m in [mp.mpf('0.5'), mp.mpf(3), mp.mpf(40)]:
    lhs = mp.quad(lambda uu: (1+4*uu**2)/(1+4*uu**2+m)*(Acx(uu)**-4).real/mp.pi, [0, mp.mpf(1)/2, 2, mp.inf])
    sv = mp.sqrt(1+m)
    num("Eq.(A.9) kinematic residue identity, m=%s" % m, lhs, -4*m/(sv*(sv+1)**4), tol=1e-20)
exact("Eq.(A.10) int_0^1 lam^(1/4) dlam = 4/5", sp.integrate(lam**R(1,4), (lam, 0, 1)) - R(4,5))
t = sp.symbols('t', positive=True)
Ik_num = mp.quad(lambda tt: tt**(-mp.mpf(1)/4)*(-4*tt)/(mp.sqrt(1+tt)*(mp.sqrt(1+tt)+1)**4), [0, 1, mp.inf])   # Eq.(A.10) as written
num("Eq.(A.10) int t^(-1/4)(-4t)/[sqrt(1+t)(sqrt(1+t)+1)^4] dt = -2 sqrt2 B(7/4,3/2)", Ik_num, -2*mp.sqrt(2)*mp.beta(mp.mpf(7)/4, mp.mpf(3)/2), tol=1e-20)
exact("Eq.(A.10) -8 int_1^oo (s-1)^(3/4)(s+1)^(-13/4) ds = -8 * 2^(-3/2) B(7/4,3/2) = -2 sqrt2 B(7/4,3/2)",
      -8*2**(R(3,4)-R(13,4)+1)*sp.beta(R(7,4), R(3,2)) + 2*sp.sqrt(2)*sp.beta(R(7,4), R(3,2)))
Bk = sp.beta(R(7,4), R(3,2))
exact("Eq.(A.10) B(7/4,3/2) = (8/15) sqrt(pi) Gamma(3/4)/Gamma(1/4)", sp.gammasimp(sp.expand_func(Bk)) - R(8,15)*sp.sqrt(pi)*sp.gamma(R(3,4))/sp.gamma(R(1,4)))
Akin = 4*pi*A0*(104*pi**2/27)*R(2,5)*(-2*sp.sqrt(2)*Bk)
exact("Eq.(A.10) A3,kin = -(416/675) C_Gamma", sp.gammasimp(sp.expand_func(Akin + R(416,675)*CG)))
print("A3,kin =", sp.N(-R(416,675)*CG, 15))

NUM  PASS  Eq.(A.9) kinematic residue identity, m=0.5   [|a-b|/max(1,|b|) = 3.7e-32; tolerance = 1e-20]
NUM  PASS  Eq.(A.9) kinematic residue identity, m=3.0   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-20]
NUM  PASS  Eq.(A.9) kinematic residue identity, m=40.0   [|a-b|/max(1,|b|) = 3.08e-33; tolerance = 1e-20]
EXACT  PASS  Eq.(A.10) int_0^1 lam^(1/4) dlam = 4/5   [simplify -> 0]
NUM  PASS  Eq.(A.10) int t^(-1/4)(-4t)/[sqrt(1+t)(sqrt(1+t)+1)^4] dt = -2 sqrt2 B(7/4,3/2)   [|a-b|/max(1,|b|) = 1.21e-25; tolerance = 1e-20]
EXACT  PASS  Eq.(A.10) -8 int_1^oo (s-1)^(3/4)(s+1)^(-13/4) ds = -8 * 2^(-3/2) B(7/4,3/2) = -2 sqrt2 B(7/4,3/2)   [simplify -> 0]
EXACT  PASS  Eq.(A.10) B(7/4,3/2) = (8/15) sqrt(pi) Gamma(3/4)/Gamma(1/4)   [simplify -> 0]
EXACT  PASS  Eq.(A.10) A3,kin = -(416/675) C_Gamma   [simplify -> 0]
A3,kin = -0.0529032639114349


<a id="check-11"></a>
## Check 11 — Lindhard coefficient and Euler integral

The response correction is $\delta c=(3/4)g\mathcal L_4(u)/q^2$, with $\mathcal L_4=-16(12u^2-1)/[15(1+4u^2)^3]$. Linearizing $R$ gives the profile $\Psi$ in Eq. (A.11). Its contour has a simple pole at $i/2$ and a double pole at $i\sqrt{1+m}/2$; their **sum** has limit $1/5$ as $m\to0$.

The cell tests the contour integral numerically and checks the $s=1/w$ Jacobian with $0<w<1$ enforced by $w=1/(1+v)$, $v>0$. The resulting Euler integral is
$$M_{\rm E}=\frac{16}{15}\int_0^1\frac{(1-w)^{3/4}(2-w+4w^2+w^3)}{\sqrt w(1+w)^{13/4}}dw
=\frac{176}{75}\sqrt{2\pi}\frac{\Gamma(3/4)}{\Gamma(1/4)}.$$
Substituting $w=v^2$ removes the inverse-square-root endpoint; quadrature tests the closed value to 22 digits. It gives $A_{3,\rm L}=-(264/125)C_\Gamma$.

The paper now uses the total-derivative proof reproduced in Check 31. The extra numerical checks here retain an alternative hypergeometric reduction: $T_j=B(j-1/2,7/4){}_2F_1(13/4,j-1/2;j+5/4;-1)$ and $M_{\rm E}=(16/15)(2T_1-T_2+4T_3+T_4)$. These auxiliary $T_j$ are used only in this notebook.

The Python variable `M_num` denotes the numerical value of $M_{\rm E}$.

In [12]:
exact("Eq.(A.11) L4(u) as stated equals the q^-4 Lindhard coefficient of Check 2", lindhard_coeff_4 - (-16*(12*u**2-1)/(15*(1+4*u**2)**3)))
exact("Eq.(18) R'(c) = -2 int lam^2 dlam/(1+lam c)^2", sp.diff(Rc, c) - sp.integrate(-2*lam**2/(1+lam*c)**2, (lam, 0, 1)))
# (1) double-pole contour identity tested by direct quadrature
for m in [mp.mpf('0.5'), mp.mpf(3), mp.mpf(40)]:
    lhs = mp.quad(lambda uu: -mp.mpf(16)/15*(12*uu**2-1)*(Acx(uu)**-2).real/((1+4*uu**2)*(1+4*uu**2+m)**2)/mp.pi, [0, mp.mpf(1)/2, 2, mp.inf])
    sv = mp.sqrt(1+m)
    num("Eq.(A.12) Lindhard-correction contour identity, m=%s" % m, lhs, 8*(2*sv**3-sv**2+4*sv+1)/(15*sv**3*(sv+1)**4), tol=1e-20)
# (2) s -> 1/w transformation of the s-integrand (ratio must be 1); w = 1/(1+v) encodes 0<w<1 for SymPy
w_ = sp.symbols('w', positive=True)
fs = (sS**2-1)**R(3,4)*(2*sS**3-sS**2+4*sS+1)/(sS**2*(sS+1)**4)
fw = (1-w_)**R(3,4)*(2-w_+4*w_**2+w_**3)/(sp.sqrt(w_)*(1+w_)**R(13,4))
v_ = sp.symbols('v', positive=True)
ratio = sp.expand_power_base((fs.subs(sS, 1/w_)/w_**2/fw).subs(w_, 1/(1+v_)), force=True)
exact("Eq.(A.14) (s^2-1)^(3/4)(2s^3-s^2+4s+1)/(s^2(s+1)^4) ds with s=1/w gives the w-integrand (ratio = 1)", sp.powdenest(ratio, force=True) - 1)
# (3) Euler integral M, evaluated after w = v^2 (removes the w^(-1/2) endpoint factor)
M_num = mp.mpf(16)/15*mp.quad(lambda v: 2*(1-v**2)**(mp.mpf(3)/4)*(2-v**2+4*v**4+v**6)/((1+v**2)**(mp.mpf(13)/4)), [0, 1])
U = mp.sqrt(2*mp.pi)*mp.gamma(mp.mpf(3)/4)/mp.gamma(mp.mpf(1)/4)
num("Eq.(A.17) M by quadrature (w=v^2) = (176/75) sqrt(2pi) Gamma(3/4)/Gamma(1/4)", M_num, mp.mpf(176)/75*U, tol=1e-22)
# T_j = B(a_j,7/4) 2F1(13/4, a_j; a_j+7/4; -1), a_j = j - 1/2   (Euler's integral)
Tj = lambda j: mp.beta(j-mp.mpf(1)/2, mp.mpf(7)/4)*mp.hyp2f1(mp.mpf(13)/4, j-mp.mpf(1)/2, j-mp.mpf(1)/2+mp.mpf(7)/4, -1)
num("Eq.(A.17) T1/U = 16/15", Tj(1)/U, mp.mpf(16)/15, tol=1e-20)
num("Eq.(A.17) T2/U = 2/15",  Tj(2)/U, mp.mpf(2)/15,  tol=1e-20)
num("Eq.(A.17) (4 T3 + T4)/U = 1/5", (4*Tj(3)+Tj(4))/U, mp.mpf(1)/5, tol=1e-20)
num("Eq.(A.17) (16/15)(2 T1 - T2 + 4 T3 + T4) = M by quadrature", mp.mpf(16)/15*(2*Tj(1) - Tj(2) + 4*Tj(3) + Tj(4)), M_num, tol=1e-20)
exact("Eq.(A.17) (16/15)(2*16/15 - 2/15 + 1/5) = 176/75", R(16,15)*(2*R(16,15) - R(2,15) + R(1,5)) - R(176,75))
AL = -R(48,5)*pi**3*A0*R(176,75)*sp.sqrt(2*pi)*sp.gamma(R(3,4))/sp.gamma(R(1,4))
exact("Eq.(A.17) A3,L = -(48/5) pi^3 A0 M = -(264/125) C_Gamma", AL + R(264,125)*CG)
print("A3,L =", sp.N(-R(264,125)*CG, 15))

EXACT  PASS  Eq.(A.11) L4(u) as stated equals the q^-4 Lindhard coefficient of Check 2   [simplify -> 0]
EXACT  PASS  Eq.(18) R'(c) = -2 int lam^2 dlam/(1+lam c)^2   [simplify -> 0]
NUM  PASS  Eq.(A.12) Lindhard-correction contour identity, m=0.5   [|a-b|/max(1,|b|) = 6.16e-32; tolerance = 1e-20]
NUM  PASS  Eq.(A.12) Lindhard-correction contour identity, m=3.0   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-20]
NUM  PASS  Eq.(A.12) Lindhard-correction contour identity, m=40.0   [|a-b|/max(1,|b|) = 9.63e-35; tolerance = 1e-20]
EXACT  PASS  Eq.(A.14) (s^2-1)^(3/4)(2s^3-s^2+4s+1)/(s^2(s+1)^4) ds with s=1/w gives the w-integrand (ratio = 1)   [simplify -> 0]
NUM  PASS  Eq.(A.17) M by quadrature (w=v^2) = (176/75) sqrt(2pi) Gamma(3/4)/Gamma(1/4)   [|a-b|/max(1,|b|) = 9.92e-32; tolerance = 1e-22]
NUM  PASS  Eq.(A.17) T1/U = 16/15   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-20]
NUM  PASS  Eq.(A.17) T2/U = 2/15   [|a-b|/max(1,|b|) = 2.47e-32; tolerance = 1e-20]
NUM  PASS  Eq.(A.17) (4 T3 + T4)/U = 1/5

<a id="check-12"></a>
## Check 12 — Coefficient assembly

The three contributions give
$$A_3=-\left(\frac{32}{15}+\frac{416}{675}+\frac{264}{125}\right)C_\Gamma
=-\frac{16408}{3375}C_\Gamma.$$
Cross-sphere and kinematic terms combine to $1856/675$, carrying $M_6$; the Lindhard term carries $M_4M_5$. Exact rational arithmetic checks the weights. The cell converts the gamma expressions to the three $r_s$ coefficients in Eq. (6) and recomputes the three-term value at $r_s=1000$.

In [13]:
exact("Eq.(A.18) 32/15 + 416/675 + 264/125 = 16408/3375", R(32,15) + R(416,675) + R(264,125) - R(16408,3375))
exact("Eq.(46) 32/15 + 416/675 = 1856/675", R(32,15) + R(416,675) - R(1856,675))
CGn = mp.mpf(sp.N(CG, 30))
num("Eq.(A.3) C_Gamma = 0.0858406325", CGn, mp.mpf('0.0858406325'), tol=1e-9)
A3 = -mp.mpf(16408)/3375*CGn
num("Eq.(42) A3 = -0.41732536262", A3, mp.mpf('-0.41732536262'), tol=1e-11)
qs_r = lambda n: (4*a_mp/3)**(mp.mpf(n)/4)          # q_*^n / r_s^(n/4)
num("Eq.(6)  A1/q*   -> -0.2213782293 r_s^(-1/4)", A1/qs_r(1), mp.mpf('-0.2213782293'), tol=1e-9)
num("Eq.(6)  A2/q*^2 -> +0.5670608947 r_s^(-1/2)", (mp.mpf(8)/15)/qs_r(2), mp.mpf('0.5670608947'), tol=1e-9)
num("Eq.(6)  A3/q*^3 -> -0.4575317487 r_s^(-3/4)", A3/qs_r(3), mp.mpf('-0.4575317487'), tol=1e-9)
rs = mp.mpf(1000); e3 = A1/qs_r(1)*rs**(-mp.mpf(1)/4) + (mp.mpf(8)/15)/qs_r(2)*rs**(-mp.mpf(1)/2) + A3/qs_r(3)*rs**(-mp.mpf(3)/4)
num("Strong-screening table: three-term expansion at r_s=10^3 = -24.008 mRy", 1000*e3, mp.mpf('-24.008'), tol=1e-4)
# Eq.(43): the q_* form combines Eqs.(34), (38), (42), equals Eq.(46) at zeta=0, and gives Eq.(6)
c41 = [-4/(7*sp.sqrt(2)*pi**R(3,2))*sp.gamma(R(1,4))/sp.gamma(R(3,4)), R(8,15), -R(16408,3375)*sp.sqrt(2*pi)/pi**2*sp.gamma(R(3,4))/sp.gamma(R(1,4))]
exact("Eq.(43) q*^-1 coefficient = A1 of Eq.(34)", c41[0] - A1_s)
exact("Eq.(43) q*^-3 coefficient = A3 of Eq.(42) = -(16408/3375) C_Gamma", c41[2] + R(16408,3375)*CG)
exact("Eqs.(43),(46) at zeta=0 (M_n=1): (1856/675 + 264/125) = 16408/3375", R(1856,675) + R(264,125) - R(16408,3375))
num("Eq.(6)  q* = (4 a r_s/3)^(1/4) = 0.9698052137 r_s^(1/4)", qs_r(1), mp.mpf('0.9698052137'), tol=1e-10)


EXACT  PASS  Eq.(A.18) 32/15 + 416/675 + 264/125 = 16408/3375   [simplify -> 0]
EXACT  PASS  Eq.(46) 32/15 + 416/675 = 1856/675   [simplify -> 0]
NUM  PASS  Eq.(A.3) C_Gamma = 0.0858406325   [|a-b|/max(1,|b|) = 4.86e-11; tolerance = 1e-09]
NUM  PASS  Eq.(42) A3 = -0.41732536262   [|a-b|/max(1,|b|) = 4.43e-12; tolerance = 1e-11]
NUM  PASS  Eq.(6)  A1/q*   -> -0.2213782293 r_s^(-1/4)   [|a-b|/max(1,|b|) = 8.79e-12; tolerance = 1e-09]
NUM  PASS  Eq.(6)  A2/q*^2 -> +0.5670608947 r_s^(-1/2)   [|a-b|/max(1,|b|) = 1.25e-11; tolerance = 1e-09]
NUM  PASS  Eq.(6)  A3/q*^3 -> -0.4575317487 r_s^(-3/4)   [|a-b|/max(1,|b|) = 8.32e-12; tolerance = 1e-09]
NUM  PASS  Strong-screening table: three-term expansion at r_s=10^3 = -24.008 mRy   [|a-b|/max(1,|b|) = 3.53e-6; tolerance = 0.0001]
EXACT  PASS  Eq.(43) q*^-1 coefficient = A1 of Eq.(34)   [simplify -> 0]
EXACT  PASS  Eq.(43) q*^-3 coefficient = A3 of Eq.(42) = -(16408/3375) C_Gamma   [simplify -> 0]
EXACT  PASS  Eqs.(43),(46) at zeta=0 (M_n=1): (18

<a id="check-13"></a>
## Check 13 — Spin moments and exact density rescaling

Equation (4) defines $M_n=\tfrac12\sum_\sigma x_\sigma^n$. The cell checks $M_n(0)=1$, the normalization of $\mathcal S_3$, and the full-polarization values.

For $x_{\rm pol}=2^{1/3}$, the single surviving response channel gives
$$c_1(x_{\rm pol}q,x_{\rm pol}^2\nu;r_s)=c_0(q,\nu;2^{-4/3}r_s).$$
The exchange-loop weight is $x_{\rm pol}^3/2=1$, yielding Eq. (49). The code keeps the unpolarized response symbolic and checks the rescaling factors. It also checks that the three asymptotic terms acquire factors $2^{1/3},2^{2/3},2$. Distinguishing $M_6$ from $M_4M_5$ requires the general-spin calculation in Check 25.

In [14]:
exact("Eq.(46)->(3) (1856/675)/(16408/3375) = 1160/2051", R(1856,675)/R(16408,3375) - R(1160,2051))
exact("Eq.(46)->(3) (264/125)/(16408/3375) = 891/2051",   R(264,125)/R(16408,3375) - R(891,2051))
exact("Eq.(5)  1160/2051 + 891/2051 = 1  (S_3(0)=1)", R(1160,2051) + R(891,2051) - 1)
S3 = (1160*Mn(6) + 891*Mn(4)*Mn(5))/2051
for n in (4, 5, 6): exact("Eq.(4)  M_%d(0) = 1" % n, Mn(n).subs(zeta, 0) - 1)
exact("Eq.(49) M_4(1) = 2^(1/3)", Mn(4).subs(zeta, 1) - 2**R(1,3))
exact("Eq.(49) M_5(1) = 2^(2/3)", Mn(5).subs(zeta, 1) - 2**R(2,3))
exact("Eq.(49) M_6(1) = 2 = M_4(1) M_5(1)", Mn(6).subs(zeta, 1) - 2)
exact("Eq.(49) S_3(1) = 2", S3.subs(zeta, 1) - 2)
for n, Mf in ((1, Mn(4)), (2, Mn(5)), (3, S3)):
    exact("Eq.(49) rescaling factor (2^(-4/3))^(-%d/4) = 2^(%d/3) equals the zeta=1 moment" % (n, n), (2**R(-4,3))**R(-n,4) - Mf.subs(zeta, 1))
# Eq.(47): c_{zeta=1}(x q, x^2 nu; rs) = c_{zeta=0}(q, nu; rs/2^(4/3)), with L kept as an undetermined symbol
xq, xnu, lindhard_symbol, ars = sp.symbols('q nu L a_rs', positive=True)
x_plus = 2**R(1,3)
lindhard_shape_pol = R(1,2)*x_plus*lindhard_symbol                      # L_{zeta=1}(x q, x^2 nu) = (1/2) x_+ L(q, nu); x_- = 0
c_pol = ars/(x_plus*xq)**2*lindhard_shape_pol
exact("Eq.(47) c_{zeta=1}(x q, x^2 nu; rs) = c_{zeta=0}(q, nu; rs/2^(4/3))", c_pol - (ars/2**R(4,3))/xq**2*lindhard_symbol)

EXACT  PASS  Eq.(46)->(3) (1856/675)/(16408/3375) = 1160/2051   [simplify -> 0]
EXACT  PASS  Eq.(46)->(3) (264/125)/(16408/3375) = 891/2051   [simplify -> 0]
EXACT  PASS  Eq.(5)  1160/2051 + 891/2051 = 1  (S_3(0)=1)   [simplify -> 0]
EXACT  PASS  Eq.(4)  M_4(0) = 1   [simplify -> 0]
EXACT  PASS  Eq.(4)  M_5(0) = 1   [simplify -> 0]
EXACT  PASS  Eq.(4)  M_6(0) = 1   [simplify -> 0]
EXACT  PASS  Eq.(49) M_4(1) = 2^(1/3)   [simplify -> 0]
EXACT  PASS  Eq.(49) M_5(1) = 2^(2/3)   [simplify -> 0]
EXACT  PASS  Eq.(49) M_6(1) = 2 = M_4(1) M_5(1)   [simplify -> 0]
EXACT  PASS  Eq.(49) S_3(1) = 2   [simplify -> 0]
EXACT  PASS  Eq.(49) rescaling factor (2^(-4/3))^(-1/4) = 2^(1/3) equals the zeta=1 moment   [simplify -> 0]
EXACT  PASS  Eq.(49) rescaling factor (2^(-4/3))^(-2/4) = 2^(2/3) equals the zeta=1 moment   [simplify -> 0]
EXACT  PASS  Eq.(49) rescaling factor (2^(-4/3))^(-3/4) = 2^(3/3) equals the zeta=1 moment   [simplify -> 0]
EXACT  PASS  Eq.(47) c_{zeta=1}(x q, x^2 nu; rs) = c_{zeta=0}

<a id="check-14"></a>
## Check 14 — Static Yukawa reference

Appendix D defines $W_Y=Z_Y/(q^2+\kappa^2)$ and $\mathcal K[W_Y]=(6/\pi^3)Z_YT(\kappa)$. Here $X_c=X_{\rm dyn}|_{R=1}$ and $\mathcal M_2=\int q^2X_c\,dq$.

The identity
$$\frac{q^2}{q^2+\kappa^2}-\frac{q^2}{\kappa^2}=-\frac{q^4}{\kappa^2(q^2+\kappa^2)}$$
and $X_c\sim C_\infty q^{-4}$ give $T=\mathcal M_2/\kappa^2-\pi C_\infty/(2\kappa^3)+O(\kappa^{-4})$. The cell checks the subtraction and tail integral. It also checks $f_L(iy)/y\to\pi/4$, which fixes the momentum-pole scale $\kappa\sim\pi q_{\rm TF}^{2}/8$. The integrable tail-remainder argument is in Appendix D.

In [15]:
kap = sp.symbols('kappa', positive=True)
exact("Eq.(D.1) q^2/(q^2+k^2) - q^2/k^2 = -q^4/[k^2 (q^2+k^2)]", qv**2/(qv**2+kap**2) - qv**2/kap**2 + qv**4/(kap**2*(qv**2+kap**2)))
exact("Eq.(D.1) -(C_inf/k^2) int dq/(q^2+k^2) = -pi C_inf/(2 k^3)", -(2*pi/9)/kap**2*sp.integrate(1/(qv**2+kap**2), (qv, 0, oo)) + pi*(2*pi/9)/(2*kap**3))
yv = sp.symbols('y', positive=True)
fLi = R(1,2) + (1+yv**2)*sp.atan(yv)/(2*yv)          # f_L(i y)
exact("Sec.IV f_L(iy) = (pi/4) y + O(1)  =>  kappa_* ~ pi q_tf_sq/8", sp.limit(fLi/yv, yv, oo) - pi/4)

EXACT  PASS  Eq.(D.1) q^2/(q^2+k^2) - q^2/k^2 = -q^4/[k^2 (q^2+k^2)]   [simplify -> 0]
EXACT  PASS  Eq.(D.1) -(C_inf/k^2) int dq/(q^2+k^2) = -pi C_inf/(2 k^3)   [simplify -> 0]
EXACT  PASS  Sec.IV f_L(iy) = (pi/4) y + O(1)  =>  kappa_* ~ pi q_tf_sq/8   [simplify -> 0]


<a id="check-15"></a>
## Check 15 — Spin geometry and radial normalization

Rescaling both momenta gives measure $x_\sigma^6$, Coulomb factor $x_\sigma^{-2}$, and quadratic moment factor $x_\sigma^2$. Thus $I_{0,\sigma}=x_\sigma^4I_0$, $I_{2,\sigma}=x_\sigma^6I_2$, and the integrated finite part scales as $x_\sigma^5$ (Eq. (45)).

The cell checks these powers and the radial normalization $e=6X_{\rm dyn}/\pi^3$. With $q=q_*x$, the profile limit in Eq. (B.1) gives the prefactor $3/\pi^2$ in $A_1$.

In [16]:
xs_ = sp.symbols('x_sigma', positive=True)
exact("Eq.(45) I_0: d^3p d^3k / |p-k|^2 scales as x^6 / x^2 = x^4", xs_**6/xs_**2 - xs_**4)
exact("Eq.(45) I_2: (p_z^2 ...) d^3p d^3k / |p-k|^2 scales as x^2 x^6 / x^2 = x^6", xs_**2*xs_**6/xs_**2 - xs_**6)
exact("Eq.(45) V_B^2 scales as x^6", (xs_**3)**2 - xs_**6)
exact("Eq.(45) Fp int Z_sigma dq = x^4 * x * Fp int Z dq = x^5", xs_**4*xs_ - xs_**5)
exact("Eq.(B.1) (6/pi^3) int q*^-2 (pi/(2x^2)) F q* dx = (3/pi^2) q*^-1 int F/x^2 dx", R(6)/pi**3*pi/2 - R(3)/pi**2)

EXACT  PASS  Eq.(45) I_0: d^3p d^3k / |p-k|^2 scales as x^6 / x^2 = x^4   [simplify -> 0]
EXACT  PASS  Eq.(45) I_2: (p_z^2 ...) d^3p d^3k / |p-k|^2 scales as x^2 x^6 / x^2 = x^6   [simplify -> 0]
EXACT  PASS  Eq.(45) V_B^2 scales as x^6   [simplify -> 0]
EXACT  PASS  Eq.(45) Fp int Z_sigma dq = x^4 * x * Fp int Z dq = x^5   [simplify -> 0]
EXACT  PASS  Eq.(B.1) (6/pi^3) int q*^-2 (pi/(2x^2)) F q* dx = (3/pi^2) q*^-1 int F/x^2 dx   [simplify -> 0]


<a id="check-16"></a>
## Check 16 — Static response, spin overlap, and polarization energy

This cell checks the static limit $\mathcal L(q,0)=f_L(q/2)$ and its normalization, the relation between $c$ and $q_*$, and the scaling of $C_\zeta$ in Eq. (44).

At fixed coupling, $W_{\lambda,\zeta}=\lambda q^{-2}/(1+\lambda c_\zeta)$. Rescaling this line gives Eq. (48). Applying the fundamental theorem of calculus to the unpolarized energy then gives
$$\Delta_{\rm pol}e_{\rm kite}=e_{\rm kite}(r_s,1)-e_{\rm kite}(r_s,0)
=-\int_{2^{-4/3}r_s}^{r_s}\partial_te_{\rm kite}(t,0)dt.$$
The symbolic tests keep the response and energy functions arbitrary; their physical definitions remain those of the manuscript.

In [17]:
exact("Eq.(10)  2 int_0^1 lam dlam = 1", 2*sp.integrate(lam, (lam, 0, 1)) - 1)
def lindhard_shape_exact(q, nu):
    f = lambda kk: kk*mp.log(((q*kk+q*q/2)**2 + nu**2)/((q*kk-q*q/2)**2 + nu**2))
    return mp.quad(f, [0, q/2, 1])/(2*q) if q < 2 else mp.quad(f, [0, 1])/(2*q)
fL = lambda x_: mp.mpf(1)/2 + (1-x_**2)*mp.log(abs((1+x_)/(1-x_)))/(4*x_)
for qv_ in [mp.mpf('0.8'), mp.mpf(3)]:
    num("Eq.(23) L(q,0) = f_L(q/2), q=%s" % qv_, lindhard_shape_exact(qv_, mp.mpf('1e-30')), fL(qv_/2), tol=1e-10)
num("Eq.(23) L(0,0) = 1 (normalization), q=1e-4", lindhard_shape_exact(mp.mpf('1e-4'), mp.mpf('1e-30')), 1, tol=1e-7)
qs_, xq_ = sp.symbols('q_* q', positive=True)
exact("Eq.(29) (q_tf_sq/q^2)(4/(3 q^2 (1+4u^2))) = (q*/q)^4/(1+4u^2) with q*^4 = 4 q_tf_sq/3",
      (q_tf_sq/xq_**2)*R(4,3)/(xq_**2*(1+4*u**2)) - ((R(4,3)*q_tf_sq)**R(1,4)/xq_)**4/(1+4*u**2))
Cf = sp.Function('C'); xs_ = sp.symbols('x_sigma', positive=True)
C_sigma = xs_**6*xs_**(-2)*Cf(xq_/xs_)                                   # p,k -> x p, x k in C_sigma(q)
exact("Eq.(44) C_sigma(q) = x^4 C(q/x)", C_sigma - xs_**4*Cf(xq_/xs_))
qq2 = sp.symbols('q2', positive=True)                                    # q = x q2
exact("Eq.(44) int C_sigma dq = x^5 int C dq   (dq = x dq2)", sp.simplify(C_sigma.subs(xq_, xs_*qq2)*xs_) - xs_**5*Cf(qq2))
exact("Eq.(44) q^2 C_sigma -> x^6 V_B^2   (using q^2 C(q) -> V_B^2)", sp.limit(xq_**2*xs_**4*(4*pi/3)**2/(xq_/xs_)**2, xq_, oo) - xs_**6*(4*pi/3)**2)
exact("Eq.(44) (1/2) sum_sigma x^5 = M_5, (1/2) sum_sigma x^6 = M_6",
      (R(1,2)*((1+zeta)**R(5,3) + (1-zeta)**R(5,3)) - Mn(5))**2 + (R(1,2)*((1+zeta)**2 + (1-zeta)**2) - Mn(6))**2)
csym = sp.symbols('c0', positive=True); Xc = 2**R(1,3)
exact("Eq.(48) x^2 (x q)^-2/(1+lam c) = q^-2/(1+lam c)", Xc**2*(Xc*xq_)**(-2)/(1+lam*csym) - xq_**(-2)/(1+lam*csym))
E = sp.Function('E'); rs_, t_ = sp.symbols('r_s t', positive=True)
exact("Eq.(50) -int_{2^(-4/3) rs}^{rs} E'(t) dt = E(2^(-4/3) rs) - E(rs)",
      -sp.integrate(sp.diff(E(t_), t_), (t_, 2**R(-4,3)*rs_, rs_)) - (E(2**R(-4,3)*rs_) - E(rs_)))

EXACT  PASS  Eq.(10)  2 int_0^1 lam dlam = 1   [simplify -> 0]
NUM  PASS  Eq.(23) L(q,0) = f_L(q/2), q=0.8   [|a-b|/max(1,|b|) = 1.58e-30; tolerance = 1e-10]
NUM  PASS  Eq.(23) L(q,0) = f_L(q/2), q=3.0   [|a-b|/max(1,|b|) = 2.47e-32; tolerance = 1e-10]
NUM  PASS  Eq.(23) L(0,0) = 1 (normalization), q=1e-4   [|a-b|/max(1,|b|) = 8.33e-10; tolerance = 1e-07]
EXACT  PASS  Eq.(29) (q_tf_sq/q^2)(4/(3 q^2 (1+4u^2))) = (q*/q)^4/(1+4u^2) with q*^4 = 4 q_tf_sq/3   [simplify -> 0]
EXACT  PASS  Eq.(44) C_sigma(q) = x^4 C(q/x)   [simplify -> 0]
EXACT  PASS  Eq.(44) int C_sigma dq = x^5 int C dq   (dq = x dq2)   [simplify -> 0]
EXACT  PASS  Eq.(44) q^2 C_sigma -> x^6 V_B^2   (using q^2 C(q) -> V_B^2)   [simplify -> 0]
EXACT  PASS  Eq.(44) (1/2) sum_sigma x^5 = M_5, (1/2) sum_sigma x^6 = M_6   [simplify -> 0]
EXACT  PASS  Eq.(48) x^2 (x q)^-2/(1+lam c) = q^-2/(1+lam c)   [simplify -> 0]
EXACT  PASS  Eq.(50) -int_{2^(-4/3) rs}^{rs} E'(t) dt = E(2^(-4/3) rs) - E(rs)   [simplify -> 0]


<a id="check-17"></a>
## Check 17 — Tabulated arithmetic and effective slopes

The cell recomputes the six three-term values in the strong-screening table, the residuals $\mathcal R_3$ from its direct column, the logarithmic slopes at $r_s=10^3,10^4,10^6$, and $q_*$ at $10^6$.

The direct measurements are inputs to these arithmetic checks. The printed residuals and their errors use the same full-precision records as the manuscript; these data transformations are not new integral tests. The last identity checks the static reference $Z_YT\sim128\mathcal M_2/(\pi^2q_{\rm TF}^{4})$.

In [18]:
e3 = lambda rs: A1/qs_r(1)*rs**(-mp.mpf(1)/4) + (mp.mpf(8)/15)/qs_r(2)*rs**(-mp.mpf(1)/2) + A3/qs_r(3)*rs**(-mp.mpf(3)/4)
for row in PRODUCTION['tableI']:
    rs_v=row['rs']
    num('Independent coefficient evaluation vs generated table model, rs=%g'%rs_v,
        1000*e3(mp.mpf(rs_v)),1000*production_common.asymptotic_kite(rs_v),tol=1e-12)
qstar=lambda rs:(4*a_mp*rs/3)**mp.mpf('0.25')
for row in PRODUCTION['tableI'][:3]:
    q_=qstar(mp.mpf(row['rs']))
    residual=q_**3*(mp.mpf(row['mean_mry'])/1000-A1/q_-(mp.mpf(8)/15)/q_**2)
    uncertainty=q_**3*mp.mpf(row['se_mry'])/1000
    print('DATA TRANSFORMATION: rs=%g, R3=%s +/- %s Ry'%(row['rs'],mp.nstr(residual,8),mp.nstr(uncertainty,5)))
for rs_v, ref in [(1e3, '-0.12'), (1e4, '-0.18'), (1e6, '-0.23')]:
    slope = mp.diff(lambda t_: mp.log(abs(e3(mp.exp(t_)))), mp.log(mp.mpf(rs_v)))
    num("Sec.IV  d log|e_2b|/d log r_s at r_s=%g = %s" % (rs_v, ref), slope, mp.mpf(ref), tol=3e-2)
num("Sec.IV  q_* = 30.7 at r_s = 10^6", qstar(mp.mpf(1e6)), mp.mpf('30.7'), tol=2e-3)
M2s, q_tf_sqs = sp.symbols('M_2 q_tf_sq', positive=True)
exact("Sec.IV  Z T ~ 2 M_2/(pi q_tf_sq/8)^2 = 128 M_2/(pi^2 q_tf_sq^2)", 2*M2s/(pi*q_tf_sqs/8)**2 - 128*M2s/(pi**2*q_tf_sqs**2))

NUM  PASS  Independent coefficient evaluation vs generated table model, rs=1000   [|a-b|/max(1,|b|) = 2.38e-16; tolerance = 1e-12]
NUM  PASS  Independent coefficient evaluation vs generated table model, rs=5000   [|a-b|/max(1,|b|) = 5.88e-16; tolerance = 1e-12]
NUM  PASS  Independent coefficient evaluation vs generated table model, rs=20000   [|a-b|/max(1,|b|) = 3.72e-16; tolerance = 1e-12]
NUM  PASS  Independent coefficient evaluation vs generated table model, rs=200000   [|a-b|/max(1,|b|) = 7.7e-17; tolerance = 1e-12]
NUM  PASS  Independent coefficient evaluation vs generated table model, rs=1e+06   [|a-b|/max(1,|b|) = 3.51e-16; tolerance = 1e-12]
NUM  PASS  Independent coefficient evaluation vs generated table model, rs=1e+08   [|a-b|/max(1,|b|) = 4.52e-16; tolerance = 1e-12]
DATA TRANSFORMATION: rs=1000, R3=-0.32168274 +/- 0.0087564 Ry
DATA TRANSFORMATION: rs=5000, R3=-0.33216851 +/- 0.033076 Ry
DATA TRANSFORMATION: rs=20000, R3=-0.5464513 +/- 0.10455 Ry
NUM  PASS  Sec.IV  d log|e_

<a id="check-18"></a>
## Check 18 — Bounds and composite endpoints

Appendix C combines fixed- and large-momentum estimates. This cell samples the Lindhard bounds in Eq. (C.1), integrates $|\Delta_{\mathbf p}|/|\delta_{\mathbf p}|$ to compare with $4\pi \mathcal L(q,0)$, and verifies that the leading composite terms reduce to $4\pi A_0K_0Z(q)$.

For the occupation integral, the cap is $\{p<1,|\mathbf p+\mathbf q|>1\}$. The angular lower endpoint is $c_0(p)=(1-p^2-q^2)/(2pq)$; the opposite-sign cap contributes equally.

Finally, quadrature of $[H(x)-4/3]/x^2$, with $H=x^{-2}J_C(x^{-4})$, is compared with the cross finite part. Symbolic endpoints and sampled inequalities support the analytical estimates; the bounds over continuous domains are proved in Appendix C.

In [19]:
ok = True
for qv_, xv_ in [(0.5, 0.3), (1.5, 2.0), (3.0, 0.1), (3.0, 40.0)]:
    qv_, xv_ = mp.mpf(qv_), mp.mpf(xv_)
    lindhard_value = lindhard_shape_exact(qv_, xv_)
    ok = ok and (qv_**2/(3*(xv_**2 + (qv_+qv_**2/2)**2)) <= lindhard_value <= qv_**2/(3*xv_**2))
check("Eq.(C.1) q^2/(3[nu^2+(q+q^2/2)^2]) <= L(q,nu) <= q^2/(3 nu^2) at 4 sampled points", ok)
def occ_int(q):
    def inner(p):
        c0 = max(mp.mpf(-1), (1-p*p-q*q)/(2*p*q))
        if c0 >= 1: return mp.mpf(0)
        return p*p/(p*q)*mp.log((p*q+q*q/2)/(p*q*c0+q*q/2))
    pts = [0, abs(1-q), 1] if q < 2 else [0, 1]
    return 4*mp.pi*mp.quad(inner, pts)
for qv_ in [mp.mpf('0.5'), mp.mpf('1.5'), mp.mpf(3)]:
    num("App.C  int |Delta_p|/|delta_p| d^3p = 4 pi L(q,0), q=%s" % qv_, occ_int(qv_), 4*mp.pi*fL(qv_/2), tol=1e-15)
Cq = sp.symbols('C_q', positive=True); Qs = sp.symbols('Q', positive=True)
exact("Eq.(C.3) sqrt(g) F(g) -> -4/3 as g -> oo", sp.limit(sp.sqrt(g)*F_cl, g, oo) + R(4,3))
exact("Eq.(C.3) sqrt(g) J_C(g) -> 4/3 as g -> oo", sp.limit(sp.sqrt(g)*JC_cl, g, oo) - R(4,3))
lead = 4*pi*A0*(8*pi**2/xq_**2*(-R(4,3))*xq_**2/Qs**2 + 2*Cq/xq_**2*R(4,3)*xq_**2/Qs**2)   # sqrt(g) = Q^2/q^2
exact("Eq.(C.3)->(C.2) leading composite density = 4 pi A0 K0 Z, K0 = -4/(3Q^2), Z = 8 pi^2 - 2C", lead - 4*pi*A0*(-R(4,3)/Qs**2)*(8*pi**2 - 2*Cq))
EC_num = mp.quad(hsub, [0, mp.mpf(1)/2, 1], method='gauss-legendre') + mp.quad(lambda xx: fprof(xx) - mp.mpf(4)/(3*xx**2), [1, 2, 4, mp.inf])
num("Eq.(C.9) int_0^oo [H(x) - 4/3] x^-2 dx = finite part of Eq.(A.4)", EC_num, mp.mpf(sp.N(Fp_t, 30)), tol=1e-18)

CHECK  PASS  Eq.(C.1) q^2/(3[nu^2+(q+q^2/2)^2]) <= L(q,nu) <= q^2/(3 nu^2) at 4 sampled points
NUM  PASS  App.C  int |Delta_p|/|delta_p| d^3p = 4 pi L(q,0), q=0.5   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-15]
NUM  PASS  App.C  int |Delta_p|/|delta_p| d^3p = 4 pi L(q,0), q=1.5   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-15]
NUM  PASS  App.C  int |Delta_p|/|delta_p| d^3p = 4 pi L(q,0), q=3.0   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-15]
EXACT  PASS  Eq.(C.3) sqrt(g) F(g) -> -4/3 as g -> oo   [simplify -> 0]
EXACT  PASS  Eq.(C.3) sqrt(g) J_C(g) -> 4/3 as g -> oo   [simplify -> 0]
EXACT  PASS  Eq.(C.3)->(C.2) leading composite density = 4 pi A0 K0 Z, K0 = -4/(3Q^2), Z = 8 pi^2 - 2C   [simplify -> 0]
NUM  PASS  Eq.(C.9) int_0^oo [H(x) - 4/3] x^-2 dx = finite part of Eq.(A.4)   [|a-b|/max(1,|b|) = 7.75e-29; tolerance = 1e-18]


<a id="check-19"></a>
## Check 19 — Additional geometry, bounds, and data arithmetic

The cell checks reference momentum scales and positivity of the Lindhard logarithm, and compares the overlap definition of $Z(q)$ with its independent Fourier representation using $\widehat\chi_B(t)=4\pi(\sin t-t\cos t)/t^3$.

It samples the ratios in $R\le\min(1,2/c)$, $c|R'|\le R$, and $c^2R''\le2R$, and computes their limiting values. The remaining checks recompute the scaled standard errors of $\mathcal R_3$ and the mean, scatter, and signs of the published-minus-direct values. These checks use the supplied table entries.

In [20]:
print("Eq.(21) alpha_L =", mp.nstr(alphaL_mp, 6), "  a = 4 alpha_L/pi =", mp.nstr(a_mp, 6))
for rs_v, ref in [(1, '0.97'), (2, '1.15'), (5, '1.45')]:
    num("Eq.(3)  q* at r_s=%g = %s (metallic values quoted in the companion paper)" % (rs_v, ref), qstar(mp.mpf(rs_v)), mp.mpf(ref), tol=6e-3)
kk_ = sp.symbols('k', positive=True)
exact("Eq.(17) (qk+q^2/2)^2 - (qk-q^2/2)^2 = 2 q^3 k >= 0  =>  L >= 0, c >= 0", sp.expand((qv*kk_+qv**2/2)**2 - (qv*kk_-qv**2/2)**2) - 2*qv**3*kk_)
lindhard_integrand = kk_*sp.log(((qv*kk_+qv**2/2)**2+nu**2)/((qv*kk_-qv**2/2)**2+nu**2))/(2*qv)
lindhard_shape_large_nu = sp.integrate(sp.series(lindhard_integrand, nu, oo, 5).removeO(), (kk_, 0, 1))
exact("Eq.(35)/App.C  L(q,nu) = q^2/(3 nu^2) + O(nu^-4)", sp.limit(lindhard_shape_large_nu*nu**2, nu, oo) - qv**2/3)
# Eq.(36): Z = 8 pi^2 - 2 C(q), tested against the Fourier representation of Z (JPA paper, Eq. (G.9))
def g_(x_): return x_**2/6 - x_**4/120 + x_**6/5040 if x_ < mp.mpf('1e-3') else 1 - mp.sin(x_)/x_
def Z_fourier(q):
    A_ = mp.quad(lambda t: (mp.sin(t)-t*mp.cos(t))**2*g_(q*t)/t**5, [0, mp.mpf(1)/2, 1])
    B_ = q**4/2*mp.quad(lambda xx: g_(xx)/xx**5, [q, 1, 10, 100, mp.inf]) + q**2/2*mp.quad(lambda xx: g_(xx)/xx**3, [q, 1, 10, 100, mp.inf])
    C_ = mp.quadosc(lambda t: (-mp.cos(2*t)/2 - t*mp.sin(2*t) + t**2*mp.cos(2*t)/2)*g_(q*t)/t**5, [1, mp.inf], omega=2)
    return 32*mp.pi**2*(A_ + B_ + C_)
mp.mp.dps = 20
for q_ in [mp.mpf('0.5'), mp.mpf(3)]:
    num("Eq.(36) Z(q) = 8 pi^2 - 2 C(q) at q=%s  (Fourier form of Z vs. overlap form of C)" % q_, Z_fourier(q_), 8*mp.pi**2 - 2*C_of_q(q_), tol=1e-6)
mp.mp.dps = 30
import numpy as np
Rp_ = sp.lambdify(c, sp.diff(Rc, c), 'mpmath'); Rpp_ = sp.lambdify(c, sp.diff(Rc, c, 2), 'mpmath')
worst = [mp.mpf(0)]*3
for cv in [mp.mpf(10)**e for e in np.linspace(-3, 4, 60)]:
    r_ = Rn(cv)
    worst[0] = max(worst[0], r_/min(1, 2/cv)); worst[1] = max(worst[1], cv*abs(Rp_(cv))/r_); worst[2] = max(worst[2], cv**2*Rpp_(cv)/(2*r_))
check("App.C  R <= min(1,2/c), c|R'| <= R, c^2 R'' <= 2R on 1e-3 <= c <= 1e4 (max ratios %s)" % [mp.nstr(w_, 5) for w_ in worst], max(worst) <= 1)
exact("App.C  limits: c R -> 2, c R'/R -> -1, c^2 R''/(2R) -> 1 as c -> oo",
      (sp.limit(c*Rc, c, oo) - 2)**2 + (sp.limit(c*sp.diff(Rc, c)/Rc, c, oo) + 1)**2 + (sp.limit(c**2*sp.diff(Rc, c, 2)/(2*Rc), c, oo) - 1)**2)
ratios = [mp.quad(lambda uu: Rn(gv/(1+4*uu**2))/(1+4*uu**2)**2, [0, 1, mp.inf])/min(1, 1/gv) for gv in [mp.mpf('0.01'), mp.mpf(1), mp.mpf(100), mp.mpf(10000)]]
check("App.C  int R(g/(1+4u^2))/(1+4u^2)^2 du / min(1,1/g) stays bounded (%s)" % [mp.nstr(r_, 4) for r_ in ratios], max(ratios) < 2)
refs={row[0]:row[1] for row in PRODUCTION['reference']['tableII']}
dif=[refs[row['rs']]-row['mean_mry'] for row in PRODUCTION['tableII']]
print('DATA SUMMARY: published-minus-direct mean, SD, positive count:',
      np.mean(dif),np.std(dif,ddof=1),sum(v>0 for v in dif))
import tempfile, subprocess
with tempfile.TemporaryDirectory(prefix='prb-notebook-exports-') as temp_exports:
    subprocess.run([sys.executable,str(BUNDLE_ROOT/'production/render.py'),
                    '--output',temp_exports],check=True,capture_output=True,text=True)
    macros=(Path(temp_exports)/'spin/generated/numbers.tex').read_text()
check('Data-derived text contains the rounded residual mean',
      ('{'+format(np.mean(dif),'+.2f')+'}') in macros)
check('Data-derived text contains the rounded residual SD',
      ('{'+format(np.std(dif,ddof=1),'.2f')+'}') in macros)


Eq.(21) alpha_L = 0.521062   a = 4 alpha_L/pi = 0.663436
NUM  PASS  Eq.(3)  q* at r_s=1 = 0.97 (metallic values quoted in the companion paper)   [|a-b|/max(1,|b|) = 0.000195; tolerance = 0.006]
NUM  PASS  Eq.(3)  q* at r_s=2 = 1.15 (metallic values quoted in the companion paper)   [|a-b|/max(1,|b|) = 0.00287; tolerance = 0.006]
NUM  PASS  Eq.(3)  q* at r_s=5 = 1.45 (metallic values quoted in the companion paper)   [|a-b|/max(1,|b|) = 0.000136; tolerance = 0.006]
EXACT  PASS  Eq.(17) (qk+q^2/2)^2 - (qk-q^2/2)^2 = 2 q^3 k >= 0  =>  L >= 0, c >= 0   [simplify -> 0]
EXACT  PASS  Eq.(35)/App.C  L(q,nu) = q^2/(3 nu^2) + O(nu^-4)   [simplify -> 0]
NUM  PASS  Eq.(36) Z(q) = 8 pi^2 - 2 C(q) at q=0.5  (Fourier form of Z vs. overlap form of C)   [|a-b|/max(1,|b|) = 2.88e-8; tolerance = 1e-06]
NUM  PASS  Eq.(36) Z(q) = 8 pi^2 - 2 C(q) at q=3.0  (Fourier form of Z vs. overlap form of C)   [|a-b|/max(1,|b|) = 1.99e-7; tolerance = 1e-06]
CHECK  PASS  App.C  R <= min(1,2/c), c|R'| <= R, c^2 R'' <= 2R 

<a id="check-20"></a>
## Check 20 — Composite prefactors and static-pole limit

Substituting $q=Qx$ in each term of Eq. (C.3) must reproduce its coefficient integral. The cell checks all four prefactors, the radial factor $64\pi^2A_0=6/\pi^3$, and the fixed-momentum contribution $A_2/Q^2$.

For the static reference in Appendix D it solves $\kappa^2=q_{\rm TF}^{2} f_L(i\kappa/2)$ numerically and evaluates the residue $Z_Y=[1+q_{\rm TF}^{2}\,df_L/d(q^2)]^{-1}$ at the pole. The sampled limits are $\kappa/(\pi q_{\rm TF}^{2}/8)\to1$ and $Z_Y\to2$.

In [21]:
exact("Eq.(20) 4 pi A0 * 16 pi = 64 pi^2 A0 = 6/pi^3", 4*pi*A0*16*pi - 6/pi**3)
exact("Eq.(B.1) (1/(16 pi)) * 8 pi^2 * q^-4 * q^2 = pi/(2 q^2)", 1/(16*pi)*8*pi**2*qv**-4*qv**2 - pi/(2*qv**2))
Q = sp.symbols('Q', positive=True)
exact("Eq.(C.3)  4 pi A0 (8 pi^2/q^2) F dq, q = Q x  ->  (3/pi^2) F x^-2 dx / Q", 4*pi*A0*8*pi**2/(Q*x)**2*Q - 3/pi**2/x**2/Q)
exact("Eq.(C.3)  4 pi A0 (2 C/q^2) J_C dq with C -> V_B^2/q^2  ->  8 pi A0 V_B^2 x^-4 J_C dx / Q^3", 4*pi*A0*2*(4*pi/3)**2/(Q*x)**4*Q - 8*pi*A0*(4*pi/3)**2*x**-4/Q**3)
exact("Eq.(C.3)  4 pi A0 (104 pi^2/(27 q^4)) Phi_4 dq  ->  4 pi A0 (104 pi^2/27) x^-4 Phi_4 dx / Q^3", 4*pi*A0*104*pi**2/(27*(Q*x)**4)*Q - 4*pi*A0*104*pi**2/27*x**-4/Q**3)
exact("Eq.(C.3)  4 pi A0 (6 pi^2 Q^4/q^8) Psi dq  ->  24 pi^3 A0 x^-8 Psi dx / Q^3", 4*pi*A0*6*pi**2*Q**4/(Q*x)**8*Q - 24*pi**3*A0*x**-8/Q**3)
exact("Eq.(C.9)  (32 pi A0/(3 Q^2)) int C dq = A2/Q^2 with int C dq = 8 pi^4/15", 32*pi*A0/(3*Q**2)*8*pi**4/15 - R(8,15)/Q**2)
fLi_ = lambda y_: mp.mpf(1)/2 + (1+y_*y_)*mp.atan(y_)/(2*y_)
for rr in [1e3, 1e5]:
    tv = a_mp*mp.mpf(rr)
    kap = mp.findroot(lambda kk: kk*kk - tv*fLi_(kk/2), mp.pi*tv/8)
    Zv  = 1/(1 + tv*mp.diff(lambda s_: fLi_(mp.sqrt(-s_)/2), -kap**2))
    num("Sec.IV  Yukawa pole: kappa_*/(pi q_tf_sq/8) -> 1 at r_s=%g" % rr, kap/(mp.pi*tv/8), mp.mpf(1), tol=1e-4)
    num("Sec.IV  Yukawa residue: Z -> 2 at r_s=%g" % rr, Zv, mp.mpf(2), tol=1e-3)

EXACT  PASS  Eq.(20) 4 pi A0 * 16 pi = 64 pi^2 A0 = 6/pi^3   [simplify -> 0]
EXACT  PASS  Eq.(B.1) (1/(16 pi)) * 8 pi^2 * q^-4 * q^2 = pi/(2 q^2)   [simplify -> 0]
EXACT  PASS  Eq.(C.3)  4 pi A0 (8 pi^2/q^2) F dq, q = Q x  ->  (3/pi^2) F x^-2 dx / Q   [simplify -> 0]
EXACT  PASS  Eq.(C.3)  4 pi A0 (2 C/q^2) J_C dq with C -> V_B^2/q^2  ->  8 pi A0 V_B^2 x^-4 J_C dx / Q^3   [simplify -> 0]
EXACT  PASS  Eq.(C.3)  4 pi A0 (104 pi^2/(27 q^4)) Phi_4 dq  ->  4 pi A0 (104 pi^2/27) x^-4 Phi_4 dx / Q^3   [simplify -> 0]
EXACT  PASS  Eq.(C.3)  4 pi A0 (6 pi^2 Q^4/q^8) Psi dq  ->  24 pi^3 A0 x^-8 Psi dx / Q^3   [simplify -> 0]
EXACT  PASS  Eq.(C.9)  (32 pi A0/(3 Q^2)) int C dq = A2/Q^2 with int C dq = 8 pi^4/15   [simplify -> 0]
NUM  PASS  Sec.IV  Yukawa pole: kappa_*/(pi q_tf_sq/8) -> 1 at r_s=1000   [|a-b|/max(1,|b|) = 5.87e-5; tolerance = 0.0001]
NUM  PASS  Sec.IV  Yukawa residue: Z -> 2 at r_s=1000   [|a-b|/max(1,|b|) = 0.000117; tolerance = 0.001]
NUM  PASS  Sec.IV  Yukawa pole: kappa_*/(pi q

<a id="check-21"></a>
## Check 21 — Sampled moving- and fixed-momentum limits

These finite-parameter tests use the exact Lindhard integral for the moving-region limit in Eq. (29). The fixed-momentum test retains nonzero particle–hole energies and compares its frequency integral with $K_0$ in Eq. (35). Uniformity is treated analytically in Appendix C.

In [22]:
import numpy as np

def lindhard_shape_num(q, nu, nk=2000):                      # Eq.(23), midpoint rule in k (smooth integrand)
    k = (np.arange(nk) + 0.5)/nk
    return np.mean(k*np.log(((q*k + q*q/2)**2 + nu**2)/((q*k - q*q/2)**2 + nu**2)))/(2*q)
def R_np(c):                                    # numpy version of R(c), series for small c
    c = np.asarray(c, float)
    return np.where(c < 1e-8, 1 - 2*c/3 + c**2/2, 2*(c - np.log1p(np.where(c < 1e-8, 0.0, c)))/np.maximum(c, 1e-300)**2)

a_np = float(a_mp)                                # a = 4 alpha_L/pi
# --- 21.1 uniformity of Eq.(29) on a compact x-set, over all u
us = 10**np.linspace(-2, 2, 25)
devs = []
for rs_v in [1e4, 1e6, 1e8]:
    q_tf_sq_v = a_np*rs_v; qstar = (4*q_tf_sq_v/3)**0.25
    worst = 0.0
    for xv in [0.5, 1.0, 2.0]:
        q_v = qstar*xv
        ex = np.array([q_tf_sq_v*lindhard_shape_num(q_v, q_v**2*uu)/q_v**2 for uu in us])
        lim = xv**-4/(1 + 4*us**2)
        worst = max(worst, np.max(np.abs(ex/lim - 1)))
    devs.append(worst)
    print("r_s = %.0e :  q_* = %6.2f   max relative deviation of c from Eq.(29) = %.3e" % (rs_v, qstar, worst))
check("Eq.(29) sampled inner-limit grid (three x values, 25 u values): deviation < 1e-3 at r_s=1e8", devs[-1] < 1e-3)
ratios = [devs[i]/devs[i+1] for i in range(len(devs)-1)]
print("reduction per two decades of r_s:", [round(r_, 2) for r_ in ratios], " (q_*^-2 scaling predicts 10)")
check("Eq.(29) deviation falls like q_*^-2 (factor 10 per two decades of r_s, +-30%)", all(7 < r_ < 13 for r_ in ratios))

# --- 21.2 the outer frequency integral tends to K_0 = -4/(3 q_*^2)
def outer_ratio(rs_v, q_v=0.7, a_=0.3, b_=-0.45, nt=400):
    q_tf_sq_v = a_np*rs_v; qstar = (4*q_tf_sq_v/3)**0.25
    t = (np.arange(nt) + 0.5)*(np.pi/2)/nt
    sc = np.sqrt(q_tf_sq_v/3)                         # the outer region lives at nu ~ sqrt(q_tf_sq/3); scale the grid with it
    nu = sc*np.tan(t); jac = (sc/np.cos(t)**2)*((np.pi/2)/nt)
    Rv = R_np(np.array([q_tf_sq_v*lindhard_shape_num(q_v, x_)/q_v**2 for x_ in nu]))
    ker = (a_*b_ - nu**2)/((a_**2 + nu**2)*(b_**2 + nu**2))
    I = np.sum(Rv*ker*jac)/np.pi
    return I/(-4/(3*qstar**2))
for rs_v in [1e4, 1e6, 1e8]:
    print("r_s = %.0e :  (exact frequency integral)/K_0 = %.4f" % (rs_v, outer_ratio(rs_v)))
check("Eq.(35) the exact outer frequency integral tends to K_0 (ratio within 1% at r_s=1e8)", abs(outer_ratio(1e8) - 1) < 0.01)

r_s = 1e+04 :  q_* =   9.70   max relative deviation of c from Eq.(29) = 3.668e-02
r_s = 1e+06 :  q_* =  30.67   max relative deviation of c from Eq.(29) = 3.420e-03
r_s = 1e+08 :  q_* =  96.98   max relative deviation of c from Eq.(29) = 3.397e-04
CHECK  PASS  Eq.(29) sampled inner-limit grid (three x values, 25 u values): deviation < 1e-3 at r_s=1e8
reduction per two decades of r_s: [np.float64(10.72), np.float64(10.07)]  (q_*^-2 scaling predicts 10)
CHECK  PASS  Eq.(29) deviation falls like q_*^-2 (factor 10 per two decades of r_s, +-30%)
r_s = 1e+04 :  (exact frequency integral)/K_0 = 1.0101
r_s = 1e+06 :  (exact frequency integral)/K_0 = 1.0008
r_s = 1e+08 :  (exact frequency integral)/K_0 = 1.0000
CHECK  PASS  Eq.(35) the exact outer frequency integral tends to K_0 (ratio within 1% at r_s=1e8)


<a id="check-22"></a>
## Check 22 — Stable overlap evaluation

The next cell evaluates $C(q)$ using a small-$q$ expansion, its removable value at $q=2$, and its inverse-power tail. Each branch is compared with the independent angular-integral route in Check 7. The frequency profiles use cancellation-free expressions:
$$F(g)=-\frac{4g}{3(\sqrt{1+g}+1)^3},\qquad J_C(g)=\frac{4(\sqrt{1+g}+2)}{3(\sqrt{1+g}+1)^2}.$$

<a id="check-23"></a>
## Check 23 — Composite matching

Integrating Eq. (C.3) checks the assembly of the asymptotic profiles. Results are labeled `MODEL`: the integrand already contains the profiles used to derive the coefficients. Independent evaluation of the original functional begins in Check 28.

In [23]:
from scipy.special import gamma as _gamma        # only for Gamma(1/4), Gamma(3/4)
A0_np = 3/(32*np.pi**5)
A1_np = -4/(7*np.sqrt(2)*np.pi**1.5)*_gamma(0.25)/_gamma(0.75)
A2_np = 8/15
CG_np = np.sqrt(2*np.pi)/np.pi**2*_gamma(0.75)/_gamma(0.25)
A3_np = -16408/3375*CG_np


# Exact ball-overlap moments; stable branches at q=0,2,infinity.
m2_,m4_,m6_=4*np.pi/9,8*np.pi/15,32*np.pi/35
def Cq(q):
    arr=np.asarray(q,dtype=float); flat=arr.ravel(); out=np.empty_like(flat)
    for j,v in enumerate(flat):
        if v<1e-3:
            out[j]=4*np.pi**2 if v==0 else 4*np.pi**2-(4*np.pi**2/3)*v*v*(np.log(2/v)+5/6)
        elif v>=20:
            # Add two further exact overlap moments for accuracy at the switch.
            out[j]=4*np.pi*sum(float(4*np.pi/3*2**(n+1)/(n+1)-np.pi*2**(n+2)/(n+2)+np.pi/12*2**(n+4)/(n+4))/((n-1)*v**n) for n in [2,4,6,8,10,12])
        else:
            # The logarithmic expression is removable at q=2; use the defining integral there.
            if abs(v-2)<1e-6:
                out[j]=float(C_of_q(mp.mpf(str(v))))
            else:
                Lm=np.log(abs(2-v));Lp=np.log(v+2);lq=np.log(v)
                inner=v*(v**4*Lm-2*v*v*(v*v-20)*lq-20*v*v*Lm+4*v*v+40*v*Lm-v*(-v**3+20*v+40)*Lp+88)+32*(Lp-Lm)
                out[j]=np.pi**2*inner/(30*v)
    return out.reshape(arr.shape)
F_np=lambda g: -(4/3)*np.asarray(g)/(np.sqrt(1+np.asarray(g))+1)**3
JC_np=lambda g: (4/3)*(np.sqrt(1+np.asarray(g))+2)/(np.sqrt(1+np.asarray(g))+1)**2
for cv in [0.,1e-4,1.,2.,20.,100.]:
    target=4*mp.pi**2 if cv==0 else C_of_q(mp.mpf(str(cv)))
    num('C(q) stable branches vs defining overlap integral, q=%g'%cv,float(Cq(cv)),target,tol=2e-8)
_lam, _wl = np.polynomial.legendre.leggauss(160); _lam = (_lam + 1)/2; _wl = _wl/2
def Phi4_np(g):                                          # (A.6) after the residue identity (A.7)
    m_ = _lam[None, :]*np.asarray(g)[:, None]; s_ = np.sqrt(1 + m_)
    return np.sum(_wl[None, :]*2*_lam[None, :]*(-4*m_/(s_*(s_ + 1)**4)), axis=1)
def Psi_np(g):                                           # (A.9) after the contour identity (A.10)
    m_ = _lam[None, :]*np.asarray(g)[:, None]; s_ = np.sqrt(1 + m_)
    return np.sum(_wl[None, :]*(-2)*_lam[None, :]**2*8*(2*s_**3 - s_**2 + 4*s_ + 1)/(15*s_**3*(s_ + 1)**4), axis=1)
def _gl(a, b, n):
    xg, wg = np.polynomial.legendre.leggauss(n); return (b - a)/2*xg + (a + b)/2, (b - a)/2*wg
def S_comp(Q, nn=200):                                   # int_0^oo e_Q^comp(q) dq, q = Q x, decades in log10 x
    tot = 0.0
    for lo, hi in [(-10, -4), (-4, -2), (-2, -1), (-1, 0), (0, 1), (1, 2), (2, 4), (4, 8)]:
        tv, wv = _gl(lo, hi, nn); xv = 10**tv; jac = wv*xv*np.log(10)
        g = xv**-4.0; qv_ = Q*xv
        val = 4*np.pi*A0_np*(8*np.pi**2/qv_**2*F_np(g) + 2*Cq(qv_)*JC_np(g)/qv_**2
                             + 104*np.pi**2/27*Phi4_np(g)/qv_**4 + 6*np.pi**2*Q**4*Psi_np(g)/qv_**8)*Q
        tot += np.sum(jac*val)
    return tot
R3s = {}
print("   Q        R1 -> A1        R2 -> A2        R3 -> A3")
for Q_ in [3, 10, 30, 100, 300, 1000]:
    S_ = S_comp(Q_)
    R1_, R2_, R3_ = Q_*S_, Q_**2*(S_ - A1_np/Q_), Q_**3*(S_ - A1_np/Q_ - A2_np/Q_**2)
    R3s[Q_] = R3_
    print("%6d   %12.6f   %12.6f   %12.5f" % (Q_, R1_, R2_, R3_))
print("targets:  A1 = %.6f   A2 = %.6f   A3 = %.5f" % (A1_np, A2_np, A3_np))
check("Composite R3 at Q=1000 agrees with its coefficient integral to 1e-4", abs(R3s[1000]-A3_np)<1e-4, kind="MODEL")
drift = abs(R3s[1000] - R3s[100])
print("drift of R_3 over one decade of Q: %.2e   (a Q^-3 log Q term of coefficient c would give 2.3 c)" % drift)
check("Composite-model R3 drift is below 1e-4 on Q=100..1000; not a test of the exact remainder", drift<1e-4, kind="MODEL")

NUM  PASS  C(q) stable branches vs defining overlap integral, q=0   [|a-b|/max(1,|b|) = 6.35e-17; tolerance = 2e-08]
NUM  PASS  C(q) stable branches vs defining overlap integral, q=0.0001   [|a-b|/max(1,|b|) = 2.53e-17; tolerance = 2e-08]
NUM  PASS  C(q) stable branches vs defining overlap integral, q=1   [|a-b|/max(1,|b|) = 9.5e-17; tolerance = 2e-08]
NUM  PASS  C(q) stable branches vs defining overlap integral, q=2   [|a-b|/max(1,|b|) = 2.06e-17; tolerance = 2e-08]
NUM  PASS  C(q) stable branches vs defining overlap integral, q=20   [|a-b|/max(1,|b|) = 5.86e-17; tolerance = 2e-08]
NUM  PASS  C(q) stable branches vs defining overlap integral, q=100   [|a-b|/max(1,|b|) = 3.71e-20; tolerance = 2e-08]
   Q        R1 -> A1        R2 -> A2        R3 -> A3
     3      -0.084116       0.391732       -0.42480
    10      -0.165542       0.491514       -0.41819
    30      -0.197380       0.519419       -0.41742
   100      -0.209402       0.529160       -0.41733
   300      -0.212921       0.

<a id="check-24"></a>
## Check 24 — Kinematic expansion and small-momentum overlap

The code expands the two same-sphere denominators symbolically, verifies the quadratic combination $p_z^2+p_zk_z+k_z^2$, and removes odd terms under simultaneous inversion. It also checks the small-$q$ overlap numerically on the declared finite grid.

In [24]:
# --- 24.1 the same-sphere kinematic expansion
uK, qK, pzK, kzK, tK = sp.symbols('u q p_z k_z t', real=True)
AK = sp.I*uK - R(1,2); aK = pzK/qK; bK = kzK/qK
lhsK = (1/((AK - aK)*(AK - bK))).subs({pzK: tK*pzK, kzK: tK*kzK})
rhsK = (AK**-2 + (aK + bK)*AK**-3 + (aK**2 + aK*bK + bK**2)*AK**-4
        + (aK**3 + aK**2*bK + aK*bK**2 + bK**3)*AK**-5).subs({pzK: tK*pzK, kzK: tK*kzK})
for nK in range(4):
    exact("Eq.(A.8)  Taylor coefficient of order %d in (p_z,k_z)/q of [(A-a)(A-b)]^-1" % nK,
          sp.diff(lhsK, tK, nK).subs(tK, 0) - sp.diff(rhsK, tK, nK).subs(tK, 0))
exact("Eq.(32)  Re A^-2 = (1/4-u^2)/(1/4+u^2)^2", sp.re(AK**-2) - (R(1,4) - uK**2)/(R(1,4) + uK**2)**2)
odd = (aK + bK) + (aK**3 + aK**2*bK + aK*bK**2 + bK**3)
check("Sec.III.C  orders 1 and 3 are odd in (p_z,k_z): they cancel on the inversion-symmetric same-sphere sectors",
      sp.simplify(odd.subs({pzK: -pzK, kzK: -kzK}) + odd) == 0)

# --- 24.2 Z(q) cross-check with the companion manuscript
def C_closed(q):
    q = np.asarray(q, float)
    Lm = np.log(np.abs(2 - q)); Lp = np.log(q + 2); lq = np.log(q)
    inner = q*(q**4*Lm - 2*q**2*(q**2 - 20)*lq - 20*q**2*Lm + 4*q**2 + 40*q*Lm
               - q*(-q**3 + 20*q + 40)*Lp + 88) + 32*(Lp - Lm)
    return np.pi**2*inner/(30*q)
num("Eq.(36)  C(q) -> 4 pi^2 as q -> 0, i.e. Z(0) = 0  (evaluated at q = 1e-3; the closed form loses digits below that)",
    mp.mpf(float(C_closed(1e-3))), 4*mp.pi**2, tol=1e-5)
consts = []
for qv_ in [1e-2, 1e-3, 1e-4]:
    Zv = 8*np.pi**2 - 2*C_closed(qv_)
    cst = Zv/(8*np.pi**2/3*qv_**2) - np.log(1/qv_)
    consts.append(cst)
    print("  q=%8.0e :  Z = %.6e    Z/[(8pi^2/3) q^2] - log(1/q) = %.4f" % (qv_, Zv, cst))
check("Eq.(36) vs JPA Eq.(36)  Z = (8 pi^2/3) q^2 [log(1/q) + const] with a stable constant over three decades",
      max(consts) - min(consts) < 0.02)

EXACT  PASS  Eq.(A.8)  Taylor coefficient of order 0 in (p_z,k_z)/q of [(A-a)(A-b)]^-1   [simplify -> 0]
EXACT  PASS  Eq.(A.8)  Taylor coefficient of order 1 in (p_z,k_z)/q of [(A-a)(A-b)]^-1   [simplify -> 0]
EXACT  PASS  Eq.(A.8)  Taylor coefficient of order 2 in (p_z,k_z)/q of [(A-a)(A-b)]^-1   [simplify -> 0]
EXACT  PASS  Eq.(A.8)  Taylor coefficient of order 3 in (p_z,k_z)/q of [(A-a)(A-b)]^-1   [simplify -> 0]
EXACT  PASS  Eq.(32)  Re A^-2 = (1/4-u^2)/(1/4+u^2)^2   [simplify -> 0]
CHECK  PASS  Sec.III.C  orders 1 and 3 are odd in (p_z,k_z): they cancel on the inversion-symmetric same-sphere sectors
NUM  PASS  Eq.(36)  C(q) -> 4 pi^2 as q -> 0, i.e. Z(0) = 0  (evaluated at q = 1e-3; the closed form loses digits below that)   [|a-b|/max(1,|b|) = 2.81e-6; tolerance = 1e-05]
  q=   1e-02 :  Z = 1.613777e-02    Z/[(8pi^2/3) q^2] - log(1/q) = 1.5264
  q=   1e-03 :  Z = 2.219802e-04    Z/[(8pi^2/3) q^2] - log(1/q) = 1.5265
  q=   1e-04 :  Z = 2.825811e-06    Z/[(8pi^2/3) q^2] - log(1/q)

<a id="check-25"></a>
## Check 25 — Separate loop and response spin sums

Rescaling the two-component response into a chosen loop sphere fixes the response factor $M_5$ separately from the geometric factor $M_4$. Their product multiplies the Lindhard correction, while $M_6$ multiplies the cross and kinematic corrections.

The code uses arbitrary positive $x_+,x_-$ before imposing $M_3=1$ and checks the small-polarization expansion of $\mathcal S_3$. Full polarization alone cannot distinguish the two factors because both equal 2 there.

In [25]:
# ---------- (a) symbolic ----------
xp_, xm_, qS_, uS_, q_tf_sqS_, xs_ = sp.symbols('x_+ x_- q u q_tf_sq x_sigma', positive=True)
g4S = -4*(12*uS_**2 - 1)/(5*(1 + 4*uS_**2)**2)
def lindhard_shape_expansion(Qv, Uv):                                   # large-q Lindhard expansion, Check 2
    return R(4,3)/(Qv**2*(1 + 4*Uv**2))*(1 + g4S.subs(uS_, Uv)/Qv**2)
def lindhard_shape_spin(qp, nup):                                 # Eq.(24)
    return sum(xx*lindhard_shape_expansion(qp/xx, (nup/xx**2)/(qp/xx)**2) for xx in (xp_, xm_))/2
c_sigma = sp.simplify(q_tf_sqS_*lindhard_shape_spin(xs_*qS_, xs_**2*qS_**2*uS_)/(xs_*qS_)**2)     # Eq.(25): c_sigma(q,nu) = c_zeta(x q, x^2 nu)
M3S = (xp_**3 + xm_**3)/2; M5S = (xp_**5 + xm_**5)/2
Qs4 = R(4,3)*q_tf_sqS_                                   # q_*^4
exact("Eqs.(24)-(26)  c_sigma(q, q^2 u) = (q*/(x_sigma q))^4/(1+4u^2) [M_3 + g4(u) M_5/(x_sigma^2 q^2)]",
      c_sigma - Qs4/(xs_**4*qS_**4*(1 + 4*uS_**2))*(M3S + g4S*M5S/(xs_**2*qS_**2)))
nuS_ = sp.symbols('nu', positive=True)
c_out = sp.simplify(q_tf_sqS_*sum(xx*((xs_*qS_/xx)**2/(3*(xs_**2*nuS_/xx**2)**2)) for xx in (xp_, xm_))/2/(xs_*qS_)**2)
exact("Eqs.(24)-(26)  outer region: c_sigma -> q_tf_sq M_3/(3 x_sigma^4 nu^2)", c_out - q_tf_sqS_*M3S/(3*xs_**4*nuS_**2))
# moment bookkeeping: per-spin scalings x, x^2, x^3, x^3, M5 x  weighted by x^3/2
Mn_ = lambda n_: (xp_**n_ + xm_**n_)/2
exact("Eq.(46)  leading  ~ x_sigma          -> sum x^3/2 * x     = M_4", sum(xx**3/2*xx for xx in (xp_, xm_)) - Mn_(4))
exact("Eq.(46)  outer    ~ x_sigma^2        -> sum x^3/2 * x^2   = M_5", sum(xx**3/2*xx**2 for xx in (xp_, xm_)) - Mn_(5))
exact("Eq.(46)  cross+kin ~ x_sigma^3       -> sum x^3/2 * x^3   = M_6", sum(xx**3/2*xx**3 for xx in (xp_, xm_)) - Mn_(6))
exact("Eq.(46)  Lindhard ~ (M_5/x^2) x^3    -> sum x^3/2 * M_5 x = M_4 M_5", sum(xx**3/2*M5S*xx for xx in (xp_, xm_)) - Mn_(4)*Mn_(5))
zS_ = sp.symbols('zeta', positive=True)
S3z = (1160*Mn_(6) + 891*Mn_(4)*Mn_(5))/2051
S3ser = sp.series(S3z.subs({xp_: (1+zS_)**R(1,3), xm_: (1-zS_)**R(1,3)}), zS_, 0, 4).removeO()
print("S_3(zeta) =", sp.nsimplify(sp.expand(S3ser), rational=False), "   (all-M6: 1 + zeta^2 ; all-M4M5: 1 + 7/9 zeta^2)")
exact("Eq.(5)  S_3 = 1 + (1853/2051) zeta^2 + O(zeta^4)", sp.expand(S3ser).coeff(zS_, 2) - R(1853, 2051))



EXACT  PASS  Eqs.(24)-(26)  c_sigma(q, q^2 u) = (q*/(x_sigma q))^4/(1+4u^2) [M_3 + g4(u) M_5/(x_sigma^2 q^2)]   [simplify -> 0]
EXACT  PASS  Eqs.(24)-(26)  outer region: c_sigma -> q_tf_sq M_3/(3 x_sigma^4 nu^2)   [simplify -> 0]
EXACT  PASS  Eq.(46)  leading  ~ x_sigma          -> sum x^3/2 * x     = M_4   [simplify -> 0]
EXACT  PASS  Eq.(46)  outer    ~ x_sigma^2        -> sum x^3/2 * x^2   = M_5   [simplify -> 0]
EXACT  PASS  Eq.(46)  cross+kin ~ x_sigma^3       -> sum x^3/2 * x^3   = M_6   [simplify -> 0]
EXACT  PASS  Eq.(46)  Lindhard ~ (M_5/x^2) x^3    -> sum x^3/2 * M_5 x = M_4 M_5   [simplify -> 0]
S_3(zeta) = 1853*zeta**2/2051 + 1    (all-M6: 1 + zeta^2 ; all-M4M5: 1 + 7/9 zeta^2)
EXACT  PASS  Eq.(5)  S_3 = 1 + (1853/2051) zeta^2 + O(zeta^4)   [simplify -> 0]


<a id="check-26"></a>
## Check 26 — RPA-ring reference

A separate one-dimensional ring integral checks the quoted leading coefficient $-0.803\,r_s^{-3/4}$ Ry and the asymptotic kite-to-ring ratio. This comparison concerns the retained diagrams.

In [26]:
for gv in ['0.3', '2', '10']:
    gv = mp.mpf(gv)
    num("Ring  int du [ln(1+g/(1+4u^2)) - g/(1+4u^2)] = (pi/2)[sqrt(1+g)-1-g/2], g=%s" % gv,
        mp.quad(lambda uu: mp.log1p(gv/(1 + 4*uu**2)) - gv/(1 + 4*uu**2), [0, 1, mp.inf]),
        mp.pi/2*(mp.sqrt(1 + gv) - 1 - gv/2), tol=1e-20)
W_cl = -sp.sqrt(pi)*sp.gamma(R(3,4))**2/5
exact("Ring  (pi/8) B(-5/4, 3/4) = -sqrt(pi) Gamma(3/4)^2/5", sp.gammasimp(sp.expand_func(pi/8*sp.beta(-R(5,4), R(3,4)) - W_cl)))
def _w_int(x):                      # large-x series: sqrt(1+g)-1-g/2 = -g^2/8 + g^3/16 - 5g^4/128, g = x^-4
    g_ = x**-4
    return x**4*(mp.pi/2)*(-g_**2/8 + g_**3/16 - 5*g_**4/128) if g_ < mp.mpf('1e-6') \
           else x**4*(mp.pi/2)*(mp.sqrt(1 + g_) - 1 - g_/2)
num("Ring  W = (pi/2) int x^4 [sqrt(1+x^-4) - 1 - x^-4/2] dx = -sqrt(pi) Gamma(3/4)^2/5",
    mp.quad(_w_int, [0, 1, 2, 10, mp.inf]), mp.mpf(sp.N(W_cl, 30)), tol=1e-15)
alphaL_s = (4/(9*pi))**R(1,3); a_sym = 4*alphaL_s/pi          # exact symbolic alpha_L and a (Eq. 18)
ring_coef = 3/(2*pi*alphaL_s**2)*(4*a_sym/3)**R(5,4)*W_cl
print("closed form of the low-density ring coefficient:", sp.N(ring_coef, 12), "Ry     (quoted value: -0.803)")
num("Low-density RPA ring coefficient = -0.803 r_s^(-3/4) Ry (quoted value reproduced)",
    mp.mpf(sp.N(ring_coef, 25)), mp.mpf('-0.803'), tol=1e-3)

print("The ratio of leading asymptotic terms grows as r_s**(1/2). Equating them at r_s≈13 is outside their controlled regime and is not a physical crossing prediction.")

NUM  PASS  Ring  int du [ln(1+g/(1+4u^2)) - g/(1+4u^2)] = (pi/2)[sqrt(1+g)-1-g/2], g=0.3   [|a-b|/max(1,|b|) = 1.53e-31; tolerance = 1e-20]
NUM  PASS  Ring  int du [ln(1+g/(1+4u^2)) - g/(1+4u^2)] = (pi/2)[sqrt(1+g)-1-g/2], g=2.0   [|a-b|/max(1,|b|) = 4.93e-32; tolerance = 1e-20]
NUM  PASS  Ring  int du [ln(1+g/(1+4u^2)) - g/(1+4u^2)] = (pi/2)[sqrt(1+g)-1-g/2], g=10.0   [|a-b|/max(1,|b|) = 1.87e-31; tolerance = 1e-20]
EXACT  PASS  Ring  (pi/8) B(-5/4, 3/4) = -sqrt(pi) Gamma(3/4)^2/5   [simplify -> 0]
NUM  PASS  Ring  W = (pi/2) int x^4 [sqrt(1+x^-4) - 1 - x^-4/2] dx = -sqrt(pi) Gamma(3/4)^2/5   [|a-b|/max(1,|b|) = 1.13e-25; tolerance = 1e-15]
closed form of the low-density ring coefficient: -0.803078642296 Ry     (quoted value: -0.803)
NUM  PASS  Low-density RPA ring coefficient = -0.803 r_s^(-3/4) Ry (quoted value reproduced)   [|a-b|/max(1,|b|) = 7.86e-5; tolerance = 0.001]
The ratio of leading asymptotic terms grows as r_s**(1/2). Equating them at r_s≈13 is outside their controlled r

<a id="check-27"></a>
## Check 27 — Remainder estimates and an endpoint counterexample

Appendix C uses three estimates distinct:
1. The bounded-momentum interval contributes at most $O_M(Q^{-4})$ to the difference between exact and composite densities. Both low and high frequencies are bounded before integration.
2. For $q\ge M$, the omitted terms obey $Kq^{-6}\min(1,(q/Q)^4)$ after the frequency integral. Its integral is $1/(MQ^4)-4/(5Q^5)$.
3. The cross-composite integral differs from $A_2/Q^2+A_{3,\rm cross}/Q^3$ by $O(Q^{-5})$. Use $C(q)-V_B^2/q^2=O(q^{-4})$ and $H(x)-4/3=O(x^4)$ at zero; the small-$x$ endpoint must be subtracted before bounding.

Together these give $O(Q^{-4})$, hence $O(r_s^{-1})$. The code below checks the elementary identities and endpoint expansions supporting this argument. **The uniform estimates are established in the manuscript, not by these numerical checks.**

A concrete counterexample to the statement “the outer series has only $Q^{-2},Q^{-6},\ldots$” is useful. For a fixed positive energy $d$, approximate only the response by its high-frequency form, and retain the energy denominator:
\[
I_d(Q)=\int_0^\infty\frac{d\nu}{\pi}R\!\left(\frac{Q^4}{4\nu^2}\right)\frac{d^2-\nu^2}{(d^2+\nu^2)^2}.
\]
At a fixed coupling let $b=Q^2\sqrt\lambda/2$. A residue or elementary rational integral gives
$\int(d\nu/\pi)\nu^2(\nu^2+b^2)^{-1}(d^2-\nu^2)/(d^2+\nu^2)^2=-b/[2(b+d)^2]$.
Averaging $2\lambda d\lambda$ and putting $\lambda=t^2$ gives
\[
 I_d(Q)=-\frac{2}{b_0}\int_0^1\frac{t^4}{(t+d/b_0)^2}\,dt
 =-\frac4{3Q^2}+\frac{8d}{Q^4}+O(Q^{-6}),\quad b_0=Q^2/2.
\]
Thus an outer $Q^{-4}$ term is possible. This example does not calculate the kite's fourth coefficient. A finite-frequency region cannot be discarded by power counting only at $\nu\sim Q^2$.

In [27]:

Qb,Mb,xb,bb,dd,tb=sp.symbols('Q M x b d t',positive=True)
exact('Appendix C radial envelope integral',sp.integrate(xb**-2/Qb**4,(xb,Mb,Qb))+sp.integrate(xb**-6,(xb,Qb,oo))-(1/(Mb*Qb**4)-4/(5*Qb**5)))
Hs=xb**-2*JC_cl.subs(g,xb**-4)
exact('H(x)-4/3 starts at -2 x^4',sp.limit((Hs-R(4,3))/xb**4,xb,0)+2)
exact('F radial profile starts -4/3 + 4x^2',sp.limit((xb**-2*F_cl.subs(g,xb**-4)+R(4,3))/xb**2,xb,0)-4)
exact('integral of (1+4u^2)^-2',sp.integrate((1+4*u*u)**-2,(u,0,oo))-pi/8)
exact('integral of (1+4u^2)^-1',sp.integrate((1+4*u*u)**-1,(u,0,oo))-pi/4)
for bval,dval in [(2,1),(7,.3)]:
    num('fixed-coupling retained-denominator integral',
        mp.quad(lambda z:z*z/(z*z+bval*bval)*(dval*dval-z*z)/(dval*dval+z*z)**2/mp.pi,[0,dval,bval,mp.inf]),
        -mp.mpf(bval)/(2*(mp.mpf(bval)+mp.mpf(dval))**2),tol=1e-15)
def outer_example(Q,d=1):
    b0=mp.mpf(Q)**2/2
    return -2/b0*mp.quad(lambda t:t**4/(t+mp.mpf(d)/b0)**2,[0,1])
for Qval in [20,50,100]:
    print('Q=%g: Q^4 [I_d + 4/(3Q^2)] = %s; limit 8d=8'%(Qval,mp.nstr(Qval**4*(outer_example(Qval)+mp.mpf(4)/(3*Qval**2)),12)))
num('outer model has a nonzero Q^-4 coefficient at d=1',100**4*(outer_example(100)+mp.mpf(4)/30000),8,tol=1e-3)


EXACT  PASS  Appendix C radial envelope integral   [simplify -> 0]
EXACT  PASS  H(x)-4/3 starts at -2 x^4   [simplify -> 0]
EXACT  PASS  F radial profile starts -4/3 + 4x^2   [simplify -> 0]
EXACT  PASS  integral of (1+4u^2)^-2   [simplify -> 0]
EXACT  PASS  integral of (1+4u^2)^-1   [simplify -> 0]
NUM  PASS  fixed-coupling retained-denominator integral   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-15]
NUM  PASS  fixed-coupling retained-denominator integral   [|a-b|/max(1,|b|) = 9.99e-20; tolerance = 1e-15]
Q=20: Q^4 [I_d + 4/(3Q^2)] = 7.88404363895; limit 8d=8
Q=50: Q^4 [I_d + 4/(3Q^2)] = 7.98094094128; limit 8d=8
Q=100: Q^4 [I_d + 4/(3Q^2)] = 7.99521058233; limit 8d=8
NUM  PASS  outer model has a nonzero Q^-4 coefficient at d=1   [|a-b|/max(1,|b|) = 0.000599; tolerance = 0.001]


<a id="check-31"></a>
## Check 31 — Exact derivation checks: residues, finite parts and proof certificates

The following independent symbolic calculation derives the transformations leading to the coefficients.
It derives residues from meromorphic frequency integrands, including the simple
and double Lindhard poles separately and their coalescent limit. It integrates
the coupling parameter to obtain $\Phi_4$ and $\Psi$, verifies their endpoints,
and derives the sphere moments from the lens geometry.

For the cross finite part, put $f(s)=(s^2-1)^{-1/4}$ and
$h(s)=(s-1)^{3/4}(s+1)^{-1/4}$. The certificate
$2h'=f+f/(s+1)$ reduces the finite part to a convergent Beta integral;
the transformed cutoff has no extra constant. For the Euler integral,
a rational total-derivative certificate supplies the reduction to Gamma functions.
The static tail and the positive integrands behind the $R$ derivative bounds
are checked as well.

The code below is the standalone `analytic_audit.py` embedded in full. It runs
in a function so its symbols cannot overwrite earlier notebook variables.
The contour domain is $s=\sqrt{1+m}>1$; limits at $m=0$ are taken only after
summing the residues. Algebraic checks do not by themselves certify contour
closure, absolute integrability or uniform estimates. The contour and domination arguments are in Appendices A and C.


In [28]:
"""Independent mathematical audit of the supplied PRB manuscript.
Domain: real u; s=sqrt(1+m)>1; w in (0,1); q,Q,kappa>0.
Run: python analytic_audit.py. Requires SymPy only. All checks use exact arithmetic.
Contour closure, endpoint domination and integration interchange are justified
in mathematical_audit_ja.md; this script checks their algebraic certificates.
"""

def run_analytic_audit(record=None):
    import sympy as S
    j=S.I
    u,s,w,z,k,r,t,g=S.symbols('u s w z k r t g', positive=True)
    checks=[]
    def eq(name,a,b=0):
        diff=S.factor(S.cancel(a-b))
        if diff!=0: diff=S.simplify(diff)
        assert diff==0,(name,diff)
        checks.append(name)
        record(name, diff == 0) if record else print('EXACT PASS:',name)
    B=1+4*u*u; A=j*u-S.Rational(1,2); m=s*s-1
    us=j*s/2
    # Poles are determined from the unsimplified frequency integrands, not their proposed results.
    f2=B/(B+m)/A**2
    f4=B/(B+m)/A**4
    fc=4/(B+m)
    fL=-S.Rational(16,15)*(12*u*u-1)/(B*(B+m)**2*A**2)
    def simple(f,p): return S.cancel(f*(u-p)).subs(u,p)
    def double(f,p): return S.diff(S.cancel(f*(u-p)**2),u).subs(u,p)
    h2=S.factor(j*simple(f2,us));h4=S.factor(j*simple(f4,us))
    eq('screening residue: same sphere',h2,-m/(s*(s+1)**2))
    eq('screening residue: kinematic',h4,-4*m/(s*(s+1)**4))
    eq('screening residue: cross sphere',j*simple(fc,us),1/s)
    l0=S.factor(j*simple(fL,j/2)); ls=S.factor(j*double(fL,us))
    eq('Lindhard simple pole',l0,16/(15*m*m))
    eq('Lindhard double pole',ls,-8*(9*s**4-6*s**3+8*s*s-2*s-1)/(15*s**3*(s-1)**2*(s+1)**4))
    hL=S.factor(l0+ls)
    eq('Lindhard sum before coalescent limit',hL,8*(2*s**3-s*s+4*s+1)/(15*s**3*(s+1)**4))
    eq('coalescent m=0 limit',S.limit(hL,s,1),S.Rational(1,5))
    # Coupling average: t=sqrt(1+lambda*g). Integrate rational t functions.
    def avg(h,weight):
        return S.factor(S.integrate(4*t*(t*t-1)**weight*h.subs(s,t),(t,1,s)))
    F=S.factor(avg(h2,1)/(s*s-1)**2)
    JC=S.factor(avg(1/s,1)/(s*s-1)**2)
    Phi4=S.factor(avg(h4,1)/(s*s-1)**2)
    Psi=S.factor(-avg(hL,2)/(s*s-1)**3)
    eq('F coupling integral',F,-S.Rational(4,3)*(s-1)/(s+1)**2)
    eq('JC coupling integral',JC,S.Rational(4,3)*(s+2)/(s+1)**2)
    print('Phi4(s)=',Phi4,'\nPsi(s)=',Psi)
    # Endpoint checks on derived expressions, with s=sqrt(1+x^-4).
    x=S.symbols('x',positive=True)
    sx=S.sqrt(1+x**4)/x**2
    for name,expr,want in [('F origin',F.subs(s,sx)/x**2,-S.Rational(4,3)),('JC origin',JC.subs(s,sx)/x**2,S.Rational(4,3)),('Phi4 origin',Phi4.subs(s,sx)/x**6,-16),('Psi origin',Psi.subs(s,sx)/x**8,-S.Rational(32,15))]:
        eq(name,S.limit(expr,x,0),want)
    # Independent logarithm expansion, including the next omitted coefficient.
    logratio=S.log((u*u+(S.Rational(1,2)+z)**2)/(u*u+(S.Rational(1,2)-z)**2))
    coef1=S.diff(logratio,z).subs(z,0)
    coef3=S.diff(logratio,z,3).subs(z,0)/6
    coef5=S.diff(logratio,z,5).subs(z,0)/120
    eq('Lindhard q^-2',coef1/6,4/(3*B))
    eq('Lindhard q^-4',coef3/10,16*(1-12*u*u)/(15*B**3))
    print('Lindhard q^-6 coefficient=',S.factor(coef5/14))
    # Geometric moments by explicit lens integration, independent of notebook constants.
    V=S.pi*(4+r)*(2-r)**2/12
    rho2=1-(z+r/2)**2
    a=1-r/2
    Jlens=2*S.pi*S.integrate(z*z*rho2+rho2**2/2,(z,0,a))
    eq('sphere Coulomb moment I0',4*S.pi*S.integrate(V,(r,0,2)),4*S.pi**2)
    eq('sphere quadratic moment I2',4*S.pi*S.integrate(Jlens+r*r*V/12,(r,0,2)),52*S.pi**2/27)
    eq('integrated overlap C',S.pi**3*S.integrate(r*V,(r,0,2)),8*S.pi**4/15)
    # Euler reduction certificate: differentiation of a rational function times a fixed weight.
    ld=-1/(2*w)-3/(4*(1-w))-13/(4*(1+w))
    r1=2*w*(1+w)*(11*w+21)/45
    r2=2*w*(1+w)*(7*w-3)/45
    r3=w*(1+w)*(15*w*w-8*w-3)/15
    for idx,(rr,pp,kk) in enumerate([(r1,1,S.Rational(16,15)),(r2,w,S.Rational(2,15)),(r3,4*w*w+w**3,S.Rational(1,5))]):
        eq('Euler derivative certificate '+str(idx+1),S.diff(rr,w)+ld*rr,pp-kk*(1+w)**2/(2*(1-w)))
    rr=S.factor(2*r1-r2+r3)
    eq('combined Euler certificate',S.diff(rr,w)+ld*rr,2-w+4*w*w+w**3-S.Rational(11,10)*(1+w)**2/(1-w))
    print('combined Euler r(w)=',rr)
    eq('M rational factor',S.Rational(16,15)*S.Rational(11,10)*2,S.Rational(176,75))
    # Finite-part integral: an integration-by-parts certificate gives a convergent integral.
    f=(s-1)**(-S.Rational(1,4))*(s+1)**(-S.Rational(1,4))
    h=(s-1)**S.Rational(3,4)*(s+1)**(-S.Rational(1,4))
    eq('finite part integration-by-parts certificate',S.diff(h,s),f/2+f/(2*(s+1)))
    eq('finite part transformed cutoff has no constant',S.limit(2*h.subs(s,S.sqrt(1+x**-4))-2/x,x,0),0)
    # Hence Fp integral f = -integral f/(s+1), all at s=1..infty;
    # convergent Beta B(3/4,1/2)/sqrt(2).
    Gamma=S.gamma
    Cg=S.sqrt(2*S.pi)/S.pi**2*Gamma(S.Rational(3,4))/Gamma(S.Rational(1,4))
    def beta(a,b):return Gamma(a)*Gamma(b)/Gamma(a+b)
    fp=S.Rational(2,3)*(-beta(S.Rational(3,4),S.Rational(1,2))/S.sqrt(2)-beta(S.Rational(3,4),S.Rational(3,2))/(2*S.sqrt(2)))
    eq('finite part coefficient via convergent Beta integrals',S.expand_func(fp),-S.Rational(8,5)*S.pi**2*Cg)
    A0=3/(32*S.pi**5)
    kin=4*S.pi*A0*104*S.pi**2/27*S.Rational(2,5)*(-2*S.sqrt(2)*beta(S.Rational(7,4),S.Rational(3,2)))
    eq('A3kin Beta to Gamma',S.expand_func(kin),-S.Rational(416,675)*Cg)
    eq('A3 decomposition',S.Rational(32,15)+S.Rational(416,675)+S.Rational(264,125),S.Rational(16408,3375))
    eq('S3 M6 weight',(S.Rational(32,15)+S.Rational(416,675))/S.Rational(16408,3375),S.Rational(1160,2051))
    eq('S3 M4M5 weight',S.Rational(264,125)/S.Rational(16408,3375),S.Rational(891,2051))
    # Remainder envelope integrated explicitly; Q>M.
    Q,M=S.symbols('Q M',positive=True)
    eq('integrated large-q error envelope',S.integrate(1/(Q**4*z**2),(z,M,Q))+S.integrate(z**-6,(z,Q,S.oo)),1/(M*Q**4)-4/(5*Q**5))
    # Cross energy shifts include the occupation sign. The linear term is odd in d_z.
    pz,kz,eps,kap,qq,lam,cc=S.symbols('p_z k_z eps kappa q lambda c',real=True)
    Ap=j*u+S.Rational(1,2)
    cross=-S.re(1/((A-eps*pz)*(Ap-eps*kz)))
    eq('occupation-weighted cross denominator',cross.subs(eps,0),4/B)
    eq('cross linear energy shift',S.diff(cross,eps).subs(eps,0),-8*(pz-kz)/B**2)
    # Derivative inequalities have nonnegative integrands for c>=0, lambda in [0,1].
    rl=2*lam/(1+lam*cc)
    eq('R-c|Rprime| positive integrand',rl+cc*S.diff(rl,cc),2*lam/(1+lam*cc)**2)
    eq('2R-c^2 Rsecond positive integrand',2*rl-cc**2*S.diff(rl,cc,2),4*lam*(1+2*lam*cc)/(1+lam*cc)**3)
    eq('Phi4 weak-screening coefficient',S.limit(Phi4/(s*s-1),s,1),-S.Rational(1,6))
    eq('Psi zero-screening value',S.limit(Psi,s,1),-S.Rational(2,15))
    # Static q>2 integral: geometric expansion in d_z/q and d^2/q^2.
    dz,dsq=S.symbols('d_z d_squared',real=True)
    stat=1/((1+2*dz*eps+dsq*eps**2)*(1+dz*eps))
    eq('static leading integrand',stat.subs(eps,0),1)
    eq('static odd correction vanishes by inversion',S.diff(stat,eps).subs(eps,0),-3*dz)
    eq('static C_infty from two cross sectors',(4*S.pi/3)**2/(8*S.pi),2*S.pi/9)
    fi=S.Rational(1,2)+(kap/4+1/kap)*S.atan(kap/2)
    eq('static imaginary-pole uniqueness certificate',kap*S.diff(fi,kap)-2*fi,-S.Rational(1,2)-(kap/4+3/kap)*S.atan(kap/2))
    eq('static imaginary-pole asymptotic scale',S.limit(fi/kap,kap,S.oo),S.pi/8)
    eq('static imaginary-pole residue asymptote',S.limit(1/(1-kap*S.diff(fi,kap)/(2*fi)),kap,S.oo),2)
    print('ALL',len(checks),'EXACT CHECKS PASSED')
    return checks


AUDIT_CERTIFICATES = run_analytic_audit(lambda label, ok: check('Analytic audit: '+label, ok, kind='EXACT'))

# --- Unnumbered steps of Appendices A and C, checked here so that they are not taken on trust
s_u, w_u, t_u = sp.symbols('s w t', positive=True)
f_u = (s_u**2 - 1)**R(-1, 4); h_u = (s_u - 1)**R(3, 4)*(s_u + 1)**R(-1, 4)
exact("Eq.(A.6) 2 h'(s) = f(s) + f(s)/(s+1), f=(s^2-1)^(-1/4), h=(s-1)^(3/4)(s+1)^(-1/4)",
      (2*sp.diff(h_u, s_u) - f_u - f_u/(s_u + 1)).subs(s_u, 1 + t_u))
fs_n = lambda s: (s*s - 1)**mp.mpf(-0.25)
I_fs = mp.quad(lambda u: 4*u**3*(u**4*(2 + u**4))**mp.mpf(-0.25)/(2 + u**4), [0, 1, 3, mp.inf])   # s = 1 + u^4, s^2 - 1 = u^4 (2 + u^4) written without cancellation
num("Eq.(A.7) int_1^oo f/(s+1) ds = 2^(-1/2) B(3/4,1/2)", I_fs, mp.beta(mp.mpf(3)/4, mp.mpf(1)/2)/mp.sqrt(2), tol=1e-20)
num("Eq.(A.7) Fp int f ds = -2^(-1/2) B(3/4,1/2) equals the continuation 2^(1/2) B(3/4,-1/2)",
    -mp.beta(mp.mpf(3)/4, mp.mpf(1)/2)/mp.sqrt(2), mp.sqrt(2)*mp.gamma(mp.mpf(3)/4)*mp.gamma(-mp.mpf(1)/2)/mp.gamma(mp.mpf(1)/4), tol=1e-25)
hs_n = lambda s: (s - 1)**mp.mpf(0.75)*(s + 1)**mp.mpf(-0.25)
for eps_u in (mp.mpf('1e-3'), mp.mpf('1e-5')):
    S_eps = mp.sqrt(1 + eps_u**-4)
    num("Eq.(A.7) 2 h(S_eps) - 2/eps -> 0 at eps=%s" % mp.nstr(eps_u, 2), 2*hs_n(S_eps) - 2/eps_u, 0, tol=10*eps_u)
f_w = (1 - w_u)**R(3, 4)/(sp.sqrt(w_u)*(1 + w_u)**R(13, 4)); r_w = w_u*(1 + w_u)*(15*w_u**2 + 2*w_u + 27)/15
bracket = 2 - w_u + 4*w_u**2 + w_u**3 - R(11, 10)*(1 + w_u)**2/(1 - w_u)
exact("Eq.(A.16) (f r)' = f [2 - w + 4w^2 + w^3 - 11(1+w)^2/(10(1-w))]",
      sp.simplify(sp.diff(f_w*r_w, w_u)/f_w - bracket))
qc, dx_, dy_, dz_ = sp.symbols('q d_x d_y d_z', real=True)
Pp = dx_**2 + dy_**2 + (qc + dz_)**2; Pm = dx_**2 + dy_**2 + (qc - dz_)**2
exact("Eq.(C.7) 1/|q z + d|^2 - 1/|q z - d|^2 = -4 q d_z/(|q z + d|^2 |q z - d|^2)", 1/Pp - 1/Pm + 4*qc*dz_/(Pp*Pm))
xs_u = sp.symbols('x', positive=True); s_x = sp.sqrt(1 + xs_u**-4)
F_closed_s = -R(4, 3)*(s_x - 1)/(s_x + 1)**2; JC_closed_s = R(4, 3)*(s_x + 2)/(s_x + 1)**2
exact("Eq.(C.4) x^-2 F(x^-4) = -4/3 + 4x^2 + O(x^4)", sp.series(xs_u**-2*F_closed_s, xs_u, 0, 4).removeO() - (-R(4, 3) + 4*xs_u**2))
exact("Eq.(C.4) H(x) = x^-2 J_C(x^-4) = 4/3 - 2x^4 + O(x^6)", sp.series(xs_u**-2*JC_closed_s, xs_u, 0, 6).removeO() - (R(4, 3) - 2*xs_u**4))


EXACT  PASS  Analytic audit: screening residue: same sphere
EXACT  PASS  Analytic audit: screening residue: kinematic
EXACT  PASS  Analytic audit: screening residue: cross sphere
EXACT  PASS  Analytic audit: Lindhard simple pole
EXACT  PASS  Analytic audit: Lindhard double pole
EXACT  PASS  Analytic audit: Lindhard sum before coalescent limit
EXACT  PASS  Analytic audit: coalescent m=0 limit
EXACT  PASS  Analytic audit: F coupling integral
EXACT  PASS  Analytic audit: JC coupling integral
Phi4(s)= -16*(s**2 - 4*s*log(s + 1) + 2*s + 4*s*log(2) - 4*log(s + 1) - 3 + 4*log(2))/((s - 1)**2*(s + 1)**3) 
Psi(s)= -32*(s**4 - 8*s**3 + 24*s**2*log(s + 1) - 24*s**2*log(2) - 12*s**2 + 24*s*log(s + 1) - 24*s*log(2) + 20*s - 1)/(15*s*(s - 1)**3*(s + 1)**4)
EXACT  PASS  Analytic audit: F origin
EXACT  PASS  Analytic audit: JC origin
EXACT  PASS  Analytic audit: Phi4 origin
EXACT  PASS  Analytic audit: Psi origin
EXACT  PASS  Analytic audit: Lindhard q^-2
EXACT  PASS  Analytic audit: Lindhard q^-4
Lin

<a id="check-28"></a>
## Check 28 — Independent original-functional integrator

The following cell contains the full contents of `reviewer_numerics.py`; the notebook does not import that file. No $F,J_C,\Phi_4,\Psi,A_1,A_2,A_3$ occurs in the estimator.

For $q\le2$, draw $p$ from the mixture of uniform balls $B$ and $B-q\hat z$. Its density is $m(p)/(2V_B)$, so its weight is $2V_B/m(p)$; this is a mixture with importance weights, not a uniform union sampler. Draw $r=p-k$ with uniform direction and radius in $[0,q+2]$. Its density is $[4\pi(q+2)r^2]^{-1}$, cancelling the bare Coulomb singularity. Multiply the actual occupation signs $\Delta_p\Delta_k$. The union contains points of zero occupation difference; these correctly contribute zero.

For $q>2$ the four sectors are disjoint. Two same-sphere sectors use radius $r\in[0,2]$ around uniform $p\in B$ and weight $2V_B(8\pi)$. The two cross sectors draw the two balls uniformly and use weight $-2V_B^2/|p-k|^2$. In either branch the exact frequency kernel is
$(\delta_p\delta_k-\nu^2)/[(\delta_p^2+\nu^2)(\delta_k^2+\nu^2)]$.

The Lindhard function is integrated over $k$, splitting at $k=q/2$ if it lies in the ball. Frequency panels resolve particle-hole and plasma scales; the inverse-frequency last panel goes to infinity. The remaining finite $q$ interval is integrated by Simpson quadrature in $\log q$, splitting at $q=2$. No hard frequency cutoff is used.

Each full momentum integral is evaluated for independent Sobol scrambles; the **whole replicate is integrated before its standard error is formed**, preserving covariance from shared samples at different momentum nodes. Replicate standard errors describe sample variation; quadrature and endpoint changes are reported separately.

The bare-line test uses the analytic frequency integral $[\operatorname{sgn}(\delta_k)-\operatorname{sgn}(\delta_p)]/[2(\delta_p-\delta_k)]$ and compares the resulting full momentum integral to the Onsager value. This tests more than multiplying the prefactor in Check 1. Samples are processed in chunks of 2048 to bound memory.


In [29]:
"""Independent imaginary-axis kite estimator. No asymptotic coefficients enter it.
Deterministic q/nu quadrature, independently scrambled Sobol replicates in p,k.
The standard error describes replicate variation, not quadrature or cutoff bias.
"""
import numpy as np
from scipy.integrate import simpson
from scipy.special import roots_legendre
from scipy.stats import qmc

A_NORMAL = 3/(32*np.pi**5)
A_SCREEN = 4/np.pi*(4/(9*np.pi))**(1/3)
V_BALL = 4*np.pi/3

def coupling(c):
    c=np.asarray(c,dtype=float)
    if np.any(c<0): raise ValueError('screening strength must be nonnegative')
    out=np.empty_like(c); small=c<1e-3
    v=c[small]
    out[small]=sum(2*(-v)**j/(j+2) for j in range(9))
    v=c[~small]; out[~small]=2*(v-np.log1p(v))/v**2
    return out

def gauss(a,b,n):
    t,w=roots_legendre(n)
    return (a+b)/2+(b-a)*t/2,(b-a)*w/2

def lindhard(q,nu,nk=48):
    """Eq. (23), direct k integral; log1p avoids high-frequency cancellation."""
    nu=np.atleast_1d(nu)
    edges=sorted(set([0., min(q/2,1.), 1.]))
    ans=np.zeros_like(nu)
    for lo,hi in zip(edges[:-1],edges[1:]):
        k,w=gauss(lo,hi,nk)
        den=(q*k[:,None]-q*q/2)**2+nu[None,:]**2
        ans+=np.sum((w*k)[:,None]*np.log1p(2*q**3*k[:,None]/den),axis=0)/(2*q)
    return ans

def spin_response(q,nu,zeta,nk=48):
    return sum(x*lindhard(q/x,nu/x**2,nk)/2
               for x in [(1+zeta)**(1/3),(1-zeta)**(1/3)] if x>0)

def frequency_grid(q,plasma,order=12):
    """Resolve low particle-hole and plasma scales; inverse-nu tail to infinity."""
    edge=np.unique([0., q*1e-4,q*1e-2,q,q*q,plasma,max(q*q,q,plasma)*10])
    xx=[];ww=[]
    for lo,hi in zip(edge[:-1],edge[1:]):
        if hi>lo:
            x,w=gauss(lo,hi,order);xx.extend(x);ww.extend(w)
    t,w=gauss(0,1,order);tail=edge[-1]
    xx.extend(tail/t);ww.extend(w*tail/t**2)
    return np.asarray(xx),np.asarray(ww)

def sphere(u):
    z=2*u[:,0]-1;ph=2*np.pi*u[:,1];rho=np.sqrt(np.maximum(0,1-z*z))
    return np.column_stack([rho*np.cos(ph),rho*np.sin(ph),z])

def ball(u):
    return sphere(u[:,:2])*u[:,2,None]**(1/3)

def kernel_integral(dp,dk,nu,pref):
    # Samples are chunked by the caller; arrays never span all samples and nodes.
    a=dp[:,None];b=dk[:,None];v=nu[None,:]
    return np.sum(((a*b-v*v)/((a*a+v*v)*(b*b+v*v)))*pref[None,:],axis=1)

def node_estimate(q,rs,u,order=12,nk=48,zeta=0.,spin_scale=1.,bare=False,chunk=2048):
    """Unit loop; c=c_zeta(x_sigma*q,x_sigma^2*nu). All occupation sectors."""
    if not bare:
        nu,w=frequency_grid(q,np.sqrt(A_SCREEN*rs/3)/spin_scale**2,order)
        c=A_SCREEN*rs*spin_response(spin_scale*q,spin_scale**2*nu,zeta,nk)/(spin_scale*q)**2
        pref=coupling(c)*w/np.pi
    def freq(a,b):
        if bare:
            den=2*(a-b)
            return np.divide(np.sign(b)-np.sign(a),den,out=np.zeros_like(a),where=den!=0)
        return kernel_integral(a,b,nu,pref)
    total=0.
    for start in range(0,len(u),chunk):
        h=u[start:start+chunk];p=ball(h[:,:3])
        if q>2:
            # Two same-ball terms, with 1/|p-k|^2 absorbed by uniform radius.
            k=p-2*h[:,5,None]*sphere(h[:,3:5]);inside=np.sum(k*k,axis=1)<1
            a=p[:,2]*q+q*q/2;b=k[:,2]*q+q*q/2
            same=2*V_BALL*8*np.pi*np.where(inside,freq(a,b),0.)
            # Two cross-ball terms; the balls are disjoint in this branch.
            k=ball(h[:,3:6]);k[:,2]-=q
            b=k[:,2]*q+q*q/2
            cross=-2*V_BALL**2*freq(a,b)/np.sum((p-k)**2,axis=1)
            total+=np.sum(same+cross)
        else:
            p[:,2]-=q*(h[:,6]<.5)
            ps=p.copy();ps[:,2]+=q
            np0=np.sum(p*p,axis=1)<1;np1=np.sum(ps*ps,axis=1)<1
            mult=np0.astype(float)+np1.astype(float)
            wp=2*V_BALL/mult
            k=p-(q+2)*h[:,5,None]*sphere(h[:,3:5]);ks=k.copy();ks[:,2]+=q
            dp=np0.astype(float)-np1.astype(float)
            dk=(np.sum(k*k,axis=1)<1).astype(float)-(np.sum(ks*ks,axis=1)<1).astype(float)
            a=p[:,2]*q+q*q/2;b=k[:,2]*q+q*q/2
            total+=np.sum(wp*4*np.pi*(q+2)*dp*dk*freq(a,b))
    return total/len(u)/(16*np.pi)

def energy(rs,zeta=0.,power=13,reps=8,nq=24,order=12,nk=48,qmin=.002,qmax=None,seed=714203,bare=False):
    """nq even subintervals on each side of q=2. Same scrambles at all q nodes.
    Propagate covariance by integrating each entire replicate first.
    """
    if nq%2: raise ValueError('nq must be even for nested Simpson grids')
    if reps<2: raise ValueError('at least two independent scrambles required')
    Q=(4*A_SCREEN*rs/3)**.25
    result=np.zeros(reps);profiles=[]
    for spin_scale in [(1+zeta)**(1/3),(1-zeta)**(1/3)]:
        if spin_scale==0:continue
        # At zeta=0 the two identical loops sum to one; do not compute twice.
        if zeta==0 and profiles:break
        upper=qmax if qmax is not None else 30*max(Q/spin_scale,1)
        logs=np.r_[np.linspace(np.log(qmin),np.log(2),nq+1),np.linspace(np.log(2),np.log(upper),nq+1)[1:]]
        qs=np.exp(logs);vals=np.empty((reps,len(qs)))
        for j in range(reps):
            u=qmc.Sobol(7,scramble=True,seed=seed+j).random_base2(power)
            for k,q in enumerate(qs):
                vals[j,k]=q*node_estimate(q,rs,u,order,nk,zeta,spin_scale,bare)
        integrals=(simpson(vals[:,:nq+1],x=logs[:nq+1],axis=1)+simpson(vals[:,nq:],x=logs[nq:],axis=1))*6/np.pi**3
        result+=(1 if zeta==0 else spin_scale**3/2)*integrals
        profiles.append({'q':qs,'qX':vals,'spin_scale':spin_scale})
    return {'mean':result.mean(),'se':result.std(ddof=1)/np.sqrt(reps),'replicates':result,'profiles':profiles,
            'Q':Q,'settings':dict(rs=rs,zeta=zeta,power=power,reps=reps,nq=nq,order=order,nk=nk,qmin=qmin,qmax=qmax,seed=seed,bare=bare)}


In [30]:

# Stability and original-functional normalization; tolerances are fixed in advance.
for cval in [0,1e-12,1e-4,1,100]:
    num('stable coupling c=%g'%cval,float(coupling(cval)),Rn(mp.mpf(str(cval))),tol=1e-14)
for qval,nuval in [(.2,.01),(2.,.001),(8.,3.),(100.,1e5)]:
    num('Gauss Lindhard vs mpmath defining integral',float(lindhard(qval,np.array([nuval]),nk=96)[0]),lindhard_shape_exact(mp.mpf(str(qval)),mp.mpf(str(nuval))),tol=2e-6)
anchor=energy(1000,power=16,reps=12,nq=48,order=16,bare=True,qmin=.001,qmax=300)
e_ons=float(sp.N(sp.log(2)/3-3*sp.zeta(3)/(2*pi**2),20))
print('Original bare functional: %.8f +/- %.8f Ry (replicate SE); Onsager %.8f'%(anchor['mean'],anchor['se'],e_ons))
check('Bare original integral agrees with Onsager within 4 replicate SE plus 0.00015 Ry quadrature allowance',abs(anchor['mean']-e_ons)<4*anchor['se']+.00015,kind='DIAGNOSTIC')


NUM  PASS  stable coupling c=0   [|a-b|/max(1,|b|) = 0.0; tolerance = 1e-14]
NUM  PASS  stable coupling c=1e-12   [|a-b|/max(1,|b|) = 2.23e-17; tolerance = 1e-14]
NUM  PASS  stable coupling c=0.0001   [|a-b|/max(1,|b|) = 6.97e-17; tolerance = 1e-14]
NUM  PASS  stable coupling c=1   [|a-b|/max(1,|b|) = 4.64e-17; tolerance = 1e-14]
NUM  PASS  stable coupling c=100   [|a-b|/max(1,|b|) = 6.06e-19; tolerance = 1e-14]
NUM  PASS  Gauss Lindhard vs mpmath defining integral   [|a-b|/max(1,|b|) = 6.72e-17; tolerance = 2e-06]
NUM  PASS  Gauss Lindhard vs mpmath defining integral   [|a-b|/max(1,|b|) = 5.1e-7; tolerance = 2e-06]
NUM  PASS  Gauss Lindhard vs mpmath defining integral   [|a-b|/max(1,|b|) = 2.45e-17; tolerance = 2e-06]
NUM  PASS  Gauss Lindhard vs mpmath defining integral   [|a-b|/max(1,|b|) = 3.53e-22; tolerance = 2e-06]
Original bare functional: 0.04853693 +/- 0.00015963 Ry (replicate SE); Onsager 0.04835832
DIAGNOSTIC  PASS  Bare original integral agrees with Onsager within 4 replic

<a id="check-29"></a>
## Check 29 — Original integral and convergence diagnostics

At $r_s=10^3,10^4$, the original integral is evaluated before forming the three-term prediction. The default coarse test requires relative agreement within 5% and improvement on adding the third term. The 5% threshold is a test setting, not an accuracy guarantee for the asymptotic formula.

At $r_s=10^3$, separate runs refine the $q$ grid, the $\nu,k$ quadrature, and the momentum endpoints. Differences are paired using common scramble seeds. The output reports statistical errors, refinement changes, and their size relative to the higher-order residual. `RUN_EXTENDED=True` enables a larger spin/density scan.

In [31]:

INDEPENDENT={}
for rsval in [1e3,1e4]:
    tic=time.perf_counter()
    run=energy(rsval,power=15,reps=12,nq=32,order=12,nk=48)
    INDEPENDENT[rsval]=run
    Qv=run['Q']; terms=np.array([float(A1)/Qv,(8/15)/Qv**2,float(A3)/Qv**3]);pred=terms.sum()
    print('rs=%g: original %.6f +/- %.6f mRy; three terms %.6f; residual %.6f'%(rsval,1000*run['mean'],1000*run['se'],1000*pred,1000*(run['mean']-pred)))
    check('Original-integral agreement within 5%% at rs=%g'%rsval,abs(run['mean']-pred)<.05*abs(run['mean']),kind='DIAGNOSTIC')
    check('Adding A3 improves finite-density agreement at rs=%g'%rsval,abs(run['mean']-pred)<abs(run['mean']-terms[:2].sum()),kind='DIAGNOSTIC')
    print('R3=%.6f +/- %.6f; target A3=%.6f. Scaled errors grow with Q.'%(Qv**3*(run['mean']-terms[:2].sum()),Qv**3*run['se'],float(A3)))

base=INDEPENDENT[1e3]; refinements={}
for name,kw in [('q grid',dict(nq=64)),('frequency and Lindhard',dict(order=24,nk=96)),('momentum endpoints',dict(qmin=.0005,qmax=60*base['Q']))]:
    settings=dict(power=15,reps=12,nq=32,order=12,nk=48);settings.update(kw)
    refined=energy(1e3,**settings);refinements[name]=refined
    delta=refined['replicates']-base['replicates'];dmean=delta.mean();dse=delta.std(ddof=1)/np.sqrt(len(delta))
    residual=base['mean']-(float(A1)/base['Q']+(8/15)/base['Q']**2+float(A3)/base['Q']**3)
    print('%s: change %.6f +/- %.6f mRy (paired SE); |change|/|three-term residual|=%.3f'%(name,1000*dmean,1000*dse,abs(dmean)/abs(residual)))
    check('%s refinement changes total by less than 1%% plus 4 paired SE'%name,abs(dmean)<.01*abs(base['mean'])+4*dse,kind='DIAGNOSTIC')

if RUN_EXTENDED:
    for rsval,zval in [(1e3,.5),(1e4,.5),(1e5,0.)]:
        rrun=energy(rsval,zeta=zval,power=17,reps=16,nq=64,order=24,nk=96)
        mv=lambda n:sum(x**n for x in [(1+zval)**(1/3),(1-zval)**(1/3)])/2
        s3=(1160*mv(6)+891*mv(4)*mv(5))/2051;Qv=rrun['Q']
        prediction=float(A1)*mv(4)/Qv+(8/15)*mv(5)/Qv**2+float(A3)*s3/Qv**3
        print('EXTENDED',rsval,zval,rrun['mean'],rrun['se'],'three terms',prediction)
else:
    SKIPPED.append('Extended high-statistics partial-polarization and density scan (RUN_EXTENDED=False)')


rs=1000: original -23.344721 +/- 0.016372 mRy; three terms -24.008085; residual 0.663364
DIAGNOSTIC  PASS  Original-integral agreement within 5% at rs=1000
DIAGNOSTIC  PASS  Adding A3 improves finite-density agreement at rs=1000
R3=-0.309727 +/- 0.002655; target A3=-0.417325. Scaled errors grow with Q.
rs=10000: original -16.841481 +/- 0.010801 mRy; three terms -16.924746; residual 0.083265
DIAGNOSTIC  PASS  Original-integral agreement within 5% at rs=10000
DIAGNOSTIC  PASS  Adding A3 improves finite-density agreement at rs=10000
R3=-0.341377 +/- 0.009852; target A3=-0.417325. Scaled errors grow with Q.
q grid: change 0.001669 +/- 0.004948 mRy (paired SE); |change|/|three-term residual|=0.003
DIAGNOSTIC  PASS  q grid refinement changes total by less than 1% plus 4 paired SE
frequency and Lindhard: change 0.000029 +/- 0.000006 mRy (paired SE); |change|/|three-term residual|=0.000
DIAGNOSTIC  PASS  frequency and Lindhard refinement changes total by less than 1% plus 4 paired SE
momentum 

<a id="check-30"></a>
## Check 30 — Exact full-polarization identity at the implementation level
The unit-loop scaling must reproduce the unpolarized integrand at $r_s'=2^{-4/3}r_s$. The paired numerical test uses identical unit-loop cutoffs, sample points, and quadrature settings. It checks the rescaling implementation, rather than the assignment of the two distinct third-order spin moments, which coincide at full polarization (Check 25).

In [32]:

spin_run=energy(1000,zeta=1.,power=12,reps=4,nq=24,order=12,qmax=100)
scaled_run=energy(1000/2**(4/3),zeta=0.,power=12,reps=4,nq=24,order=12,qmax=100)
err_spin=np.max(np.abs(spin_run['replicates']-scaled_run['replicates']))
check('Full-polarization rescaling agrees replicate by replicate to 1e-12 Ry',err_spin<1e-12,detail=str(err_spin),kind='DIAGNOSTIC')


DIAGNOSTIC  PASS  Full-polarization rescaling agrees replicate by replicate to 1e-12 Ry   [3.469446951953614e-18]


## Production figures and tables: provenance and regeneration

| Manuscript output | Raw dataset | Generation |
|---|---|---|
| Fig. 1 | none | TikZ in the manuscript |
| Fig. 2 | analytic schematic | `figs/make_matching_fig.py` |
| Fig. 3 | none | Occupation geometry drawn in TikZ |
| Fig. 4 | `shell` | Imaginary-axis density, including a frequency-refinement repeat |
| Figs. 5 and 9; Table I | `tableI` | Direct BRM integrals, then model comparisons and residuals |
| Fig. 6 | `map` | (a) Direct BRM energy map; (b) paired polarization differences with error bars; crossing recomputed from the grid |
| Fig. 7 | three analytic energy coefficients | `figs/make_spin_interpolation.py`; no new integral or FRG calculation |
| Fig. 8 | saved `map` replicates | `figs/make_spin_bias.py`; subtraction retaining all three-polarization covariances |
| Table II | `tableII` | Direct BRM integrals |
| Table III | `grid`, `tableIII_direct` | Node quadrature and independent direct representation |
| Table IV | `tableIV` | Direct BRM integrals near the minimum |

The following cell **rebuilds the figures and tables from saved measurements** and checks their consistency. It does not label this as a new evaluation of the integral. To repeat the production integrals, use the separate commands in `REPRODUCE_SPIN.md` or enable `RUN_PRODUCTION` at setup and rerun the notebook. The node-based datasets store both standard and finer frequency quadrature. Statistical errors and quadrature changes are displayed separately.

All direct-term uncertainties use the quadrature sum of the two independent term SEMs. The polarization crossing uses a delete-one-replicate jackknife: omit each of the eight independent randomizations from the entire grid in turn, then recompute the crossing. Shared random numbers are retained across the grid; the standard error is sqrt((R-1)/R times the sum of squared deviations of the R omitted-replicate crossings from their mean). Fig. 6(b) subtracts the unpolarized term within each replicate before combining the two independent term variances. The spline-to-PCHIP shift is an interpolation diagnostic, not part of that statistical error. Raw records are independently randomized **replicate means**, not every individual Monte Carlo sample. The seed, point count, generator and source hashes specify the samples.



In [33]:
import subprocess
from IPython.display import display, Markdown
if RUN_PRODUCTION:
    for dataset in ['tableI','tableII','tableIII_direct','tableIV','map','shell','grid']:
        subprocess.run([sys.executable,str(BUNDLE_ROOT/'production/compute.py'),'--dataset',dataset,'--force'],check=True)
    subprocess.run([sys.executable,str(BUNDLE_ROOT/'production/render.py')],check=True)
    PRODUCTION=json.loads((BUNDLE_ROOT/'production/generated/summary.json').read_text())
else:
    SKIPPED.append('Full production integral rerun: saved measurements were used for rendering; Checks 28-30 run independently.')
result=subprocess.run([sys.executable,str(BUNDLE_ROOT/'production/verify.py'),'--render'],text=True,capture_output=True)
print(result.stdout)
if result.returncode: print(result.stderr)
check('PROVENANCE: complete raw coverage and reproducible generated outputs',result.returncode==0)
rows=['| rs | quadrature, extended (mRy) | direct (mRy) | frequency-order shift (mRy) |',
      '|---:|---:|---:|---:|']
for g,d in zip(PRODUCTION['grid'],PRODUCTION['tableIII_direct']):
    v=g['extended']
    rows.append('| %g | %.6f +/- %.6f | %.6f +/- %.6f | %.3g |'%(g['rs'],v['mean_mry'],v['se_mry'],d['mean_mry'],d['se_mry'],v['frequency_shift_mry']))
display(Markdown('\n'.join(rows)))
for row in PRODUCTION['shell']:
    shift=np.abs(row['frequency_shift_ry']);se=np.array(row['se_ry'])
    print('Shell rs=%g: max paired frequency shift / SEM = %.3g'%(row['rs'],np.max(shift/se)))
m=PRODUCTION['map']
print('Full-polarization crossing: %.6f; jackknife SEM %.6f; PCHIP %.6f'%(m['crossing'],m['crossing_jackknife_se'],m['pchip_crossing']))


Generated 4 numerical figures, 4 table bodies, numerical text macros and CSV/JSON data.
PASS: 134 numerical records, complete configured coverage and aggregate formulas.
PASS: all generated figure PDFs, table rows, numerical macros and CSV/JSON outputs reproduce byte for byte.

CHECK  PASS  PROVENANCE: complete raw coverage and reproducible generated outputs
Shell rs=1000: max paired frequency shift / SEM = 1.46e-05
Shell rs=100000: max paired frequency shift / SEM = 9.27e-08
Shell rs=1e+07: max paired frequency shift / SEM = 9.22e-08
Full-polarization crossing: 464.255931; jackknife SEM 6.870048; PCHIP 464.676441


| rs | quadrature, extended (mRy) | direct (mRy) | frequency-order shift (mRy) |
|---:|---:|---:|---:|
| 1 | 27.877987 +/- 0.053280 | 27.834315 +/- 0.020458 | 1.22e-10 |
| 2 | 19.581162 +/- 0.059326 | 19.505431 +/- 0.026946 | 6.84e-10 |
| 5 | 6.815071 +/- 0.074686 | 6.684721 +/- 0.035897 | 2.33e-10 |

## Execution summary through Check 31

The next cell verifies the source hash and reports every result by evidence type, together with package versions, the production runtime contract, and any optional skips. Any failed test raises an error. The equation table at the beginning uses labels from the compiled manuscript.

This summary counts 284 main checks through Check 31. Checks 32–34 follow below and passed in the final fresh-process verification; their additional 5, 9 and 8 PASS groups are reported separately and are not double-counted here.

In [34]:

texpath=Path('main_brm_kite_asymptotics.tex')
mapfile=Path('equation_map.json')
if texpath.exists() and mapfile.exists():
    emap=json.loads(mapfile.read_text())
    check('Equation map belongs to the supplied TeX source',hashlib.sha256(texpath.read_bytes()).hexdigest()==emap['tex_sha256'])
    check('Generated table and numerical-text inputs match the compiled manuscript',
          all(hashlib.sha256(Path(name).read_bytes()).hexdigest()==h for name,h in emap['generated_sha256'].items()))
    print('Compiled equation labels:',len(emap['labels']))
    thisbook=json.loads(Path('spin_reviewer_verification.ipynb').read_text())
    check('Notebook equation references and source version are synchronized',
          thisbook['metadata'].get('equation_references')==emap['labels'] and
          thisbook['metadata'].get('manuscript_tex_sha256')==emap['tex_sha256'])
else:
    SKIPPED.append('Source-hash verification (standalone notebook without TeX/map files)')
versions={p:importlib.metadata.version(p) for p in ['numpy','scipy','sympy','mpmath','nbformat','nbclient','nbconvert']}
print('Package versions:',versions)
print('Deterministic production runtime:', production_runtime.contract())
for kind in sorted(set(KINDS)):
    vals=[ok for (_,ok),k in zip(RESULTS,KINDS) if k==kind]
    print('%s: %d PASS, %d FAIL'%(kind,sum(vals),len(vals)-sum(vals)))
for msg in SKIPPED:print('SKIP:',msg)
failures=[label for label,ok in RESULTS if not ok]
print('TOTAL: %d checks, %d PASS, %d FAIL; %d optional skips'%(len(RESULTS),len(RESULTS)-len(failures),len(failures),len(SKIPPED)))
assert not failures, 'Failed checks: '+ '; '.join(failures)
print('All executed tests passed within their stated scope. Uniform bounds and diagrammatic premises are not certified by this output.')


Package versions: {'numpy': '2.3.5', 'scipy': '1.17.0', 'sympy': '1.14.0', 'mpmath': '1.3.0', 'nbformat': '5.11.1', 'nbclient': '0.11.0', 'nbconvert': '7.17.1'}
Deterministic production runtime: {'python': '3.12.14', 'packages': {'numpy': '2.3.5', 'scipy': '1.17.0', 'matplotlib': '3.10.8', 'threadpoolctl': '3.6.0'}}
CHECK: 11 PASS, 0 FAIL
DIAGNOSTIC: 9 PASS, 0 FAIL
EXACT: 168 PASS, 0 FAIL
MODEL: 2 PASS, 0 FAIL
NUM: 94 PASS, 0 FAIL
SKIP: Extended high-statistics partial-polarization and density scan (RUN_EXTENDED=False)
SKIP: Full production integral rerun: saved measurements were used for rendering; Checks 28-30 run independently.
SKIP: Source-hash verification (standalone notebook without TeX/map files)
TOTAL: 284 checks, 284 PASS, 0 FAIL; 3 optional skips
All executed tests passed within their stated scope. Uniform bounds and diagrammatic premises are not certified by this output.


<a id="check-32"></a>
## Check 32 — Spin interpolation and its first correction

Added for the FRG-context revision of 2026-09-27 and rerun in the final verification of 2026-09-28. It derives the first correction by symbolic differentiation of the energy quotient, checks both endpoint normalizations, and compares the plot with independent 70-digit evaluations of the three-term energy. It also tests the predicted convergence of the quotient and verifies its CSV data.

These are checks of consequences of the previously derived coefficients. They do not evaluate new many-body integrals, test an FRG flow, or prove equivalence between BRM and FRG resummations. The final reviewer-notebook run and the additional fresh-process validation are recorded in the audit log; the full production integral suite was not rerun.

In [35]:
from pathlib import Path
import importlib.util
_spin_check = Path("verify_spin_interpolation.py")
_spin_spec = importlib.util.spec_from_file_location("spin_interpolation_audit", _spin_check)
_spin_module = importlib.util.module_from_spec(_spin_spec)
_spin_spec.loader.exec_module(_spin_module)
_spin_module.run_checks()


EXACT PASS Eqs.(53),(54) symbolic quotient gives f_x and H; the next coefficient also cancels.
EXACT PASS Eq.(54) H(0)=H(+1)=H(-1)=0 exactly.
NUM PASS Eq.(53) all plotted formulas match the direct 70-digit energy quotient.
NUM PASS Eq.(53) subtracted ratio converges at the predicted first-correction order.
NUM PASS spin-interpolation CSV: all 501 rows reproduce the analytic plotting functions.


<a id="check-33"></a>
## Check 33 — Signed bias of spin interpolation

Exact algebra checks the endpoint construction, the residual coefficient, and every ingredient of the analytic positivity argument in Eqs. (55)–(57). Independent 70-digit checks supplement that proof. The conclusion holds at each fixed intermediate polarization; no derivative of an asymptotic remainder or spin susceptibility is inferred.

Executed in the final fresh-process validation of 2026-09-28. All preceding reviewer-notebook computations were also rerun for this final audit. The full production integral suite was not rerun.

In [36]:
from pathlib import Path
import importlib.util
_spec = importlib.util.spec_from_file_location("verify_spin_bias_algebra", Path("verify_spin_bias_algebra.py"))
_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_module)
_module.run_checks()


EXACT PASS Eq.(55) [eq:spin-endpoint-interpolation] epsilon_int = epsilon_0 + f_x*(epsilon_1-epsilon_0) matches both endpoints.
EXACT PASS Eq.(56) [eq:spin-interpolation-bias] the 1/q_* term cancels and the 1/q_*^2 residual is A_2*B; the next coefficient is retained independently.
EXACT PASS Eq.(56) [eq:spin-interpolation-bias] matching the first two powers requires shapes differing by B/d_5; a density-independent shape cannot match both at interior polarization.
EXACT PASS Eq.(56) [eq:spin-interpolation-bias] B is even, B(0)=B(+1)=B(-1)=0, and d_5/d_4=1+2^(1/3).
EXACT PASS Eq.(57) [eq:spin-bias-derivative] B'=(x-y)*[5*(x+y)-4*(1+c)]/6 by the exact chain rule.
EXACT PASS Eq.(57) [eq:spin-bias-derivative] analytic interval proof: 1<c<3/2, x>y, and x+y strictly decreases from 2 to c; B' changes sign exactly once (+ to -). With zero endpoint values, B>0 for 0<|zeta|<1.
EXACT PASS Eq.(56) [eq:spin-interpolation-bias] A_1<0<A_2 and d_4>0 imply H<0 and positive leading energy bias at each fi

<a id="check-34"></a>
## Check 34 — Finite-density interpolation error from saved replicates

This reanalysis uses the same 104 records as Figure 6 to produce Figure 8. It verifies raw-data and evaluator hashes, subtraction of energy means, both endpoint zeros, and the full three-polarization covariance separately for each independent term. The plotted uncertainties are empirical replicate standard errors, not confidence bounds. No new integral is evaluated.

Executed in the final fresh-process validation of 2026-09-28. All preceding reviewer-notebook computations were also rerun for this final audit. The full production integral suite was not rerun.

In [37]:
from pathlib import Path
import importlib.util
_spec = importlib.util.spec_from_file_location("verify_spin_bias", Path("verify_spin_bias.py"))
_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_module)
_module.run_checks()


PROVENANCE PASS: all 104 raw records match reference hashes, exact or explicitly audited compatible sources, and common sampling settings.
NUM PASS: every exported residual agrees with direct subtraction of the three raw energy means.
NUM PASS: all 104 error bars agree with the independent 3x3 covariance-matrix propagation for each term.
NUM PASS: both endpoints vanish exactly and adding a common bare-exchange constant leaves every residual unchanged.
EXACT PASS: common constants and the exchange-shaped leading term cancel; residual second and third coefficients assemble correctly.
NUM PASS: zeta=0.8, rs=1: -0.629824805791 +/- 0.005670003085 mRy.
NUM PASS: zeta=0.8, rs=10: +0.224252998179 +/- 0.008112140232 mRy.
NUM PASS: zeta=0.8, rs=60: +0.638429263372 +/- 0.007604148498 mRy.
SCOPE: uncertainties are empirical replicate standard errors; neither confidence coverage nor sampler variance finiteness is established here.
SCOPE: stored-data reanalysis only; no Monte Carlo/HPC rerun, no fit

<a id="equation-audit"></a>

# Equation coverage of the PRB manuscript

D: definition or theory input; A: symbolic identity on the stated domain; N: sampled numerical check; P: symbolic and numerical checks of components, with the assembly argued in the paper; -: not checked computationally. These categories are not proof-completion scores. Built from the executed notebook by `build_coverage_ledger.py`.

88 numbered and 0 unnumbered displays. Status counts (numbered): {'definition': 19, 'algebra': 35, 'numerical': 12, 'partial': 18, 'unchecked': 4}.

## Numbered displays

| Equation / label | Scope | Checks | Evidence and analytic role |
|---|---|---|---|
| (1) / `eq:u1` | D |  | Screened line written for the introduction; the same object as Eqs. (dielectric)-(static-kernel). |
| (2) / `eq:u2` | D |  | Large-q consequence at fixed recoil-scaled frequency u=nu/q^2: q^2/(q_tf_sq G) ~ 3 q^4 (1+4u^2)/(4 q_tf_sq). Reviewed analytically; not independently checked by the notebook. |
| (3) / `eq:qstar-intro` | D | 13, 19 | Definition of the moving momentum scale. |
| (4) / `eq:spin-moments-intro` | D | 2, 13 | Definitions of x_sigma and M_n; M_3 = 1 is checked. |
| (5) / `eq:S3-intro` | D | 13, 25 | Definition of S_3; its normalization and small-zeta form are checked. |
| (6) / `eq:main-rs` | N | 5, 12 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(6)  A1 (4a/3)^(-1/4) = -0.2213782293; Eq.(6)  A1/q*   -> -0.2213782293 r_s^(-1/4); Eq.(6)  A2/q*^2 -> +0.5670608947 r_s^(-1/2) |
| (7) / `eq:coupling-energy` | D |  | Theory input: Hellmann-Feynman and Galitskii-Migdal representation. |
| (8) / `eq:dielectric` | D |  | Definition of the RPA dielectric function. |
| (9) / `eq:static-kernel` | D |  | Definition of the screened line and its coupling-stripped form. |
| (10) / `eq:K-normalization` | A | 16 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(10)  2 int_0^1 lam dlam = 1 |
| (11) / `eq:onsager` | N | 1 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(11)  e2x = 0.0483583 Ry |
| (12) / `eq:reduced-convention` | D |  | Unit convention; its consequence for A_0 is checked with Eqs. (11)-(12). |
| (13) / `eq:A0-normalization` | A | 1 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eqs.(13),(14) 2*(1/2)*2*(1/2)*(4pi)^2 (2pi)^-9 (3pi^2) = 3/(32 pi^5) |
| (14) / `eq:A0-factors` | A | 1 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eqs.(13),(14) 2*(1/2)*2*(1/2)*(4pi)^2 (2pi)^-9 (3pi^2) = 3/(32 pi^5) |
| (15) / `eq:energy-differences` | D |  | Definitions of the occupation difference and energy denominator. |
| (16) / `eq:K-master` | N | 1, 15, 28 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(16) static line: int dnu/pi Re[1/((i nu-a)(i nu-b))] = -Theta(-ab)/(∣a∣+∣b∣), a=0.7 b=-1.3; Eq.(16) static line: int dnu/pi Re[1/((i nu-a)(i nu-b))] = -Theta(-ab)/(∣a∣+∣b∣), a=-0.2 b=0.9; Eq.(16) static line: int dnu/pi Re[1/((i nu-a)(i nu-b))] = -Theta(-ab)/(∣a∣+∣b∣), a=0.4 b=1.1 |
| (17) / `eq:c-definition` | A | 19 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(17) (qk+q^2/2)^2 - (qk-q^2/2)^2 = 2 q^3 k >= 0  =>  G >= 0, c >= 0 |
| (18) / `eq:coupling-average` | A | 1, 11 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(18) 2 int lam dlam/(1+lam c) = 2[c - ln(1+c)]/c^2; Eq.(18) R(0) = 1; Eq.(18) R(c) = 1 - 2c/3 + c^2/2 + O(c^3) |
| (19) / `eq:sector-split` | A | 1 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eqs.(18),(19) (R(c)-1)/c -> -2/3: e_2b - e_2x = K[v(R-1)] vanishes with the screening |
| (20) / `eq:Xdyn-definition` | D | 1, 15, 20, 28 | Definition of the radial kernel; the normalization 6/pi^3 is checked. |
| (21) / `eq:tau-def` | D |  | Definition of q_tf_sq, a and alpha_L; numerical values are checked. |
| (22) / `eq:cG` | D |  | Definition of the normalized Lindhard response G. |
| (23) / `eq:Gexact` | N | 16 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(23) \mathcal L(q,0) = f_L(q/2), q=0.8; Eq.(23) \mathcal L(q,0) = f_L(q/2), q=3.0; Eq.(23) \mathcal L(0,0) = 1 (normalization), q=1e-4 |
| (24) / `eq:polarized-response` | A | 2, 25 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(24)/(28) x \mathcal L(q/x, nu/x^2): leading term scales as x^3  =>  M_3; Eq.(24)/(28) x \mathcal L(q/x, nu/x^2): next term scales as x^5     =>  M_5; Eqs.(24)-(26)  c_sigma(q, q^2 u) = (q*/(x_sigma q))^4/(1+4u^2) [M_3 + g4(u) M_5/(x_sigma^2 q^2)] |
| (25) / `eq:spin-rescaled-line` | D | 25 | Definition of the spin-rescaled line. |
| (26) / `eq:spin-loop-weight` | A | 25 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eqs.(24)-(26)  c_sigma(q, q^2 u) = (q*/(x_sigma q))^4/(1+4u^2) [M_3 + g4(u) M_5/(x_sigma^2 q^2)]; Eqs.(24)-(26)  outer region: c_sigma -> q_tf_sq M_3/(3 x_sigma^4 nu^2) |
| (27) / `eq:inner-scale` | D |  | Definition of the inner variables x and u. |
| (28) / `eq:lindhard-largeq` | A | 2 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(28) q^-2 coefficient = 4/[3(1+4u^2)]; Eq.(28) q^-4 coefficient = 16(1-12u^2)/[15(1+4u^2)^3]; Eq.(28) no odd powers |
| (29) / `eq:c-inner` | P | 16, 21 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(29) (q_tf_sq/q^2)(4/(3 q^2 (1+4u^2))) = (q*/q)^4/(1+4u^2) with q*^4 = 4 q_tf_sq/3; Eq.(29) sampled inner-limit grid (three x values, 25 u values): deviation < 1e-3 at r_s=1e8; Eq.(29) deviation falls like q_*^-2 (factor 10 per two decades of r_s, +-30%) |
| (30) / `eq:same-sphere` | P | 3, 8 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(30) overlap volume = 2 caps of height 1-r/2; Eq.(30) V(0) = 4pi/3, V(2) = 0; Eq.(30) 4 pi int_0^2 V(r) dr = 4 pi^2 |
| (31) / `eq:Fprofile` | P | 4 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(31) residue identity, m=0.5; Eq.(31) residue identity, m=3.0; Eq.(31) residue identity, m=40.0 |
| (32) / `eq:bare-same-sphere-zero` | A | 4, 24 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(32) (1/4-u^2)/(1/4+u^2)^2 = d/du [u/(1/4+u^2)]  =>  bare integral vanishes; Eq.(32) int_0^oo (1/4-u^2)/(1/4+u^2)^2 du = 0; Eq.(32)  Re A^-2 = (1/4-u^2)/(1/4+u^2)^2 |
| (33) / `eq:A1-integral` | P | 5 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(33) 32 pi^3 A0 = 3/pi^2; Eqs.(33),(34) A1 = (3/pi^2) int F(x^-4)/x^2 dx equals the companion route -(48/(7 pi^2)) int s_lambda/((1+2mu)^2 t^2 |
| (34) / `eq:A1` | P | 5 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(34) A1 = (3/pi^2) * Beta form; Eq.(34) A1 = -0.21469376098; Eqs.(33),(34) A1 = (3/pi^2) int F(x^-4)/x^2 dx equals the companion route -(48/(7 pi^2)) int s_lambda/((1+2mu)^2 t^2 |
| (35) / `eq:K0-outer` | P | 6, 21 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(35) int dnu/(pi nu^2) [1+lam q_tf_sq/(3 nu^2)]^-1 = (1/2) sqrt(3/(lam q_tf_sq)); Eq.(35) K0 = -2/sqrt(3 q_tf_sq); Eq.(35) -2/sqrt(3 q_tf_sq) = -4/(3 q*^2) with q*^4 = 4 q_tf_sq/3 |
| (36) / `eq:ZC` | N | 19, 24 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(36) Z(q) = 8 pi^2 - 2 C(q) at q=0.5  (Fourier form of Z vs. overlap form of C); Eq.(36) Z(q) = 8 pi^2 - 2 C(q) at q=3.0  (Fourier form of Z vs. overlap form of C); Eq.(36)  C(q) -> 4 pi^2 as q -> 0, i.e. Z(0) = 0  (evaluated at q = 1e-3; the closed form loses digits below t |
| (37) / `eq:Zfinitepart` | A | 7 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(37) Fp int Z dq = -2 int C dq = -16 pi^4/15 |
| (38) / `eq:A2` | A | 7 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(38) A2 = 4 pi A0 (-4/3)(-16 pi^4/15) = 8/15 |
| (39) / `eq:fp-moment` | N | 8 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(39) int (p_z^2 + k_z^2 + p_z k_z)/∣p-k∣^2 = 3 U2 - D2 = 52 pi^2/27 |
| (40) / `eq:JC-overlap` | P | 9 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(40) 2 int lam dlam/sqrt(1+lam g) = (4/3g^2)[(1+g)^(3/2) - 3(1+g)^(1/2) + 2]; Eq.(40) J_C(0) = 1; Eq.(40) J_C(g) = 1 - g/3 + 3g^2/16 - g^3/8 + 35g^4/384 + O(g^5)  [series used by JCn below] |
| (41) / `eq:JC-endpoints` | A | 9 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(41) x^-4 J_C = 4/(3x^2) - 2x^2 + O(x^4) as x->0; Eq.(41) x^-4 J_C = 4/(3x^2) - 2x^2 + 8x^4/3 - 3x^6/2 + 5x^10/12 + O(x^12)  [series used by hsub below]; Eq.(41) x^-4 J_C = x^-4 + O(x^-8) as x->oo |
| (42) / `eq:A3` | N | 12 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(42) A3 = -0.41732536262 |
| (43) / `eq:main-qstar` | A | 12 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(43) q*^-1 coefficient = A1 of Eq.(34); Eq.(43) q*^-3 coefficient = A3 of Eq.(42) = -(16408/3375) C_Gamma; Eqs.(43),(46) at zeta=0 (M_n=1): (1856/675 + 264/125) = 16408/3375 |
| (44) / `eq:spin-cross-composite` | A | 16 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(44) C_sigma(q) = x^4 C(q/x); Eq.(44) int C_sigma dq = x^5 int C dq   (dq = x dq2); Eq.(44) q^2 C_sigma -> x^6 V_B^2   (using q^2 C(q) -> V_B^2) |
| (45) / `eq:spin-geometry-scaling` | A | 15 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(45) I_0: d^3p d^3k / ∣p-k∣^2 scales as x^6 / x^2 = x^4; Eq.(45) I_2: (p_z^2 ...) d^3p d^3k / ∣p-k∣^2 scales as x^2 x^6 / x^2 = x^6; Eq.(45) V_B^2 scales as x^6 |
| (46) / `eq:main-qstar-zeta` | A | 12, 13, 25 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(46) 32/15 + 416/675 = 1856/675; Eqs.(43),(46) at zeta=0 (M_n=1): (1856/675 + 264/125) = 16408/3375; Eq.(46)->(3) (1856/675)/(16408/3375) = 1160/2051 |
| (47) / `eq:polarized-c-scaling` | A | 13 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(47) c_{zeta=1}(x q, x^2 nu; rs) = c_{zeta=0}(q, nu; rs/2^(4/3)) |
| (48) / `eq:polarized-D-scaling` | A | 16 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(48) x^2 (x q)^-2/(1+lam c) = q^-2/(1+lam c) |
| (49) / `eq:full-polarization-scaling` | A | 13 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(49) M_4(1) = 2^(1/3); Eq.(49) M_5(1) = 2^(2/3); Eq.(49) M_6(1) = 2 = M_4(1) M_5(1) |
| (50) / `eq:ferro-integral` | A | 16 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(50) -int_{2^(-4/3) rs}^{rs} E'(t) dt = E(2^(-4/3) rs) - E(rs) |
| (51) / `eq:spin-interpolation-definition` | D |  | Definition where the polarization-energy denominator is nonzero; the low-density nonzero denominator follows from A_1 d_4 < 0. |
| (52) / `eq:exchange-spin-interpolation` | D |  | Exchange interpolation defined from M_4; equality of the two printed forms follows by cancelling their factors 1/2. |
| (53) / `eq:spin-interpolation-expansion` | P | 32 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eqs.(53),(54) symbolic quotient gives f_x and H; the next coefficient also cancels.; Eq.(53) all plotted formulas match the direct 70-digit energy quotient.; Eq.(53) subtracted ratio converges at the predicted first-correction order. |
| (54) / `eq:spin-interpolation-correction` | A | 32 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eqs.(53),(54) symbolic quotient gives f_x and H; the next coefficient also cancels.; Eq.(54) H(0)=H(+1)=H(-1)=0 exactly. |
| (55) / `eq:spin-endpoint-interpolation` | D | 33 | Exchange-form interpolation between the exact unpolarized and fully polarized kite energies; endpoint reproduction is checked. |
| (56) / `eq:spin-interpolation-bias` | P | 33 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(56) [eq:spin-interpolation-bias] the 1/q_* term cancels and the 1/q_*^2 residual is A_2*B; the next coeffi; Eq.(56) [eq:spin-interpolation-bias] matching the first two powers requires shapes differing by B/d_5; a densi; Eq.(56) [eq:spin-interpolation-bias] B is even, B(0)=B(+1)=B(-1)=0, and d_5/d_4=1+2^(1/3). |
| (57) / `eq:spin-bias-derivative` | P | 33 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(57) [eq:spin-bias-derivative] B'=(x-y)*[5*(x+y)-4*(1+c)]/6 by the exact chain rule.; Eq.(57) [eq:spin-bias-derivative] analytic interval proof: 1<c<3/2, x>y, and x+y strictly decreases from 2 to ; Eq.(57) [eq:spin-bias-derivative] supplemental 70-digit evaluations confirm the identities and signs at 199 in |
| (A.1) / `eq:A1-beta` | P | 5 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(A.1) int F(x^-4)/x^2 dx = -(4/(7 sqrt2)) B(5/4,1/2); Eq.(A.1) -(4/(7 sqrt2)) B(5/4,1/2) = -(4 sqrt(pi)/(21 sqrt2)) Gamma(1/4)/Gamma(3/4) |
| (A.2) / `eq:C-integral` | P | 7 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(A.2) int_0^oo dq/(rho^2+(q+z)^2) = [pi/2 - arctan(z/rho)]/rho; Eq.(A.2) int dOmega / sin(theta) = 2 pi^2; Eq.(A.2) int_0^oo C dq = pi^3 int r V dr = 8 pi^4/15 |
| (A.3) / `eq:CGamma` | N | 12 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(A.3) C_Gamma = 0.0858406325 |
| (A.4) / `eq:A3cross-profile` | A | 9 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(A.4) 8 pi A0 V_B^2 = 4/(3 pi^2); Eq.(A.4) A3,cross = -(32/15) C_Gamma |
| (A.5) / `eq:cross-beta` | P | 9 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(A.5) x^-4 J_C = (4/3)(s-1)(s+2)/(s+1) with s=sqrt(1+x^-4); Eq.(A.5) dx/ds = -(s/2)(s^2-1)^(-5/4); Eq.(A.5) (2/3) s(s+2)(s-1)^(-1/4)(s+1)^(-9/4) = (2/3)[(s^2-1)^(-1/4) - (s-1)^(-1/4)(s+1)^(-9/4)] |
| (A.6) / `eq:u3` | A | 31 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(A.6) 2 h'(s) = f(s) + f(s)/(s+1), f=(s^2-1)^(-1/4), h=(s-1)^(3/4)(s+1)^(-1/4) |
| (A.7) / `eq:u4` | N | 31 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(A.7) int_1^oo f/(s+1) ds = 2^(-1/2) B(3/4,1/2); Eq.(A.7) Fp int f ds = -2^(-1/2) B(3/4,1/2) equals the continuation 2^(1/2) B(3/4,-1/2); Eq.(A.7) 2 h(S_eps) - 2/eps -> 0 at eps=0.001 |
| (A.8) / `eq:A3fp-profile` | A | 24 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(A.8)  Taylor coefficient of order 0 in (p_z,k_z)/q of [(A-a)(A-b)]^-1; Eq.(A.8)  Taylor coefficient of order 1 in (p_z,k_z)/q of [(A-a)(A-b)]^-1; Eq.(A.8)  Taylor coefficient of order 2 in (p_z,k_z)/q of [(A-a)(A-b)]^-1 |
| (A.9) / `eq:kin-residue` | N | 10 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(A.9) kinematic residue identity, m=0.5; Eq.(A.9) kinematic residue identity, m=3.0; Eq.(A.9) kinematic residue identity, m=40.0 |
| (A.10) / `eq:kin-beta` | P | 10 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(A.10) int_0^1 lam^(1/4) dlam = 4/5; Eq.(A.10) int t^(-1/4)(-4t)/[sqrt(1+t)(sqrt(1+t)+1)^4] dt = -2 sqrt2 B(7/4,3/2); Eq.(A.10) -8 int_1^oo (s-1)^(3/4)(s+1)^(-13/4) ds = -8 * 2^(-3/2) B(7/4,3/2) = -2 sqrt2 B(7/4,3/2) |
| (A.11) / `eq:A3L-profile` | A | 11 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(A.11) G4(u) as stated equals the q^-4 Lindhard coefficient of Check 2 |
| (A.12) / `eq:L-residue` | N | 11 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(A.12) Lindhard-correction contour identity, m=0.5; Eq.(A.12) Lindhard-correction contour identity, m=3.0; Eq.(A.12) Lindhard-correction contour identity, m=40.0 |
| (A.13) / `eq:u5` | P | 31 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Checks listed in the notebook equation table. |
| (A.14) / `eq:MEuler` | A | 11 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(A.14) (s^2-1)^(3/4)(2s^3-s^2+4s+1)/(s^2(s+1)^4) ds with s=1/w gives the w-integrand (ratio = 1) |
| (A.15) / `eq:u6` | D |  | Definitions of f and r used for the total derivative of the next display. |
| (A.16) / `eq:u7` | A | 31 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(A.16) (f r)' = f [2 - w + 4w^2 + w^3 - 11(1+w)^2/(10(1-w))] |
| (A.17) / `eq:MEuler-value` | P | 11 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(A.17) M by quadrature (w=v^2) = (176/75) sqrt(2pi) Gamma(3/4)/Gamma(1/4); Eq.(A.17) T1/U = 16/15; Eq.(A.17) T2/U = 2/15 |
| (A.18) / `eq:A3-sum` | A | 12 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(A.18) 32/15 + 416/675 + 264/125 = 16408/3375 |
| (B.1) / `eq:scaled-profile` | A | 15, 17, 19, 20 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(B.1) (6/pi^3) int q*^-2 (pi/(2x^2)) F q* dx = (3/pi^2) q*^-1 int F/x^2 dx; Eq.(B.1) (1/(16 pi)) * 8 pi^2 * q^-4 * q^2 = pi/(2 q^2) |
| (B.2) / `eq:energy-residual-tests` | P | 15, 17, 19 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Checks listed in the notebook equation table. |
| (C.1) / `eq:G-bounds` | N | 18 | Agreement at the sampled arguments and tolerances printed by the checks; not a proof for all arguments. Tested by: Eq.(C.1) q^2/(3[nu^2+(q+q^2/2)^2]) <= \mathcal L(q,nu) <= q^2/(3 nu^2) at 4 sampled points |
| (C.2) / `eq:fixed-q-bound` | A | 18 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(C.3)->(C.2) leading composite density = 4 pi A0 K0 Z, K0 = -4/(3Q^2), Z = 8 pi^2 - 2C |
| (C.3) / `eq:composite-density` | A | 18, 20 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(C.3) sqrt(g) F(g) -> -4/3 as g -> oo; Eq.(C.3) sqrt(g) J_C(g) -> 4/3 as g -> oo; Eq.(C.3)->(C.2) leading composite density = 4 pi A0 K0 Z, K0 = -4/(3Q^2), Z = 8 pi^2 - 2C |
| (C.4) / `eq:u8` | A | 31 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(C.4) x^-2 F(x^-4) = -4/3 + 4x^2 + O(x^4); Eq.(C.4) H(x) = x^-2 J_C(x^-4) = 4/3 - 2x^4 + O(x^6) |
| (C.5) / `eq:u9` | - |  | No computational check; the manuscript gives the analytic argument. |
| (C.6) / `eq:u10` | - |  | No computational check; the manuscript gives the analytic argument. |
| (C.7) / `eq:u11` | A | 31 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(C.7) 1/∣q z + d∣^2 - 1/∣q z - d∣^2 = -4 q d_z/(∣q z + d∣^2 ∣q z - d∣^2) |
| (C.8) / `eq:u12` | - |  | No computational check; the manuscript gives the analytic argument. |
| (C.9) / `eq:cross-integrated` | P | 18, 20 | Symbolic and numerical checks of components; the assembly, interchange of limits or uniformity is argued analytically in the manuscript. Tested by: Eq.(C.9) int_0^oo [H(x) - 4/3] x^-2 dx = finite part of Eq.(A.4); Eq.(C.9)  (32 pi A0/(3 Q^2)) int C dq = A2/Q^2 with int C dq = 8 pi^4/15 |
| (C.10) / `eq:u13` | - |  | No computational check; the manuscript gives the analytic argument. |
| (D.1) / `eq:yukawaT` | A | 14 | Symbolic identity on the stated domain; its place in the derivation is analytic. Tested by: Eq.(D.1) q^2/(q^2+k^2) - q^2/k^2 = -q^4/[k^2 (q^2+k^2)]; Eq.(D.1) -(C_inf/k^2) int dq/(q^2+k^2) = -pi C_inf/(2 k^3) |

## Unnumbered displays

| Display | Line | Scope | Checks | Content |
|---|---|---|---|---|
